# Phase 4 — Professional Python: Packaging, Typing, Testing and the Standard Library

**Outcome:** ship a typed, tested, logged and installable package, and reach for the
right standard-library or third-party module with confidence.

| Unit | Topic |
| --- | --- |
| 4.1 | Package management and project structure |
| 4.2 | The type system (Python's TypeScript) |
| 4.3 | Testing, debugging and logging |
| 4.4 | Standard library essentials |
| 4.5 | File I/O, encodings and serialization |
| 4.6 | Essential third-party libraries |
| Project | Publish the Log Analyzer |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- This phase is about **tools**: uv, pip, mypy, pyright, ruff and friends. Tool demos
  write small throwaway projects under `notebooks/_scratch/` (git-ignored) and run
  the real tool in a child process, printing its real output. Absolute paths are
  shown as `<scratch>` (the unit's scratch folder), `<repo>` or `~`, and timings as
  `<time>`.
- No cell changes the course's own `pyproject.toml` or `uv.lock` (a check at the end
  of Unit 4.1 proves it), and nothing is ever published.
- Packaging demos resolve against a **local wheelhouse** of tiny demo packages built
  in Unit 4.1, so they work offline and give the same answer every time. The only
  cells that need the network (a vulnerability audit and one third-party tool)
  check first and print a skip message when you are offline.
- Tool output can change with the tool's version; the versions used are printed.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub (or fix the code, where
  the exercise says so), then run the check cell. Until then the checks print FAIL;
  that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1–3

Phase 4 turns notebook code into software other people install. If one of these
feels shaky, re-read the unit before going on.

| Earlier idea | Unit | What Phase 4 adds |
| --- | --- | --- |
| Virtual environments, `uv init`, `uv add`, `uv run` | 0.2–0.3 | The whole packaging toolchain: wheels, build backends, lockfiles, workspaces, publishing |
| Functions, type hints on every signature, `Callable` | 1.10–1.12 | Hints become a checked contract: mypy and pyright, generics, `ParamSpec`, `@overload` |
| Decorators (`timer`, `retry`) | 1.12 | Decorators that keep their wrapped function's signature for the type checker |
| Exceptions, `logging` basics | 1.14 | Testing, debugging and logging configuration (Unit 4.3) |
| Protocols, ABCs, dataclasses, enums | 2.3, 2.6 | Structural typing checked statically; `TypedDict`, `Literal`, discriminated unions |
| `Money`, the inventory model | 2.4, Project 2 | Typed to pass `mypy --strict` (Lab 4.2) |
| Packages, `sys.path`, `importlib.metadata`, entry points | 3.3 | `src/` layout, editable installs, `[project.scripts]`, single-sourced `__version__` |
| `pyinspect` (a package without packaging metadata) | Project 3 | The Phase 4 Project makes code like this installable |

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 4.1 — Package Management and Project Structure

**Objectives**

- Read a wheel filename and know whether it installs on a given interpreter; tell
  wheels from sdists and build frontends from build backends.
- Use pip's resolver, editable installs, constraints files and cache, and explain why
  `pip freeze` is not a lockfile.
- Drive a project end to end with uv: `init`, `add`, `lock`, `sync`, `run`, `tool`,
  inline-script metadata, workspaces, `export`, `build` and `publish`.
- Write every table of a `pyproject.toml` from memory, and PEP 440 version
  specifiers that mean what you intend.
- Choose a pinning strategy (application vs library), a project layout (`src/`) and a
  release scheme (SemVer or CalVer); keep the supply chain safe.

**JS bridge: npm and `package.json`**

| npm / Node.js | Python | The difference that bites |
| --- | --- | --- |
| registry.npmjs.org | PyPI (pypi.org) | A release holds several files: one sdist plus wheels per platform |
| `package.json` | `pyproject.toml` | One file for metadata, build config **and** tool settings (ruff, mypy, pytest) |
| `dependencies` | `[project] dependencies` | Same idea |
| `devDependencies` | `[dependency-groups] dev` (PEP 735) | Groups are never published in the package metadata |
| `optionalDependencies` / `peerDependencies` | extras: `[project.optional-dependencies]` | Extras are opt-in feature sets (`pkg[report]`), not "install if it builds" |
| `node_modules/`, nested per package | One flat `site-packages/` per virtual environment | Only **one** version of each package per environment: conflicts are errors, not nested copies |
| `package-lock.json`, `pnpm-lock.yaml` | `uv.lock`, `pylock.toml` (PEP 751) | `uv.lock` is cross-platform (one lock for Linux, macOS, Windows) |
| `npm ci` | `uv sync --locked` / `--frozen` | Same promise: install exactly the lock |
| `npx tool` | `uvx tool` | Same idea; uvx caches tool environments |
| `"bin"` | `[project.scripts]` | uv/pip generate the launcher scripts |
| `"scripts"` + `npm run` | no built-in task runner; `uv run <command>` | Use a Makefile, `just`, `poe` or plain `uv run` |
| `^1.2.3` (caret) | `~=1.2` or `>=1.2,<2` | PEP 440 is not SemVer: there is no caret |
| `.tgz` tarball | wheel (built, may be platform-specific) + sdist (source) | Installing a wheel runs no code; building an sdist does |
| `"engines": {"node": ">=20"}` | `requires-python = ">=3.12"` | The resolver enforces it |
| `npm link` | editable install (`uv sync`, `pip install -e`) | Implemented with a `.pth` file or an import hook |
| `npm audit` | `pip-audit` | Same idea, different database (PyPA advisory DB / OSV) |
| npm / pnpm workspaces | uv workspaces | One lockfile for all members |
| `npm publish` | `uv publish` | Prefer trusted publishing (OIDC) over tokens |

**How the demos run.** Every command runs in a child process inside
`notebooks/_scratch/unit41/` (shown as `<scratch>`), with a scratch uv cache and the
course's own environment variables removed, so nothing touches the course project.
The setup cell builds a **local wheelhouse**, a folder of tiny demo packages
(`rupee-fmt`, `gst-calc`, `table-print`, `win-colors`, `legacy-invoice`) in several
versions. Demo projects point uv and pip at it with `no-index` + `find-links`, the
standard way to install from a folder of wheels, so every resolution is real, offline
and repeatable.

| Demo package | Versions | Depends on |
| --- | --- | --- |
| `rupee-fmt` | 1.0.0, 1.1.0, 1.2.0, 2.0.0, 2.1.0rc1 | nothing (`fmt(amount)` changes output in 2.x) |
| `gst-calc` | 1.0.0, 2.0.0 | `rupee-fmt>=1.0,<2` (1.x) or `>=2` (2.x); `win-colors` on Windows only; extra `report` → `table-print`; console script `gst` |
| `table-print`, `win-colors` | 0.1.0, 1.0.0 | nothing |
| `legacy-invoice` | 1.0.0 | `rupee-fmt<1.1` |

In [2]:
import base64
import email.parser
import hashlib
import importlib.machinery
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import shlex
import shutil
import socket
import subprocess
import sys
import sysconfig
import tarfile
import textwrap
import tomllib
import zipfile
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT41_DIR = scratch_dir / "unit41"
WHEELHOUSE = UNIT41_DIR / "wheelhouse"
UNIT41_DIR.mkdir(parents=True, exist_ok=True)
VENV_BIN = Path(sys.executable).parent
COURSE_FILES = ("pyproject.toml", "uv.lock")


def file_digest(path: Path) -> str:
    """SHA-256 of a file's bytes, as hex."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


COURSE_DIGESTS_AT_START = {name: file_digest(REPO_ROOT / name) for name in COURSE_FILES}


def tool_env(**overrides: str) -> dict[str, str]:
    """Environment for child tools: plain output, scratch caches, nothing pointing at the course."""
    inherited = {
        key: value
        for key, value in os.environ.items()
        if not key.startswith(("UV_", "PIP_", "VIRTUAL_ENV", "CONDA_", "PYTHONPATH"))
    }
    return {
        **inherited,
        "NO_COLOR": "1",
        "PYTHON_COLORS": "0",
        "UV_NO_PROGRESS": "1",
        "UV_PYTHON": "3.15",
        "UV_PYTHON_DOWNLOADS": "never",
        "UV_CACHE_DIR": str(UNIT41_DIR / "uv-cache"),
        "UV_TOOL_DIR": str(UNIT41_DIR / "uv-tools"),
        "UV_TOOL_BIN_DIR": str(UNIT41_DIR / "uv-tools-bin"),
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "PIP_NO_COLOR": "1",
        "PIP_PROGRESS_BAR": "off",
        "PIP_CACHE_DIR": str(UNIT41_DIR / "pip-cache"),
        "GIT_CEILING_DIRECTORIES": str(scratch_dir),
        **overrides,
    }


TIMING = re.compile(r"\b(in|after) \d+(\.\d+)?(ms|s|µs)\b")
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")


def tidy(text: str) -> str:
    """Show absolute paths as <scratch>, <repo> or ~ and timings as <time>; drop colour codes."""
    for folder, label in ((UNIT41_DIR, "<scratch>"), (REPO_ROOT, "<repo>"), (Path.home(), "~")):
        text = text.replace(f"file://{folder}", label).replace(str(folder), label)
    return ANSI_ESCAPE.sub("", TIMING.sub(r"\1 <time>", text))


def executable(name: str) -> str | None:
    """A path as given, the kernel's python, a tool from the course venv, or one on PATH."""
    if "/" in name:
        return name if Path(name).exists() else None
    if name == "python":
        return sys.executable
    local_tool = VENV_BIN / name
    return str(local_tool) if local_tool.exists() else shutil.which(name)


def run_tool(
    *command: str, cwd: Path = UNIT41_DIR, env: dict[str, str] | None = None, timeout: int = 300
) -> tuple[int, str]:
    """Run a CLI tool in a child process; return (exit code, tidied stdout + stderr)."""
    found = executable(command[0])
    if found is None:
        return 127, f"SKIP: {command[0]} is not installed on this machine"
    completed = subprocess.run(
        [found, *command[1:]],
        cwd=cwd,
        env=env or tool_env(),
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    return completed.returncode, tidy((completed.stdout + completed.stderr).rstrip())


def shell(
    *command: str,
    cwd: Path = UNIT41_DIR,
    env: dict[str, str] | None = None,
    keep: str | None = None,
    max_lines: int | None = None,
) -> int:
    """Print `[folder]$ command` and the tool's real output (only lines matching keep, if given)."""
    exit_code, output = run_tool(*command, cwd=cwd, env=env)
    shown = [Path(command[0]).name if "/" in command[0] else command[0], *command[1:]]
    print(f"[{tidy(str(cwd))}]$ {tidy(shlex.join(shown))}")
    lines = output.splitlines()
    if keep is not None:
        lines = [line for line in lines if re.search(keep, line)]
    if max_lines is not None and len(lines) > max_lines:
        lines = [*lines[:max_lines], f"... ({len(lines) - max_lines} more lines)"]
    if lines:
        print(textwrap.indent("\n".join(lines), "  "))
    if exit_code:
        print(f"  (exit code {exit_code})")
    return exit_code


def fresh_dir(name: str) -> Path:
    """An empty folder <scratch>/<name>, recreated on every run."""
    folder = UNIT41_DIR / name
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    return folder


def write_files(root: Path, files: dict[str, str]) -> Path:
    """Write relative path -> text (dedented) under root; return root."""
    for relative_path, text in files.items():
        target = root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(text).lstrip("\n"))
    return root


SKIPPED_PARTS = {".venv", "__pycache__", ".pytest_cache", ".mypy_cache"}


def show_tree(root: Path) -> None:
    """Print every file under root (skipping virtual environments and caches)."""
    print(f"{tidy(str(root))}/")
    for path in sorted(root.rglob("*")):
        if path.is_file() and not SKIPPED_PARTS & set(path.relative_to(root).parts):
            print(f"  {path.relative_to(root).as_posix()}")


def show_file(path: Path, *, after: str | None = None) -> None:
    """Print a text file (optionally only from the first line containing after)."""
    text = path.read_text()
    if after is not None and after in text:
        text = text[text.index(after) :]
    print(f"--- {path.name} ---\n{tidy(text.rstrip())}")


@dataclass(frozen=True)
class DemoPackage:
    """A tiny package for the local wheelhouse (never published anywhere)."""

    name: str
    version: str
    source: str
    dependencies: tuple[str, ...] = ()
    extras: dict[str, list[str]] = field(default_factory=dict)
    scripts: dict[str, str] = field(default_factory=dict)

    @property
    def module(self) -> str:
        return self.name.replace("-", "_")

    def files(self) -> dict[str, str]:
        """pyproject.toml and src/<module>/__init__.py for this package."""
        lines = [
            "[project]",
            f'name = "{self.name}"',
            f'version = "{self.version}"',
            'description = "Course demo package (never published)"',
            'requires-python = ">=3.12"',
            'license = "MIT"',
            f"dependencies = {json.dumps(list(self.dependencies))}",
        ]
        if self.extras:
            lines += ["", "[project.optional-dependencies]"]
            lines += [f"{extra} = {json.dumps(reqs)}" for extra, reqs in self.extras.items()]
        if self.scripts:
            lines += ["", "[project.scripts]"]
            lines += [f'{command} = "{target}"' for command, target in self.scripts.items()]
        lines += ["", "[build-system]", 'requires = ["uv_build>=0.13,<0.14"]']
        lines += ['build-backend = "uv_build"']
        return {
            "pyproject.toml": "\n".join(lines) + "\n",
            f"src/{self.module}/__init__.py": textwrap.dedent(self.source).lstrip("\n"),
        }

    def wheel_name(self) -> str:
        return f"{self.module}-{self.version}-py3-none-any.whl"


RUPEE_V1 = """
    def fmt(amount: float) -> str:
        return f"INR {amount:,.2f}"
"""
RUPEE_V2 = """
    def fmt(amount: float, symbol: str = "\u20b9") -> str:
        return f"{symbol}{amount:,.2f}"
"""
GST_SOURCE = """
    import sys

    from rupee_fmt import fmt


    def with_gst(amount: float, rate: float = 0.18) -> float:
        return round(amount * (1 + rate), 2)


    def main() -> None:
        print(fmt(with_gst(float(sys.argv[1]))))
"""
WHEELHOUSE_PACKAGES = [
    *(DemoPackage("rupee-fmt", version, RUPEE_V1) for version in ("1.0.0", "1.1.0", "1.2.0")),
    *(DemoPackage("rupee-fmt", version, RUPEE_V2) for version in ("2.0.0", "2.1.0rc1")),
    DemoPackage(
        "table-print", "0.1.0", "def table(rows):\n    return '\\n'.join(map(str, rows))\n"
    ),
    DemoPackage("win-colors", "1.0.0", "RED = '\\x1b[31m'\n"),
    DemoPackage(
        "gst-calc",
        "1.0.0",
        GST_SOURCE,
        dependencies=("rupee-fmt>=1.0,<2", "win-colors>=1.0; sys_platform == 'win32'"),
        extras={"report": ["table-print>=0.1"]},
        scripts={"gst": "gst_calc:main"},
    ),
    DemoPackage(
        "gst-calc",
        "2.0.0",
        GST_SOURCE,
        dependencies=("rupee-fmt>=2",),
        extras={"report": ["table-print>=0.1"]},
        scripts={"gst": "gst_calc:main"},
    ),
    DemoPackage("legacy-invoice", "1.0.0", "TEMPLATE = 'plain'\n", dependencies=("rupee-fmt<1.1",)),
]


def publish_to_wheelhouse(package: DemoPackage) -> None:
    """Build one demo package (sdist + wheel) into the local wheelhouse with `uv build`."""
    source_root = write_files(
        fresh_dir(f"sources/{package.name}-{package.version}"), package.files()
    )
    exit_code, output = run_tool(
        "uv", "build", "--offline", "--out-dir", str(WHEELHOUSE), cwd=source_root
    )
    if exit_code:
        raise RuntimeError(f"uv build failed for {package.name} {package.version}:\n{output}")


def remove_from_wheelhouse(name: str, version: str) -> None:
    """Delete one release (wheel and sdist) from the local wheelhouse."""
    for path in WHEELHOUSE.glob(f"{name.replace('-', '_')}-{version}[-.]*"):
        path.unlink()


def build_wheelhouse() -> Path:
    """Build every demo package that is missing from the wheelhouse; return its folder."""
    WHEELHOUSE.mkdir(parents=True, exist_ok=True)
    for package in WHEELHOUSE_PACKAGES:
        if not (WHEELHOUSE / package.wheel_name()).exists():
            publish_to_wheelhouse(package)
    return WHEELHOUSE


def network_available(host: str = "pypi.org", port: int = 443, timeout: float = 3.0) -> bool:
    """Can we open a TCP connection to host? (Only two cells in this unit need the network.)"""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


build_wheelhouse()
print(f"Python {platform.python_version()}; {run_tool('uv', '--version')[1]}")
print(f"wheelhouse: {sorted(path.name for path in WHEELHOUSE.glob('*.whl'))}")

Python 3.15.1; uv 0.13.5 (x86_64-unknown-linux-gnu)
wheelhouse: ['gst_calc-1.0.0-py3-none-any.whl', 'gst_calc-2.0.0-py3-none-any.whl', 'legacy_invoice-1.0.0-py3-none-any.whl', 'rupee_fmt-1.0.0-py3-none-any.whl', 'rupee_fmt-1.1.0-py3-none-any.whl', 'rupee_fmt-1.2.0-py3-none-any.whl', 'rupee_fmt-2.0.0-py3-none-any.whl', 'rupee_fmt-2.1.0rc1-py3-none-any.whl', 'table_print-0.1.0-py3-none-any.whl', 'win_colors-1.0.0-py3-none-any.whl']


### 4.1.1 The ecosystem

**PyPI** (the Python Package Index) is Python's npm registry. A *project* has
*releases* (versions), and each release has one or more **distribution files**:

| File | Extension | What it is | Installing it |
| --- | --- | --- | --- |
| **sdist** (source distribution) | `.tar.gz` | Source code + `pyproject.toml` + `PKG-INFO` | Must be **built** first: runs the project's build backend, maybe a C compiler |
| **wheel** (built distribution) | `.whl` (a zip) | Files laid out ready to copy into `site-packages`, plus a `.dist-info/` folder | Unzip and copy: **no code runs** |

Installers prefer a wheel that matches the interpreter and fall back to the sdist.
Which wheel matches is decided by the three **tags** in its filename:

```text
{name}-{version}(-{build})?-{python tag}-{abi tag}-{platform tag}.whl
numpy-2.3.4-cp315-cp315-manylinux_2_28_x86_64.whl
            ^^^^^ ^^^^^ ^^^^^^^^^^^^^^^^^^^^^
            |     |     platform: Linux with glibc >= 2.28 on x86-64
            |     ABI: CPython 3.15's C ABI (cp315t = free-threaded build)
            interpreter: CPython 3.15
```

| Tag example | Meaning |
| --- | --- |
| `py3-none-any` | Pure Python, any Python 3, no C ABI, any OS: one wheel fits all |
| `cp315-cp315-manylinux_2_28_x86_64` | Compiled for CPython 3.15, Linux glibc ≥ 2.28, x86-64 |
| `cp315-cp315t-…` | The **free-threaded** (no-GIL) 3.15 build: the `t` ABI flag (Unit 3.1.5) |
| `cp312-abi3-…` | Uses only the **stable ABI** (`abi3`): one wheel for CPython 3.12 and every later version |
| `…-musllinux_1_2_aarch64` | Linux with musl libc (Alpine) on ARM64 |
| `…-macosx_11_0_arm64`, `…-win_amd64` | macOS on Apple Silicon, 64-bit Windows |

The next cell parses some (illustrative) wheel filenames with `packaging`, the
library pip and uv's ideas come from, and asks whether each would install **here**.

In [3]:
if importlib.util.find_spec("packaging") is None:
    print("SKIP: the packaging library is not installed")
else:
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename

    supported_here = set(sys_tags())
    example_wheels = [
        "requests-2.32.5-py3-none-any.whl",
        "numpy-2.3.4-cp315-cp315-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",
        "numpy-2.3.4-cp315-cp315t-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",
        "cryptography-46.0.3-cp311-abi3-manylinux_2_34_x86_64.whl",
        "pydantic_core-2.50.4-cp315-cp315-musllinux_1_2_aarch64.whl",
        "numpy-2.3.4-cp314-cp314-win_amd64.whl",
    ]
    for filename in example_wheels:
        name, version, _build, wheel_tags = parse_wheel_filename(filename)
        tag = sorted(wheel_tags, key=str)[0]
        installs = bool(wheel_tags & supported_here)
        print(
            f"{name:<14} {str(version):<8} {tag.interpreter:<6} {tag.abi:<7} "
            f"{tag.platform:<26} installs here: {installs}"
        )

requests       2.32.5   py3    none    any                        installs here: True
numpy          2.3.4    cp315  cp315   manylinux_2_27_x86_64      installs here: True
numpy          2.3.4    cp315  cp315t  manylinux_2_27_x86_64      installs here: False
cryptography   46.0.3   cp311  abi3    manylinux_2_34_x86_64      installs here: True
pydantic-core  2.50.4   cp315  cp315   musllinux_1_2_aarch64      installs here: False
numpy          2.3.4    cp314  cp314   win_amd64                  installs here: False


A wheel installs when **any** of its tags is in the interpreter's list of supported
tags, which is ordered from most to least specific. The installer picks the wheel
whose best tag comes first in this list. Here is the start of that list for this
kernel:

In [4]:
if importlib.util.find_spec("packaging") is not None:
    from packaging.tags import sys_tags

    all_tags = [str(tag) for tag in sys_tags()]
    print(f"{len(all_tags)} supported tags; the first eight and the last:")
    for tag in [*all_tags[:8], "...", all_tags[-1]]:
        print(f"  {tag}")
print(f"{sys.implementation.cache_tag=}  {sys.abiflags=}")
print(f"{sysconfig.get_config_var('SOABI')=}")

1437 supported tags; the first eight and the last:
  cp315-cp315-linux_x86_64
  cp315-cp315-manylinux_2_43_x86_64
  cp315-cp315-manylinux_2_42_x86_64
  cp315-cp315-manylinux_2_41_x86_64
  cp315-cp315-manylinux_2_40_x86_64
  cp315-cp315-manylinux_2_39_x86_64
  cp315-cp315-manylinux_2_38_x86_64
  cp315-cp315-manylinux_2_37_x86_64
  ...
  py30-none-any
sys.implementation.cache_tag='cpython-315'  sys.abiflags=''
sysconfig.get_config_var('SOABI')='cpython-315-x86_64-linux-gnu'


**manylinux.** Compiled Linux wheels must run on many distributions. A
`manylinux_2_28` wheel promises it only needs glibc 2.28 or newer, so it runs on any
distribution at least that recent. Your glibc version decides which `manylinux_2_X`
tags appear in the list above (`musllinux` is the same idea for Alpine's musl libc).

**abi3, the stable ABI.** An extension compiled against the *limited C API* uses only
the stable ABI and is tagged `cp3X-abi3`: one wheel serves that CPython version and
all later ones. Its files end in `.abi3.so`. The free-threaded build has a different
object layout, so it has its own ABI tag (`cp315t`) and suffixes; 3.15 adds `abi3t`,
a stable ABI for free-threaded builds (*New in 3.15*). The next cell asks
this kernel and, if one is installed, the uv-managed free-threaded 3.15 (Unit 3.1.5)
in a child process.

In [5]:
def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None."""
    env = {key: value for key, value in tool_env().items() if key != "UV_PYTHON"}
    found = subprocess.run(
        [executable("uv"), "python", "find", "3.15t"], capture_output=True, text=True, env=env
    )
    return found.stdout.strip() if found.returncode == 0 else None


ABI_PROBE = """
import importlib.machinery, platform, sys, sysconfig
soabi = sysconfig.get_config_var("SOABI")
print(platform.python_version(), f"abiflags={sys.abiflags!r}", f"SOABI={soabi}")
print("  extension suffixes:", importlib.machinery.EXTENSION_SUFFIXES)
"""
print("default build:", run_tool("python", "-c", ABI_PROBE)[1])
libc_name, libc_version = platform.libc_ver()
print(f"  C library: {libc_name} {libc_version} -> manylinux_2_X tags up to that glibc")
free_threaded_python = find_free_threaded_python()
if free_threaded_python is None:
    print("SKIP: no free-threaded 3.15 installed (uv python install 3.15t)")
else:
    print("free-threaded build:", run_tool(free_threaded_python, "-c", ABI_PROBE)[1])

default build: 3.15.1 abiflags='' SOABI=cpython-315-x86_64-linux-gnu
  extension suffixes: ['.cpython-315-x86_64-linux-gnu.so', '.abi3-x86_64-linux-gnu.so', '.abi3.so', '.abi3t-x86_64-linux-gnu.so', '.abi3t.so', '.so']
  C library: glibc 2.43 -> manylinux_2_X tags up to that glibc
free-threaded build: 3.15.1 abiflags='t' SOABI=cpython-315t-x86_64-linux-gnu
  extension suffixes: ['.cpython-315t-x86_64-linux-gnu.so', '.abi3t-x86_64-linux-gnu.so', '.abi3t.so', '.so']


**Build frontends and backends (PEP 517 / PEP 518).** Building a wheel is split
between two programs that talk through a small, standard Python API:

| Role | Job | Examples |
| --- | --- | --- |
| **Frontend** | Reads `[build-system]`, creates an isolated build environment, installs `requires`, calls the backend's hooks | pip, uv, `python -m build` |
| **Backend** | Turns the source tree into a wheel or sdist | `setuptools`, `hatchling`, `flit-core`, `pdm-backend`, `poetry-core`, `uv_build`, `maturin` (Rust), `scikit-build-core` (CMake) |

PEP 518 added `[build-system] requires` (what the build needs installed), PEP 517
the hooks (`build_wheel`, `build_sdist`, `get_requires_for_build_wheel`, …) and PEP
660 the editable-install hook (`build_editable`). Any frontend can build any project.

```mermaid
flowchart LR
    F["frontend (uv, pip)"] -->|"reads [build-system]"| P[pyproject.toml]
    F -->|"creates isolated env, installs requires"| E[build env]
    F -->|"calls build_wheel() / build_sdist()"| B["backend (uv_build, hatchling, ...)"]
    B -->|"writes"| W[".whl / .tar.gz"]
    F -->|"installs the wheel"| S[site-packages]
```

<details><summary>ASCII fallback</summary>

```text
frontend (uv, pip) --reads--> pyproject.toml [build-system]
         |--creates isolated env + installs `requires`
         |--calls build_wheel()/build_sdist()--> backend --writes--> .whl/.tar.gz
         '--installs the wheel--> site-packages
```

</details>

**Under the hood: a complete build backend in 40 lines.** A backend is just a module
with those functions. The next cell writes one (`tiny_backend.py`), a project that
uses it through `backend-path` (an in-tree backend, so `requires` is empty), and
builds it with uv as the frontend.

In [6]:
TINY_BACKEND = r"""
    # A minimal PEP 517 + PEP 660 build backend (pure Python, py3-none-any only).
    import base64, hashlib, io, os, tarfile, tomllib, zipfile
    from pathlib import Path

    PROJECT = tomllib.loads(Path("pyproject.toml").read_text())["project"]
    NAME, VERSION = PROJECT["name"], PROJECT["version"]
    MODULE = NAME.replace("-", "_")
    DIST_INFO = f"{MODULE}-{VERSION}.dist-info"
    METADATA = f"Metadata-Version: 2.4\nName: {NAME}\nVersion: {VERSION}\n"

    def _record_line(path, data):
        digest = base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=")
        return f"{path},sha256={digest.decode()},{len(data)}"

    def _wheel(directory, files):
        files = {**files, f"{DIST_INFO}/METADATA": METADATA.encode(),
                 f"{DIST_INFO}/WHEEL": b"Wheel-Version: 1.0\nGenerator: tiny\n"
                 b"Root-Is-Purelib: true\nTag: py3-none-any\n"}
        record = [_record_line(path, data) for path, data in files.items()]
        name = f"{MODULE}-{VERSION}-py3-none-any.whl"
        with zipfile.ZipFile(Path(directory) / name, "w") as wheel:
            for path, data in files.items():
                wheel.writestr(path, data)
            wheel.writestr(f"{DIST_INFO}/RECORD", "\n".join(record) + f"\n{DIST_INFO}/RECORD,,\n")
        return name

    def get_requires_for_build_wheel(config_settings=None):
        return []

    def build_wheel(wheel_directory, config_settings=None, metadata_directory=None):
        sources = {p.as_posix(): p.read_bytes() for p in Path(MODULE).rglob("*.py")}
        return _wheel(wheel_directory, sources)

    def build_editable(wheel_directory, config_settings=None, metadata_directory=None):
        # PEP 660: the "wheel" holds only a .pth file pointing at the source folder
        return _wheel(wheel_directory, {f"_{MODULE}_editable.pth": (os.getcwd() + "\n").encode()})

    def build_sdist(sdist_directory, config_settings=None):
        name = f"{MODULE}-{VERSION}.tar.gz"
        with tarfile.open(Path(sdist_directory) / name, "w:gz") as sdist:
            sources = ["pyproject.toml", "tiny_backend.py", *map(str, Path(MODULE).rglob("*.py"))]
            for path in sources:
                sdist.add(path, f"{MODULE}-{VERSION}/{path}")
            info = tarfile.TarInfo(f"{MODULE}-{VERSION}/PKG-INFO")
            info.size = len(METADATA)
            sdist.addfile(info, io.BytesIO(METADATA.encode()))
        return name
"""
tiny_root = write_files(
    fresh_dir("tiny-tax"),
    {
        "pyproject.toml": """
            [build-system]
            requires = []                    # nothing to install: the backend is in-tree
            build-backend = "tiny_backend"   # module that defines the hooks
            backend-path = ["."]             # ...found in the project folder itself

            [project]
            name = "tiny-tax"
            version = "0.1.0"
        """,
        "tiny_backend.py": TINY_BACKEND,
        "tiny_tax/__init__.py": "GST_RATE = 0.18\n",
    },
)
shell("uv", "build", "--offline", cwd=tiny_root)

[<scratch>/tiny-tax]$ uv build --offline
  Building source distribution...
  Successfully built dist/tiny_tax-0.1.0.tar.gz
  Successfully built dist/tiny_tax-0.1.0-py3-none-any.whl


0

uv (the frontend) built the sdist first, then a wheel *from* the unpacked sdist, which
proves the sdist is complete. The frontend's side of the protocol is just "import
the backend and call a hook". Calling the hooks yourself in a child interpreter shows
there is nothing more to it, and lists what each archive contains:

In [7]:
HOOK_CALLER = """
import sys, tarfile, zipfile
sys.path.insert(0, ".")          # backend-path = ["."]
import tiny_backend as backend    # build-backend = "tiny_backend"
print("get_requires_for_build_wheel ->", backend.get_requires_for_build_wheel())
wheel = backend.build_wheel("hook-dist")
sdist = backend.build_sdist("hook-dist")
print("build_wheel ->", wheel, zipfile.ZipFile(f"hook-dist/{wheel}").namelist())
print("build_sdist ->", sdist, tarfile.open(f"hook-dist/{sdist}").getnames())
"""
(tiny_root / "hook-dist").mkdir(exist_ok=True)
print(run_tool("python", "-c", HOOK_CALLER, cwd=tiny_root)[1])

get_requires_for_build_wheel -> []
build_wheel -> tiny_tax-0.1.0-py3-none-any.whl ['tiny_tax/__init__.py', 'tiny_tax-0.1.0.dist-info/METADATA', 'tiny_tax-0.1.0.dist-info/WHEEL', 'tiny_tax-0.1.0.dist-info/RECORD']
build_sdist -> tiny_tax-0.1.0.tar.gz ['tiny_tax-0.1.0/pyproject.toml', 'tiny_tax-0.1.0/tiny_backend.py', 'tiny_tax-0.1.0/tiny_tax/__init__.py', 'tiny_tax-0.1.0/PKG-INFO']


### 4.1.2 pip

pip is the reference installer that ships with CPython (through `ensurepip`). uv
projects do not include it, so the demo makes a classic virtual environment with
`python -m venv`, which installs pip from the wheel bundled with the interpreter
(no network needed). pip reads its options from `PIP_*` environment variables, so
the demo sets `PIP_NO_INDEX=1` and `PIP_FIND_LINKS=<wheelhouse>` once.

| npm | pip |
| --- | --- |
| `npm install pkg` | `pip install pkg` |
| `npm install` (from package.json) | `pip install -r requirements.txt` or `pip install .` |
| `npm ls` | `pip list`, `pip show pkg`, `pip freeze` |
| `npm link` | `pip install -e path` |

In [8]:
PIP_VENV = UNIT41_DIR / "pip-venv"
shutil.rmtree(PIP_VENV, ignore_errors=True)
print(run_tool("python", "-m", "venv", str(PIP_VENV))[1] or "venv created")
PIP_PYTHON = str(PIP_VENV / "bin" / "python")
PIP_ENV = tool_env(PIP_NO_INDEX="1", PIP_FIND_LINKS=str(WHEELHOUSE))


def pip(*arguments: str, cwd: Path = UNIT41_DIR, **options: object) -> int:
    """Run pip inside the scratch venv against the local wheelhouse."""
    return shell(PIP_PYTHON, "-m", "pip", *arguments, cwd=cwd, env=PIP_ENV, **options)


pip("--version")
pip("list")

venv created
[<scratch>]$ python -m pip --version
  pip 26.2.2 from <scratch>/pip-venv/lib/python3.15/site-packages/pip (python 3.15)
[<scratch>]$ python -m pip list
  Package Version
  ------- -------
  pip     26.2.2


0

**The resolver.** Since pip 20.3, pip uses a *backtracking* resolver: it picks
candidate versions, and when a later requirement conflicts it backs up and tries
older ones. Below, `gst-calc 1.0.0` accepts `rupee-fmt>=1.0,<2` and `legacy-invoice`
needs `rupee-fmt<1.1`, so the only answer is `rupee-fmt 1.0.0`. `--dry-run` resolves
without installing.

In [9]:
pip("install", "--dry-run", "gst-calc==1.0.0", "legacy-invoice", keep="Would install")

[<scratch>]$ python -m pip install --dry-run gst-calc==1.0.0 legacy-invoice
  Would install gst-calc-1.0.0 legacy-invoice-1.0.0 rupee-fmt-1.0.0


0

`gst-calc 2.0.0` needs `rupee-fmt>=2`, which can never satisfy `legacy-invoice`'s
`<1.1`. npm would nest two copies; Python has one flat environment, so the resolver
reports a conflict and installs nothing:

In [10]:
pip(
    "install",
    "--dry-run",
    "gst-calc==2.0.0",
    "legacy-invoice",
    keep=r"ERROR|requires|conflict|depends",
)

[<scratch>]$ python -m pip install --dry-run gst-calc==2.0.0 legacy-invoice
  The conflict is caused by:
      gst-calc 2.0.0 depends on rupee-fmt>=2
      legacy-invoice 1.0.0 depends on rupee-fmt<1.1
  Additionally, some packages in these conflicts have no matching distributions available for your environment:
  2. remove package versions to allow pip to attempt to solve the dependency conflict
  ERROR: Cannot install gst-calc==2.0.0 and legacy-invoice==1.0.0 because these package versions have conflicting dependencies.
  ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  (exit code 1)


1

**Editable installs** (`pip install -e path`, PEP 660). The project is "installed"
but its code stays in the working folder, so edits show up without reinstalling,
like `npm link`. pip asks the backend's `build_editable` hook for a wheel; our tiny
backend's editable wheel holds a single `.pth` file naming the source folder, and
`site` adds that folder to `sys.path` at startup (Unit 3.3.5).

In [11]:
pip("install", "-e", str(tiny_root), keep="Successfully|ERROR")
site_packages = next(PIP_VENV.glob("lib/python3*/site-packages"))
for pth_file in site_packages.glob("_tiny*.pth"):
    print(f"{pth_file.name}: {tidy(pth_file.read_text().strip())}")
WHERE = "import tiny_tax; print(tiny_tax.__file__, tiny_tax.GST_RATE)"
print("before edit:", run_tool(PIP_PYTHON, "-c", WHERE)[1])
(tiny_root / "tiny_tax" / "__init__.py").write_text("GST_RATE = 0.12\n")
print("after edit: ", run_tool(PIP_PYTHON, "-c", WHERE)[1], "(no reinstall)")

[<scratch>]$ python -m pip install -e <scratch>/tiny-tax
  Successfully built tiny-tax
_tiny_tax_editable.pth: <scratch>/tiny-tax
before edit: <scratch>/tiny-tax/tiny_tax/__init__.py 0.18
after edit:  <scratch>/tiny-tax/tiny_tax/__init__.py 0.12 (no reinstall)


**`pip freeze` vs a lockfile.** `pip freeze` prints what is installed *right now*
in requirements format. It is a snapshot, not a lockfile:

| `pip freeze > requirements.txt` | A real lockfile (`uv.lock`, `pylock.toml`) |
| --- | --- |
| Only this machine's platform and Python | Resolved for every platform and marker |
| No hashes (unless you add them) | Hashes for every file |
| Mixes runtime and dev tools, and anything installed by hand | Generated from declared dependencies and groups |
| No record of *why* a package is there | Records the dependency graph |
| Editable installs appear as paths | Editable sources recorded properly |

In [12]:
pip("install", "gst-calc==1.0.0", keep="Successfully")
pip("freeze")

[<scratch>]$ python -m pip install gst-calc==1.0.0
[<scratch>]$ python -m pip freeze
  gst-calc==1.0.0
  rupee-fmt==1.2.0
  # Editable install with no version control (tiny-tax==0.1.0)
  -e <scratch>/tiny-tax


0

**Constraints files** (`-c constraints.txt`) cap versions *without* asking for
anything to be installed: a constraint only matters if something else pulls that
package in. Teams use one shared constraints file to keep many services on the same
versions. Here a constraint holds `rupee-fmt` below 1.2 even though `gst-calc`
allows up to `<2`:

In [13]:
(UNIT41_DIR / "constraints.txt").write_text("rupee-fmt<1.2\ntable-print==0.1.0\n")
pip(
    "install",
    "--dry-run",
    "--ignore-installed",
    "-c",
    "constraints.txt",
    "gst-calc==1.0.0",
    keep="Would install",
)

[<scratch>]$ python -m pip install --dry-run --ignore-installed -c constraints.txt gst-calc==1.0.0
  Would install gst-calc-1.0.0 rupee-fmt-1.1.0


0

**The cache.** pip caches downloaded files (HTTP cache) and wheels it built from
sdists, so a second install is fast. `pip cache dir / info / list / purge` manage
it. The demo's `PIP_CACHE_DIR` points into the scratch folder; your real one is
usually `~/.cache/pip`. Installing from local wheels stores nothing, so the cache is
empty. uv's cache (next section) works differently.

In [14]:
pip("cache", "dir")
pip("cache", "info")

[<scratch>]$ python -m pip cache dir
  <scratch>/pip-cache
[<scratch>]$ python -m pip cache info
  Package index page cache location (pip v23.3+): <scratch>/pip-cache/http-v2
  Package index page cache location (older pips): <scratch>/pip-cache/http
  Package index page cache size: 0 bytes
  Number of HTTP files: 0
  Locally built wheels location: <scratch>/pip-cache/wheels
  Locally built wheels size: 0 bytes
  Number of locally built wheels: 0


0

### 4.1.3 uv in depth

uv (from Astral, written in Rust) replaces pip, pip-tools, virtualenv, pipx, pyenv
and twine with one fast binary. You met `uv init`, `uv add` and `uv run` in Unit
0.3; this section covers the rest.

**The global cache.** uv keeps one cache per machine (`uv cache dir`). Installing a
package into a virtual environment **hardlinks** (or copy-on-write clones) files
out of the cache instead of copying them, like pnpm's content-addressed store. Ten
projects using the same package share one copy on disk. The first command below
asks uv for your real cache location; the demos use a scratch cache.

In [15]:
real_env = {key: value for key, value in tool_env().items() if key != "UV_CACHE_DIR"}
shell("uv", "cache", "dir", env=real_env)
shell("uv", "cache", "dir")
shell("uv", "cache", "size", "--human", keep="iB$")  # experimental command in uv 0.13

[<scratch>]$ uv cache dir
  ~/.cache/uv
[<scratch>]$ uv cache dir
  <scratch>/uv-cache
[<scratch>]$ uv cache size --human
  9.8MiB


0

**`uv init`: three kinds of project.** All three create a `pyproject.toml`; they
differ in whether the project is *built and installed* into its own environment.

| Command | Layout | `[build-system]` | Use it for |
| --- | --- | --- | --- |
| `uv init --app --no-package` | `main.py` at the root | none (a *virtual* project: only its dependencies are installed) | Scripts, notebooks, web services you never `pip install` |
| `uv init --app` (packaged on this uv) or `--package` | `src/<name>/__init__.py` + `[project.scripts]` | yes | CLIs and apps that need an entry point |
| `uv init --lib` | `src/<name>/__init__.py` + `py.typed` | yes | Libraries others import |

The course itself is a virtual project (`uv init --bare`, Unit 0.3). Every `uv init`
below passes `--no-workspace` (otherwise uv would offer to add the scratch project
to an enclosing workspace), `--author-from none` (otherwise uv copies your git name
and e-mail into the file) and `--vcs none`.

In [16]:
UV_INIT = ("uv", "init", "--no-workspace", "--author-from", "none", "--vcs", "none", "--no-readme")
init_root = fresh_dir("init-demo")
for kind, project in [
    ("--app --no-package", "shop-app"),
    ("--app", "shop-cli"),
    ("--package", "invoicing-cli"),
    ("--lib", "price-lib"),
]:
    shell(*UV_INIT, *kind.split(), project, cwd=init_root)
show_tree(init_root)

[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --app --no-package shop-app
  Initialized project `shop-app` at `<scratch>/init-demo/shop-app`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --app shop-cli
  Initialized project `shop-cli` at `<scratch>/init-demo/shop-cli`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --package invoicing-cli
  Initialized project `invoicing-cli` at `<scratch>/init-demo/invoicing-cli`
[<scratch>/init-demo]$ uv init --no-workspace --author-from none --vcs none --no-readme --lib price-lib
  Initialized project `price-lib` at `<scratch>/init-demo/price-lib`
<scratch>/init-demo/
  invoicing-cli/.python-version
  invoicing-cli/pyproject.toml
  invoicing-cli/src/invoicing_cli/__init__.py
  price-lib/.python-version
  price-lib/pyproject.toml
  price-lib/src/price_lib/__init__.py
  price-lib/src/price_lib/py.typed
  shop-app/.python-version


Compare the generated `pyproject.toml` files of a virtual app and a library:

In [17]:
show_file(init_root / "shop-app" / "pyproject.toml")
show_file(init_root / "price-lib" / "pyproject.toml")
show_file(init_root / "invoicing-cli" / "src" / "invoicing_cli" / "__init__.py")

--- pyproject.toml ---
[project]
name = "shop-app"
version = "0.1.0"
description = "Add your description here"
requires-python = ">=3.15"
dependencies = []
--- pyproject.toml ---
[project]
name = "price-lib"
version = "0.1.0"
description = "Add your description here"
requires-python = ">=3.15"
dependencies = []

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"
--- __init__.py ---
def main() -> None:
    print("Hello from invoicing-cli!")


**A working project: `invoicing`.** The rest of this section drives one packaged
project. First it is pointed at the wheelhouse through a `[tool.uv]` table (the
same settings a company uses for a private index). Before locking, the newest
`rupee-fmt` 1.x (1.2.0) is removed from the wheelhouse; it is "released" again
later to show how upgrades work.

In [18]:
projects_root = fresh_dir("projects")
shell(*UV_INIT, "--package", "invoicing", cwd=projects_root)
invoicing = projects_root / "invoicing"
with (invoicing / "pyproject.toml").open("a") as pyproject_file:
    pyproject_file.write(
        "\n[tool.uv]\nno-index = true               # never contact PyPI\n"
        'find-links = ["../../wheelhouse"]   # resolve from the local wheelhouse\n'
    )
remove_from_wheelhouse("rupee-fmt", "1.2.0")
print(sorted(path.name for path in WHEELHOUSE.glob("rupee_fmt*.whl")))

[<scratch>/projects]$ uv init --no-workspace --author-from none --vcs none --no-readme --package invoicing
  Initialized project `invoicing` at `<scratch>/projects/invoicing`
['rupee_fmt-1.0.0-py3-none-any.whl', 'rupee_fmt-1.1.0-py3-none-any.whl', 'rupee_fmt-2.0.0-py3-none-any.whl', 'rupee_fmt-2.1.0rc1-py3-none-any.whl']


**`uv add` with version specifiers.** `uv add` edits `pyproject.toml`, re-locks and
syncs the environment in one step. Quote specifiers so the shell does not treat
`<` or `>` as redirection. The four flags decide *where* the requirement goes:

| Command | Written to | Installed by `uv sync`? | Published in the wheel? |
| --- | --- | --- | --- |
| `uv add "gst-calc~=1.0"` | `[project] dependencies` | yes | yes (`Requires-Dist`) |
| `uv add --dev pkg` | `[dependency-groups] dev` | yes (default group) | no |
| `uv add --group docs pkg` | `[dependency-groups] docs` | only with `--group docs` | no |
| `uv add --optional report pkg` | `[project.optional-dependencies] report` | only with `--extra report` | yes, as an extra |

In [19]:
shell("uv", "add", "gst-calc~=1.0", cwd=invoicing)
shell("uv", "add", "--dev", "rupee-fmt>=1.1", cwd=invoicing)
shell("uv", "add", "--group", "docs", "table-print", cwd=invoicing)
shell("uv", "add", "--optional", "report", "table-print>=0.1", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv add 'gst-calc~=1.0'
  Using CPython 3.15.1
  Creating virtual environment at: .venv
  Resolved 4 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Installed 3 packages in <time>
   + gst-calc==1.0.0
   + invoicing==0.1.0 (from <scratch>/projects/invoicing)
   + rupee-fmt==1.1.0
[<scratch>/projects/invoicing]$ uv add --dev 'rupee-fmt>=1.1'
  Resolved 4 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)
[<scratch>/projects/invoicing]$ uv add --group docs table-print
  Resolved 5 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepare

0

The four tables uv wrote (note that `uv add --dev rupee-fmt>=1.1` kept the user's
specifier, while a bare name gets a lower bound from the version it resolved):

In [20]:
show_file(invoicing / "pyproject.toml", after="dependencies")

--- pyproject.toml ---
dependencies = [
    "gst-calc~=1.0",
]

[project.scripts]
invoicing = "invoicing:main"

[project.optional-dependencies]
report = [
    "table-print>=0.1",
]

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"

[tool.uv]
no-index = true               # never contact PyPI
find-links = ["../../wheelhouse"]   # resolve from the local wheelhouse

[dependency-groups]
dev = [
    "rupee-fmt>=1.1",
]
docs = [
    "table-print>=0.1.0",
]


**`uv lock --upgrade-package`.** A lockfile is *stable*: `uv lock` keeps every
locked version that still satisfies `pyproject.toml`, even when newer ones exist.
Below, `rupee-fmt 1.2.0` is "released" into the wheelhouse. Plain `uv lock` changes
nothing; `uv lock --upgrade-package rupee-fmt` moves just that package (and only as
far as the specifiers allow, so 2.x is never chosen). `uv lock --upgrade` would move
everything.

In [21]:
def locked_rupee() -> str:
    """The rupee-fmt version recorded in invoicing/uv.lock."""
    lock_text = (invoicing / "uv.lock").read_text()
    return re.search(r'name = "rupee-fmt"\nversion = "([^"]+)"', lock_text)[1]


print(f"locked before the release: rupee-fmt {locked_rupee()}")
publish_to_wheelhouse(next(p for p in WHEELHOUSE_PACKAGES if p.version == "1.2.0"))
shell("uv", "lock", cwd=invoicing)
print(f"after plain `uv lock`:     rupee-fmt {locked_rupee()}")
shell("uv", "lock", "--upgrade-package", "rupee-fmt", cwd=invoicing)
print(f"after --upgrade-package:   rupee-fmt {locked_rupee()}")

locked before the release: rupee-fmt 1.1.0
[<scratch>/projects/invoicing]$ uv lock
  Resolved 5 packages in <time>
after plain `uv lock`:     rupee-fmt 1.1.0


[<scratch>/projects/invoicing]$ uv lock --upgrade-package rupee-fmt
  Resolved 5 packages in <time>
  Updated rupee-fmt v1.1.0 -> v1.2.0
after --upgrade-package:   rupee-fmt 1.2.0


**`uv sync --locked` vs `--frozen`.** Both promise not to change the lockfile, which
is what CI and Docker builds want:

| Flag | If `pyproject.toml` and `uv.lock` disagree | Use it |
| --- | --- | --- |
| (none) | Re-locks, then installs | Day-to-day development |
| `--locked` | **Fails** with an error | CI: catch a forgotten `uv lock` |
| `--frozen` | Installs the lock **as is**, without checking `pyproject.toml` | Docker layers that copy only `uv.lock`, or when resolution is impossible here |

The demo edits `pyproject.toml` by hand (adding `win-colors` for every platform)
without re-locking:

In [22]:
pyproject_path = invoicing / "pyproject.toml"
original_pyproject = pyproject_path.read_text()
pyproject_path.write_text(
    original_pyproject.replace('"gst-calc~=1.0",', '"gst-calc~=1.0",\n    "win-colors",')
)
shell("uv", "sync", "--locked", cwd=invoicing)
shell("uv", "sync", "--frozen", cwd=invoicing)
pyproject_path.write_text(original_pyproject)  # undo the hand edit
shell("uv", "sync", "--locked", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv sync --locked


  Resolved 5 packages in <time>
  error: The lockfile at `uv.lock` needs to be updated, but `--locked` was provided.

  hint: To update the lockfile, run `uv lock`.
  (exit code 1)
[<scratch>/projects/invoicing]$ uv sync --frozen
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 3 packages in <time>
  Installed 2 packages in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)
   - rupee-fmt==1.1.0
   + rupee-fmt==1.2.0
   - table-print==0.1.0


[<scratch>/projects/invoicing]$ uv sync --locked
  Resolved 5 packages in <time>
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Prepared 1 package in <time>
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
   ~ invoicing==0.1.0 (from <scratch>/projects/invoicing)


0

`uv run` syncs first, then runs a command inside the project environment. The
project's console script (`invoicing = "invoicing:main"` in `[project.scripts]`) is
installed as a command. Edit `main()` to use the dependency and run it again; no
reinstall is needed because the project itself is installed *editable*.

In [23]:
shell("uv", "run", "invoicing", cwd=invoicing)
write_files(
    invoicing / "src" / "invoicing",
    {
        "__init__.py": """
    import sys

    from gst_calc import with_gst
    from rupee_fmt import fmt


    def main() -> None:
        amount = float(sys.argv[1]) if len(sys.argv) > 1 else 100.0
        print(f"{fmt(amount)} + GST = {fmt(with_gst(amount))}")
"""
    },
)
shell("uv", "run", "invoicing", "2500", cwd=invoicing)

[<scratch>/projects/invoicing]$ uv run invoicing
  Hello from invoicing!


[<scratch>/projects/invoicing]$ uv run invoicing 2500
  INR 2,500.00 + GST = INR 2,950.00


0

**`uv tool` and `uvx`** are uv's pipx and npx. `uv tool install` puts a CLI into
its own isolated environment and links its commands into a bin folder;
`uvx` (short for `uv tool run`) runs a tool in a cached throwaway environment. The
demo redirects `UV_TOOL_DIR` and `UV_TOOL_BIN_DIR` into the scratch folder, so
nothing lands in your real `~/.local/bin`.

| npm | uv |
| --- | --- |
| `npm install -g pkg` | `uv tool install pkg` |
| `npx pkg` | `uvx pkg` (`uvx --from pkg command` when names differ) |
| `npm ls -g` | `uv tool list` |

In [24]:
shutil.rmtree(UNIT41_DIR / "uv-tools", ignore_errors=True)
shell(
    "uv",
    "tool",
    "install",
    "--offline",
    "../wheelhouse/gst_calc-1.0.0-py3-none-any.whl",
    "--no-index",
    "--find-links",
    "../wheelhouse",
    cwd=projects_root,
)
shell("uv", "tool", "list")
shell(str(UNIT41_DIR / "uv-tools-bin" / "gst"), "1000")
shell(
    "uvx",
    "--offline",
    "--no-index",
    "--find-links",
    "wheelhouse",
    "--from",
    "gst-calc==2.0.0",
    "gst",
    "1000",
)
shell("uv", "tool", "uninstall", "gst-calc")

[<scratch>/projects]$ uv tool install --offline ../wheelhouse/gst_calc-1.0.0-py3-none-any.whl --no-index --find-links ../wheelhouse
  Resolved 2 packages in <time>
  Installed 2 packages in <time>
   + gst-calc==1.0.0 (from <scratch>/wheelhouse/gst_calc-1.0.0-py3-none-any.whl)
   + rupee-fmt==1.2.0
  Installed 1 executable: gst
[<scratch>]$ uv tool list
  gst-calc v1.0.0
  - gst
[<scratch>]$ gst 1000
  INR 1,180.00


[<scratch>]$ uvx --offline --no-index --find-links wheelhouse --from gst-calc==2.0.0 gst 1000
  ₹1,180.00
[<scratch>]$ uv tool uninstall gst-calc
  Uninstalled 1 executable: gst


0

**PEP 723 inline script metadata.** A single-file script can declare its own
dependencies in a special comment block; `uv run script.py` reads it and runs the
script in a cached environment that has exactly those packages. No project, no
virtual environment to manage: ideal for one-off tools. `uv init --script` creates
the block and `uv add --script` edits it.

```python
# /// script
# requires-python = ">=3.12"
# dependencies = ["rupee-fmt>=2"]
# ///
```

In [25]:
scripts_root = fresh_dir("scripts")
shell("uv", "init", "--script", "report.py", cwd=scripts_root)
FIND_LINKS = ("--no-index", "--find-links", str(WHEELHOUSE))
shell("uv", "add", "--script", "report.py", "rupee-fmt>=2", *FIND_LINKS, cwd=scripts_root)
report_script = scripts_root / "report.py"
report_script.write_text(
    report_script.read_text().replace(
        'print("Hello from report.py!")',
        "import importlib.metadata\n    from rupee_fmt import fmt\n\n"
        '    print(fmt(1234567.5), "with rupee-fmt", importlib.metadata.version("rupee-fmt"))',
    )
)
show_file(report_script)
shell("uv", "run", *FIND_LINKS, "report.py", cwd=scripts_root)

[<scratch>/scripts]$ uv init --script report.py
  Initialized script at `report.py`
[<scratch>/scripts]$ uv add --script report.py 'rupee-fmt>=2' --no-index --find-links <scratch>/wheelhouse
  Resolved 1 package in <time>
--- report.py ---
# /// script
# requires-python = ">=3.15"
# dependencies = [
#     "rupee-fmt>=2",
# ]
# ///


def main() -> None:
    import importlib.metadata
    from rupee_fmt import fmt

    print(fmt(1234567.5), "with rupee-fmt", importlib.metadata.version("rupee-fmt"))


if __name__ == "__main__":
    main()


[<scratch>/scripts]$ uv run --no-index --find-links <scratch>/wheelhouse report.py
  ₹1,234,567.50 with rupee-fmt 2.0.0


0

**Workspaces** are uv's version of npm/pnpm workspaces: several packages in one
repository share **one** `uv.lock` (so they agree on every version) and one
environment. The root declares `[tool.uv.workspace] members`; a member depends on
another with `[tool.uv.sources] name = { workspace = true }`, and both are installed
editable. Here `uv init` is run *without* `--no-workspace` inside the root, so it
adds each member automatically.

In [26]:
monorepo = write_files(
    fresh_dir("monorepo"),
    {
        "pyproject.toml": """
    [project]
    name = "monorepo"
    version = "0.1.0"
    requires-python = ">=3.15"

    [tool.uv.workspace]
    members = ["packages/*"]
"""
    },
)
UV_INIT_MEMBER = ("uv", "init", "--author-from", "none", "--vcs", "none", "--no-readme")
shell(*UV_INIT_MEMBER, "--lib", "packages/pricing-core", cwd=monorepo)
shell(*UV_INIT_MEMBER, "--package", "packages/billing-api", cwd=monorepo)
shell("uv", "add", "--offline", "--package", "billing-api", "pricing-core", cwd=monorepo)
show_file(monorepo / "packages" / "billing-api" / "pyproject.toml", after="dependencies")
shell(
    "uv",
    "run",
    "--offline",
    "--package",
    "billing-api",
    "python",
    "-c",
    "import pricing_core; print(pricing_core.hello())",
    cwd=monorepo,
)
workspace_lock = tomllib.loads((monorepo / "uv.lock").read_text())
print(f"one uv.lock for the workspace, members: {workspace_lock['manifest']['members']}")

[<scratch>/monorepo]$ uv init --author-from none --vcs none --no-readme --lib packages/pricing-core
  Project `pricing-core` is already a member of workspace `<scratch>/monorepo`
  Initialized project `pricing-core` at `<scratch>/monorepo/packages/pricing-core`
[<scratch>/monorepo]$ uv init --author-from none --vcs none --no-readme --package packages/billing-api
  Project `billing-api` is already a member of workspace `<scratch>/monorepo`
  Initialized project `billing-api` at `<scratch>/monorepo/packages/billing-api`


[<scratch>/monorepo]$ uv add --offline --package billing-api pricing-core
  Using CPython 3.15.1
  Creating virtual environment at: .venv
  Resolved 3 packages in <time>
     Building pricing-core @ <scratch>/monorepo/packages/pricing-core
     Building billing-api @ <scratch>/monorepo/packages/billing-api
        Built pricing-core @ <scratch>/monorepo/packages/pricing-core
        Built billing-api @ <scratch>/monorepo/packages/billing-api
  Prepared 2 packages in <time>
  Installed 2 packages in <time>
   + billing-api==0.1.0 (from <scratch>/monorepo/packages/billing-api)
   + pricing-core==0.1.0 (from <scratch>/monorepo/packages/pricing-core)
--- pyproject.toml ---
dependencies = [
    "pricing-core",
]

[project.scripts]
billing-api = "billing_api:main"

[build-system]
requires = ["uv_build>=0.13.5,<0.14.0"]
build-backend = "uv_build"

[tool.uv.sources]
pricing-core = { workspace = true }


[<scratch>/monorepo]$ uv run --offline --package billing-api python -c 'import pricing_core; print(pricing_core.hello())'
  Hello from pricing-core!
one uv.lock for the workspace, members: ['billing-api', 'monorepo', 'pricing-core']


**`uv export`** converts `uv.lock` into formats other tools understand:
`requirements.txt` (for pip, Dependabot, older platforms), `pylock.toml` (the PEP 751
standard lockfile, 4.1.6) or a CycloneDX SBOM (`cyclonedx1.5`, for security teams).

In [27]:
shell("uv", "export", "--format", "requirements.txt", "--no-hashes", cwd=invoicing)
shell(
    "uv",
    "export",
    "--format",
    "requirements.txt",
    "--no-dev",
    "--extra",
    "report",
    "--no-hashes",
    "--no-header",
    cwd=invoicing,
)

[<scratch>/projects/invoicing]$ uv export --format requirements.txt --no-hashes
  # This file was autogenerated by uv via the following command:
  #    uv export --format requirements.txt --no-hashes
  -e .
  gst-calc==1.0.0
      # via invoicing
  rupee-fmt==1.2.0
      # via gst-calc
  win-colors==1.0.0 ; sys_platform == 'win32'
      # via gst-calc
  Resolved 5 packages in <time>


[<scratch>/projects/invoicing]$ uv export --format requirements.txt --no-dev --extra report --no-hashes --no-header
  -e .
  gst-calc==1.0.0
      # via invoicing
  rupee-fmt==1.2.0
      # via gst-calc
  table-print==0.1.0
      # via invoicing
  win-colors==1.0.0 ; sys_platform == 'win32'
      # via gst-calc
  Resolved 5 packages in <time>


0

**`uv build`** is a build frontend (4.1.1): it builds an sdist into `dist/`, then a
wheel from that sdist. `uv_build` projects are built by uv's built-in copy of the
backend, with no extra download. 4.1.8 opens both archives.

In [28]:
shell("uv", "build", "--offline", cwd=invoicing)
print(sorted(path.name for path in (invoicing / "dist").iterdir()))

[<scratch>/projects/invoicing]$ uv build --offline
  Building source distribution...
  Successfully built dist/invoicing-0.1.0.tar.gz
  Successfully built dist/invoicing-0.1.0-py3-none-any.whl
['.gitignore', 'invoicing-0.1.0-py3-none-any.whl', 'invoicing-0.1.0.tar.gz']


**`uv publish`** uploads `dist/*` to an index (PyPI by default; `--index` or
`--publish-url` for TestPyPI or a private index). The course never publishes, so
this cell only prints the relevant options from `uv publish --help`. 4.1.8 shows the
full release flow with TestPyPI and trusted publishing.

In [29]:
shell(
    "uv",
    "publish",
    "--help",
    keep=r"^\s+(--index |--publish-url|--trusted-publishing|--check-url|--dry-run|\[FILES\])",
)

[<scratch>]$ uv publish --help
    [FILES]...  Paths to the files to upload. Accepts glob expressions [default: dist/*]
        --index <INDEX>
        --trusted-publishing <TRUSTED_PUBLISHING>
        --publish-url <PUBLISH_URL>
        --check-url <CHECK_URL>
        --dry-run


0

### 4.1.4 `pyproject.toml` anatomy

`pyproject.toml` is TOML (like `package.json` with comments and typed tables). Its
tables come from different standards:

| Table | Standard | Read by | npm analogue |
| --- | --- | --- | --- |
| `[build-system]` | PEP 518 / 517 | build frontends (pip, uv) | none (npm has no build step) |
| `[project]` | PEP 621 | backends, which turn it into wheel `METADATA` | `name`, `version`, `dependencies`, `engines`, `license` |
| `[project.scripts]` | PEP 621 (entry points) | installers, which generate launchers | `"bin"` |
| `[project.optional-dependencies]` | PEP 621 | resolvers (`pkg[extra]`) | `optionalDependencies` / `peerDependencies`, sort of |
| `[project.urls]` | PEP 621 | PyPI's sidebar | `homepage`, `repository` |
| `[dependency-groups]` | PEP 735 | uv, pip ≥ 25.1 (`--group`) | `devDependencies` |
| `[tool.<name>]` | each tool's own docs | uv, ruff, mypy, pytest, … | `"eslintConfig"`, `"jest"` keys |

Here is a complete, annotated example for a CLI called `invoice-kit`:

In [30]:
ANATOMY_PYPROJECT = """
[build-system]                                  # PEP 518: how to build this project
requires = ["uv_build>=0.13,<0.14"]             # installed into the isolated build env
build-backend = "uv_build"                      # module whose PEP 517 hooks build it

[project]                                       # PEP 621: the package metadata
name = "invoice-kit"                            # PyPI name (normalized: invoice_kit)
version = "1.4.0"                               # PEP 440 version (or dynamic = ["version"])
description = "GST invoices from the command line"
readme = "README.md"                            # long description shown on PyPI
requires-python = ">=3.12"                      # resolvers skip this release on older Pythons
license = "MIT"                                 # SPDX expression (PEP 639)
authors = [{ name = "Course Author" }]
classifiers = ["Programming Language :: Python :: 3", "Typing :: Typed"]
dependencies = [                                # runtime requirements -> Requires-Dist
    "gst-calc>=1.0,<2",
    "win-colors>=1.0; sys_platform == 'win32'", # environment marker: Windows only
]

[project.optional-dependencies]                 # extras: pip install "invoice-kit[report]"
report = ["table-print>=0.1"]

[project.scripts]                               # console scripts: command = "module:function"
invoice-kit = "invoice_kit.cli:main"

[project.urls]
Homepage = "https://example.com/invoice-kit"

[dependency-groups]                             # PEP 735: local-only groups, never published
dev = ["rupee-fmt>=1.1", { include-group = "lint" }]
lint = ["table-print"]

[tool.uv]                                       # uv settings for this project
no-index = true
find-links = ["../wheelhouse"]

[tool.ruff]                                     # linter / formatter
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B"]

[tool.mypy]                                     # type checker (Unit 4.2)
strict = true
python_version = "3.12"

[tool.pytest.ini_options]                       # test runner (Unit 4.3)
testpaths = ["tests"]
addopts = "-ra --strict-markers"
"""
anatomy = tomllib.loads(ANATOMY_PYPROJECT)
for table, content in anatomy.items():
    keys = list(content)
    print(f"[{table}] -> {keys}")

[build-system] -> ['requires', 'build-backend']
[project] -> ['name', 'version', 'description', 'readme', 'requires-python', 'license', 'authors', 'classifiers', 'dependencies', 'optional-dependencies', 'scripts', 'urls']
[dependency-groups] -> ['dev', 'lint']
[tool] -> ['uv', 'ruff', 'mypy', 'pytest']


`tomllib` (standard library since 3.11) parses TOML into dicts; note how `[tool.uv]`
and `[tool.ruff]` become nested keys of `tool`. The `[tool.*]` tables configure the
tools:

In [31]:
for tool_name, settings in anatomy["tool"].items():
    print(f"tool.{tool_name}: {settings}")
print(f"{anatomy['dependency-groups']=}")

tool.uv: {'no-index': True, 'find-links': ['../wheelhouse']}
tool.ruff: {'line-length': 100, 'target-version': 'py312', 'lint': {'select': ['E', 'F', 'I', 'UP', 'B']}}
tool.mypy: {'strict': True, 'python_version': '3.12'}
tool.pytest: {'ini_options': {'testpaths': ['tests'], 'addopts': '-ra --strict-markers'}}
anatomy['dependency-groups']={'dev': ['rupee-fmt>=1.1', {'include-group': 'lint'}], 'lint': ['table-print']}


**Where each table ends up.** Build the project and read the wheel's `METADATA`:
`[project]` becomes `Name`, `Version`, `Requires-Python`, `License-Expression`,
`Requires-Dist`; each extra becomes `Provides-Extra` plus `Requires-Dist: …; extra ==
"report"`; `[project.scripts]` becomes `entry_points.txt`. `[dependency-groups]` and
`[tool.*]` are **not** in the wheel: they only matter in the source tree.

In [32]:
anatomy_root = write_files(
    fresh_dir("projects/invoice-kit"),
    {
        "pyproject.toml": ANATOMY_PYPROJECT.replace("../wheelhouse", "../../wheelhouse"),
        "README.md": "# invoice-kit\n\nGST invoices from the command line.\n",
        "src/invoice_kit/__init__.py": "",
        "src/invoice_kit/cli.py": "def main() -> None:\n    print('invoice-kit')\n",
        "src/invoice_kit/py.typed": "",
    },
)
shell("uv", "build", "--offline", "--wheel", cwd=anatomy_root, keep="Successfully")
kit_wheel = next((anatomy_root / "dist").glob("*.whl"))
with zipfile.ZipFile(kit_wheel) as wheel:
    metadata_text = wheel.read("invoice_kit-1.4.0.dist-info/METADATA").decode()
    entry_points = wheel.read("invoice_kit-1.4.0.dist-info/entry_points.txt").decode()
print(metadata_text.split("\n\n")[0])  # headers only (the README body follows)
print("--- entry_points.txt ---\n" + entry_points.strip())
print(f"{'dependency-groups' in metadata_text=}  {'ruff' in metadata_text=}")

[<scratch>/projects/invoice-kit]$ uv build --offline --wheel
  Successfully built dist/invoice_kit-1.4.0-py3-none-any.whl
Metadata-Version: 2.4
Name: invoice-kit
Version: 1.4.0
Summary: GST invoices from the command line
Author: Course Author
License-Expression: MIT
Classifier: Programming Language :: Python :: 3
Classifier: Typing :: Typed
Requires-Dist: gst-calc>=1.0,<2
Requires-Dist: win-colors>=1.0 ; sys_platform == 'win32'
Requires-Dist: table-print>=0.1 ; extra == 'report'
Requires-Python: >=3.12
Project-URL: Homepage, https://example.com/invoice-kit
Provides-Extra: report
Description-Content-Type: text/markdown
--- entry_points.txt ---
[console_scripts]
invoice-kit = invoice_kit.cli:main
'dependency-groups' in metadata_text=False  'ruff' in metadata_text=False


The course's own `pyproject.toml` is a *virtual* project: `[project]` without
`[build-system]`, a `dev` group and tool tables. Reading it is fine; no cell in this
notebook writes to it.

In [33]:
course = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text())
print(f"{course['project']['name']=}  {course['project']['requires-python']=}")
print(f"{'build-system' in course=}  (virtual project: not built or installed)")
print(f"{sorted(course['dependency-groups'])=}  {sorted(course['tool'])=}")

course['project']['name']='python-course'  course['project']['requires-python']='>=3.15'
'build-system' in course=False  (virtual project: not built or installed)
sorted(course['dependency-groups'])=['dev']  sorted(course['tool'])=['ruff']


### 4.1.5 Version specifiers (PEP 440)

Python versions follow **PEP 440**, not SemVer. A version looks like
`[N!]N(.N)*[{a|b|rc}N][.postN][.devN]`, and ordering is defined by the standard:
`1.0.dev1 < 1.0a1 < 1.0b1 < 1.0rc1 < 1.0 < 1.0.post1`. A requirement combines a
name, optional extras, specifiers and an optional environment marker:

```text
gst-calc[report] >=1.0,<2 ; python_version >= "3.12"
^name    ^extras ^specifiers ^marker
```

| Operator | Meaning | npm equivalent |
| --- | --- | --- |
| `==1.4.2` | Exactly this version (`==1.4.*` = any 1.4.x) | `1.4.2`, `1.4.x` |
| `>=1.4` | This version or newer | `>=1.4` |
| `~=1.4` | **Compatible release**: `>=1.4, ==1.*` | `^1.4` (for ≥ 1.0) |
| `~=1.4.2` | `>=1.4.2, ==1.4.*` | `~1.4.2` |
| `!=1.5.0` | Anything except this version | `!=` does not exist in npm (`<1.5.0 \|\| >1.5.0`) |
| `<2`, `>`, `<=` | Upper bounds and exclusive comparisons | same |
| `===` | Arbitrary string equality (rare, legacy) | none |

`packaging.specifiers.SpecifierSet` implements these rules exactly as pip and uv do.
Each operator gets its own demo below.

In [34]:
from packaging.specifiers import SpecifierSet
from packaging.version import Version

CANDIDATES = ["1.3.9", "1.4.0", "1.4.2", "1.4.9", "1.5.0", "1.9.1", "2.0.0", "2.1.0rc1"]


def matching(specifier: str) -> list[str]:
    """Which candidate versions satisfy the specifier (pre-releases excluded by default)?"""
    return [str(version) for version in SpecifierSet(specifier).filter(map(Version, CANDIDATES))]


print(f"{matching('==1.4.2')=}")
print(f"{matching('==1.4.*')=}  (wildcard: any 1.4.x)")

matching('==1.4.2')=['1.4.2']
matching('==1.4.*')=['1.4.0', '1.4.2', '1.4.9']  (wildcard: any 1.4.x)


`>=` is a lower bound only: it happily accepts the next major version, breaking changes included.

In [35]:
print(f"{matching('>=1.4')=}")
print(f"{matching('>=1.4,<2')=}  (the usual way to say: 1.4 or newer, but not 2.x)")

matching('>=1.4')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1', '2.0.0']
matching('>=1.4,<2')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1']  (the usual way to say: 1.4 or newer, but not 2.x)


`~=` (*compatible release*) drops the last component you write and pins the rest,
so the number of components you write decides how much can float:

In [36]:
print(f"{matching('~=1.4')=}    (>=1.4, ==1.*)")
print(f"{matching('~=1.4.2')=}  (>=1.4.2, ==1.4.*)")
print(f"{SpecifierSet('~=1.4.2') == SpecifierSet('>=1.4.2,==1.4.*')=}  (different spelling)")
print(f"{Version('1.4.2') in SpecifierSet('~=1.4.2')=}")

matching('~=1.4')=['1.4.0', '1.4.2', '1.4.9', '1.5.0', '1.9.1']    (>=1.4, ==1.*)
matching('~=1.4.2')=['1.4.2', '1.4.9']  (>=1.4.2, ==1.4.*)
SpecifierSet('~=1.4.2') == SpecifierSet('>=1.4.2,==1.4.*')=False  (different spelling)
Version('1.4.2') in SpecifierSet('~=1.4.2')=True


`!=` excludes a known-bad release while still accepting everything around it:

In [37]:
print(f"{matching('>=1.4,!=1.5.0,<2')=}")
print(f"{matching('!=1.4.*')=}  (exclude a whole minor series)")

matching('>=1.4,!=1.5.0,<2')=['1.4.0', '1.4.2', '1.4.9', '1.9.1']
matching('!=1.4.*')=['1.3.9', '1.5.0', '1.9.1', '2.0.0']  (exclude a whole minor series)


**Pre-releases** (`a`, `b`, `rc`, `.dev`) are excluded unless a specifier names one
or you opt in. This keeps `2.1.0rc1` out of production by accident. The same rule
applies in uv: `uv pip compile` picks 2.0.0 unless you pass `--prerelease allow`.

In [38]:
print(f"{matching('>=2')=}")
print(
    f"{[str(v) for v in SpecifierSet('>=2', prereleases=True).filter(map(Version, CANDIDATES))]=}"
)
print(f"{matching('>=2.1.0rc1')=}  (naming a pre-release opts in)")
ordering = ["1.0", "1.0.post1", "1.0rc1", "1.0a1", "1.0.dev1", "1.0b2"]
print(f"{sorted(ordering, key=Version)=}")
requirements_in = UNIT41_DIR / "requirements.in"
requirements_in.write_text("rupee-fmt>=2\n")
for extra_flags in ([], ["--prerelease", "allow"]):
    shell(
        "uv",
        "pip",
        "compile",
        "requirements.in",
        "--no-header",
        "--no-annotate",
        "--no-index",
        "--find-links",
        "wheelhouse",
        "--quiet",
        *extra_flags,
    )

matching('>=2')=['2.0.0']
[str(v) for v in SpecifierSet('>=2', prereleases=True).filter(map(Version, CANDIDATES))]=['2.0.0', '2.1.0rc1']
matching('>=2.1.0rc1')=['2.1.0rc1']  (naming a pre-release opts in)
sorted(ordering, key=Version)=['1.0.dev1', '1.0a1', '1.0b2', '1.0rc1', '1.0', '1.0.post1']
[<scratch>]$ uv pip compile requirements.in --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  rupee-fmt==2.0.0


[<scratch>]$ uv pip compile requirements.in --no-header --no-annotate --no-index --find-links wheelhouse --quiet --prerelease allow
  rupee-fmt==2.1.0rc1


**Environment markers** (PEP 508) make a requirement conditional on where it is
installed: `sys_platform`, `platform_machine`, `python_version`,
`implementation_name`, `os_name` and more. `gst-calc` uses one to pull in
`win-colors` on Windows only. A marker is evaluated against an environment; the
default is the running interpreter.

In [39]:
from packaging.markers import Marker, default_environment

windows_only = Marker("sys_platform == 'win32'")
modern_cpython = Marker("python_version >= '3.13' and implementation_name == 'cpython'")
here = default_environment()
print(f"{here['sys_platform']=}  {here['python_version']=}  {here['platform_machine']=}")
print(f"{windows_only.evaluate()=}  {windows_only.evaluate({'sys_platform': 'win32'})=}")
print(f"{modern_cpython.evaluate()=}")
old_python = Marker('python_version < "3.11"')
print(f"{old_python.evaluate()=}  (e.g. a tomli backport)")

here['sys_platform']='linux'  here['python_version']='3.15'  here['platform_machine']='x86_64'
windows_only.evaluate()=False  windows_only.evaluate({'sys_platform': 'win32'})=True
modern_cpython.evaluate()=True
old_python.evaluate()=False  (e.g. a tomli backport)


**Extras** name optional feature sets. `pkg[extra]` adds the extra's requirements;
inside the published metadata they appear as markers on `extra`. `Requirement`
parses a whole PEP 508 line:

In [40]:
from packaging.requirements import Requirement

requirement = Requirement('gst-calc[report]>=1.0,<2; python_version >= "3.12"')
print(f"{requirement.name=}  {requirement.extras=}")
print(f"{str(requirement.specifier)=}  {str(requirement.marker)=}")
published = Requirement('table-print>=0.1; extra == "report"')
print(
    f"{published.marker.evaluate({'extra': 'report'})=}\n"
    f"{published.marker.evaluate({'extra': ''})=}"
)

requirement.name='gst-calc'  requirement.extras={'report'}
str(requirement.specifier)='<2,>=1.0'  str(requirement.marker)='python_version >= "3.12"'
published.marker.evaluate({'extra': 'report'})=True
published.marker.evaluate({'extra': ''})=False


And for real: installing `gst-calc[report]` adds `table-print`, the plain name does not.

In [41]:
for requirement_text in ("gst-calc==1.0.0", "gst-calc[report]==1.0.0"):
    pip("install", "--dry-run", "--ignore-installed", requirement_text, keep="Would install")

[<scratch>]$ python -m pip install --dry-run --ignore-installed gst-calc==1.0.0
  Would install gst-calc-1.0.0 rupee-fmt-1.2.0


[<scratch>]$ python -m pip install --dry-run --ignore-installed 'gst-calc[report]==1.0.0'
  Would install gst-calc-1.0.0 rupee-fmt-1.2.0 table-print-0.1.0


**Predict the output.**

```python
from packaging.specifiers import SpecifierSet

spec = SpecifierSet("~=2.2")
print([v for v in ["2.1", "2.2", "2.9", "3.0", "2.10"] if v in spec])
```

<details><summary>Answer</summary>

`['2.2', '2.9', '2.10']`. `~=2.2` means `>=2.2, ==2.*`, so 2.1 is too old and 3.0
is a new major series. Versions compare component by component as numbers, so
`2.10` is **newer** than `2.9` (unlike string comparison).

</details>

Run the next cell to check your prediction.

In [42]:
from packaging.specifiers import SpecifierSet

spec = SpecifierSet("~=2.2")
print([v for v in ["2.1", "2.2", "2.9", "3.0", "2.10"] if v in spec])

['2.2', '2.9', '2.10']


### 4.1.6 Lockfiles

`pyproject.toml` says what you *accept* (`gst-calc~=1.0`); a lockfile records what
you *got*: every package, transitive ones included, at an exact version, with file
hashes and where it came from. Installing from the lock gives the same environment
on every machine.

**`uv.lock`** is uv's own format: TOML, **universal** (resolved once for every
platform and Python version the project supports; markers record which packages
apply where) and meant to be committed. Never edit it by hand.

In [43]:
lock = tomllib.loads((invoicing / "uv.lock").read_text())
print(f"{lock['version']=}  {lock['requires-python']=}")
for package in lock["package"]:
    source = package["source"]
    wheels = len(package.get("wheels", []))
    dependencies = [
        dependency["name"] + (f" ; {dependency['marker']}" if "marker" in dependency else "")
        for dependency in package.get("dependencies", [])
    ]
    print(
        f"{package['name']:<12} {package['version']:<7} {tidy(str(source)):<34} "
        f"wheels={wheels} deps={dependencies}"
    )

lock['version']=1  lock['requires-python']='>=3.15'
gst-calc     1.0.0   {'registry': '../../wheelhouse'}   wheels=1 deps=['rupee-fmt', "win-colors ; sys_platform == 'win32'"]
invoicing    0.1.0   {'editable': '.'}                  wheels=0 deps=['gst-calc']
rupee-fmt    1.2.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]
table-print  0.1.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]
win-colors   1.0.0   {'registry': '../../wheelhouse'}   wheels=1 deps=[]


Note `win-colors`: it is locked even on Linux, with the marker
`sys_platform == 'win32'` on the edge from `gst-calc`, so a Windows teammate gets the
same lock. `uv sync` only installs what applies to the current machine:

In [44]:
installed = run_tool("uv", "pip", "list", "--format", "freeze", cwd=invoicing)[1]
print(installed)
print(f"{'win-colors' in installed=}  (locked for Windows, not installed here)")

gst-calc==1.0.0
invoicing==0.1.0
rupee-fmt==1.2.0
'win-colors' in installed=False  (locked for Windows, not installed here)


**`pylock.toml` (PEP 751, accepted 2025)** is the *standard* lockfile format that
any installer can read: a flat list of packages, each with its exact files and
hashes. uv exports it from `uv.lock`; pip ≥ 25.1 can write one with `pip lock`
(experimental). Think of `uv.lock` as the rich working format and `pylock.toml` as
the interchange format for other tools and auditors.

In [45]:
shell("uv", "export", "--format", "pylock.toml", "-o", "pylock.toml", "--quiet", cwd=invoicing)
pylock = tomllib.loads((invoicing / "pylock.toml").read_text())
print(f"{pylock['lock-version']=}  {pylock['created-by']=}")
for package in pylock["packages"]:
    files = [*package.get("wheels", []), *([package["sdist"]] if "sdist" in package else [])]
    hashes = [next(iter(file["hashes"].values()))[:12] + "…" for file in files]
    print(
        f"{package['name']:<12} {package.get('version', '(editable)'):<10} "
        f"marker={package.get('marker', '-'):<24} sha256={hashes}"
    )

[<scratch>/projects/invoicing]$ uv export --format pylock.toml -o pylock.toml --quiet
pylock['lock-version']='1.0'  pylock['created-by']='uv'
gst-calc     1.0.0      marker=-                        sha256=['c09ec26299b9…', '5fa5a288e797…']
invoicing    (editable) marker=-                        sha256=[]
rupee-fmt    1.2.0      marker=-                        sha256=['479bc9002b86…', '15ebb6744931…']
win-colors   1.0.0      marker=sys_platform == 'win32'  sha256=['def335bdaa52…', '59e220dec9e4…']


**Pinning strategy: applications vs libraries.** The same project can be both, so
decide per artefact:

| | Application (service, CLI you deploy) | Library (published for others to import) |
| --- | --- | --- |
| `dependencies` | Reasonable ranges (`>=1.4,<2`) | **Wide** ranges: lower bound you tested, upper bound only for known breaks |
| Lockfile | Committed; deploy exactly the lock (`uv sync --locked`) | Committed for CI and contributors, but **users never see it** |
| Upgrades | Deliberate: `uv lock --upgrade-package`, Dependabot/Renovate PRs | Test against the newest *and* the oldest allowed versions |
| Exact pins (`==`) in `pyproject.toml` | Acceptable | Never: they cause conflicts in every app that uses you |

uv can resolve to the **lowest** versions your specifiers allow, which is how a
library tests its lower bounds:

In [46]:
requirements_in.write_text("gst-calc>=1.0,<2\n")
for strategy in ("highest", "lowest"):
    print(f"--resolution {strategy}:")
    shell(
        "uv",
        "pip",
        "compile",
        "requirements.in",
        "--resolution",
        strategy,
        "--no-header",
        "--no-annotate",
        "--no-index",
        "--find-links",
        "wheelhouse",
        "--quiet",
    )

--resolution highest:
[<scratch>]$ uv pip compile requirements.in --resolution highest --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  gst-calc==1.0.0
  rupee-fmt==1.2.0
--resolution lowest:
[<scratch>]$ uv pip compile requirements.in --resolution lowest --no-header --no-annotate --no-index --find-links wheelhouse --quiet
  gst-calc==1.0.0
  rupee-fmt==1.0.0


### 4.1.7 Project layouts

| Flat layout | `src/` layout |
| --- | --- |
| `myproject/mypkg/__init__.py` | `myproject/src/mypkg/__init__.py` |
| `import mypkg` works from the project root **without installing**, because the current folder is on `sys.path` | `import mypkg` works **only** after installing (usually editable) |
| Tests may pass against the working copy while the built wheel is broken (missing files, wrong package list) | Tests run against the installed package, like users |

uv (`--lib`, `--package`), Hatch, PDM and the Python Packaging User Guide default to
`src/`. The demo runs `python -c "import pricing"` from the root of each layout,
without installing anything:

In [47]:
PRICING_MODULE = "def price(amount):\n    return amount\n"
flat_root = write_files(fresh_dir("layouts/flat"), {"pricing/__init__.py": PRICING_MODULE})
src_root = write_files(fresh_dir("layouts/src"), {"src/pricing/__init__.py": PRICING_MODULE})
IMPORT_CHECK = "import pricing; print('imported from', pricing.__file__)"
print("flat:", run_tool("python", "-c", IMPORT_CHECK, cwd=flat_root)[1])
print("src: ", run_tool("python", "-c", IMPORT_CHECK, cwd=src_root)[1].splitlines()[-1])

flat: imported from <scratch>/layouts/flat/pricing/__init__.py
src:  ModuleNotFoundError: No module named 'pricing'. Did you mean: 'profiling'?


**Editable-install mechanics.** `uv sync` installs the project itself in editable
mode (like `pip install -e .`). What actually lands in `site-packages` is tiny: a
`.dist-info/` folder (metadata, so `importlib.metadata` finds the project) with a
`direct_url.json` saying *editable*, and a `.pth` file that adds `src/` to
`sys.path`. setuptools sometimes uses an import-hook finder instead of a plain path
(`__editable___*_finder.py`), which can hide new top-level modules; a `.pth` path
does not.

In [48]:
invoicing_site = next((invoicing / ".venv").glob("lib/python3*/site-packages"))
for path in sorted(invoicing_site.glob("*invoicing*")):
    print(path.name + ("/" if path.is_dir() else ""))
pth_text = next(invoicing_site.glob("*invoicing*.pth")).read_text().strip()
print(f"pth -> {tidy(pth_text)}")
dist_info = next(invoicing_site.glob("invoicing-*.dist-info"))
print(f"direct_url.json -> {tidy((dist_info / 'direct_url.json').read_text())}")

invoicing-0.1.0.dist-info/
invoicing.pth
pth -> <scratch>/projects/invoicing/src
direct_url.json -> {"url":"<scratch>/projects/invoicing","dir_info":{"editable":true}}


**Under the hood: hardlinks from the cache.** The other packages in that
environment were not copied: each file is a hardlink into uv's cache, so its link
count is above 1 (the same inode appears in two places).

In [49]:
rupee_init = next(invoicing_site.glob("rupee_fmt/__init__.py"))
status = rupee_init.stat()
print(f"rupee_fmt/__init__.py: inode {status.st_ino}, {status.st_nlink} links")
linked_in_cache = [
    path
    for path in (UNIT41_DIR / "uv-cache").rglob("__init__.py")
    if path.stat().st_ino == status.st_ino
]
print(f"same inode in the uv cache: {[tidy(str(path))[:60] + '…' for path in linked_in_cache[:1]]}")

rupee_fmt/__init__.py: inode 17586286, 2 links
same inode in the uv cache: ['<scratch>/uv-cache/archive-v0/PHyvdcUWKyA319Lc/rupee_fmt/__i…']


**`__version__` via `importlib.metadata`.** Keep the version in exactly one place,
`pyproject.toml`, and read it at run time from the installed metadata. A bumped
version shows up after the next `uv sync` (which re-writes the metadata):

```python
from importlib.metadata import version

__version__ = version("invoicing")
```

In [50]:
init_file = invoicing / "src" / "invoicing" / "__init__.py"
init_file.write_text(
    'from importlib.metadata import version\n\n__version__ = version("invoicing")\n'
    + init_file.read_text()
)
VERSION_CHECK = ("python", "-c", "import invoicing; print(invoicing.__version__)")
shell("uv", "run", *VERSION_CHECK, cwd=invoicing)
pyproject_path.write_text(
    pyproject_path.read_text().replace('version = "0.1.0"', 'version = "0.2.0"')
)
shell("uv", "run", *VERSION_CHECK, cwd=invoicing)  # re-syncs: metadata rewritten
print(
    f"in this kernel: {importlib.metadata.version('mypy')=}  "
    f"{importlib.metadata.version('pydantic')=}"
)

[<scratch>/projects/invoicing]$ uv run python -c 'import invoicing; print(invoicing.__version__)'
  0.1.0


[<scratch>/projects/invoicing]$ uv run python -c 'import invoicing; print(invoicing.__version__)'
  0.2.0
     Building invoicing @ <scratch>/projects/invoicing
        Built invoicing @ <scratch>/projects/invoicing
  Uninstalled 1 package in <time>
  Installed 1 package in <time>
in this kernel: importlib.metadata.version('mypy')='2.4.1'  importlib.metadata.version('pydantic')='2.14.3'


### 4.1.8 Building and publishing

**Wheel contents.** A wheel is a zip file. Everything outside the `.dist-info/`
folder is copied into `site-packages` as is; the `.dist-info/` folder holds the
metadata installers and `importlib.metadata` read:

| File | Content |
| --- | --- |
| `METADATA` | Core metadata (name, version, `Requires-Dist`, `Requires-Python`, README) |
| `WHEEL` | Wheel format version, generator, `Root-Is-Purelib`, the tags |
| `RECORD` | Every file with its SHA-256 and size: the installer checks them and uses it to uninstall |
| `entry_points.txt` | Console scripts and plugin entry points |
| `licenses/` | License files (PEP 639) |

The demo opens the `invoicing` wheel and sdist built in 4.1.3.

In [51]:
shutil.rmtree(invoicing / "dist", ignore_errors=True)
shell("uv", "build", "--offline", cwd=invoicing, keep="Successfully")
invoicing_wheel = next((invoicing / "dist").glob("*.whl"))
invoicing_sdist = next((invoicing / "dist").glob("*.tar.gz"))
with zipfile.ZipFile(invoicing_wheel) as wheel:
    print(f"{invoicing_wheel.name}:")
    for info in wheel.infolist():
        print(f"  {info.filename:<44} {info.file_size:>6} bytes")
    dist_info_name = next(name.split("/")[0] for name in wheel.namelist() if ".dist-info/" in name)
    print(wheel.read(f"{dist_info_name}/WHEEL").decode())

[<scratch>/projects/invoicing]$ uv build --offline
  Successfully built dist/invoicing-0.2.0.tar.gz
  Successfully built dist/invoicing-0.2.0-py3-none-any.whl
invoicing-0.2.0-py3-none-any.whl:
  invoicing/                                        0 bytes
  invoicing/__init__.py                           289 bytes
  invoicing-0.2.0.dist-info/                        0 bytes
  invoicing-0.2.0.dist-info/WHEEL                  80 bytes
  invoicing-0.2.0.dist-info/entry_points.txt       46 bytes
  invoicing-0.2.0.dist-info/METADATA              216 bytes
  invoicing-0.2.0.dist-info/RECORD                385 bytes
Wheel-Version: 1.0
Generator: uv 0.13.5
Root-Is-Purelib: true
Tag: py3-none-any



`METADATA` uses e-mail header syntax, so the standard library's `email.parser` reads
it, which is exactly what `importlib.metadata` does:

In [52]:
with zipfile.ZipFile(invoicing_wheel) as wheel:
    headers = email.parser.Parser().parsestr(wheel.read(f"{dist_info_name}/METADATA").decode())
    entry_points = wheel.read(f"{dist_info_name}/entry_points.txt").decode()
for key in ("Metadata-Version", "Name", "Version", "Requires-Python"):
    print(f"{key}: {headers[key]}")
print(f"Requires-Dist: {headers.get_all('Requires-Dist')}")
print(f"Provides-Extra: {headers.get_all('Provides-Extra')}")
print(f"entry_points.txt: {entry_points.split()}")

Metadata-Version: 2.3
Name: invoicing
Version: 0.2.0
Requires-Python: >=3.15
Requires-Dist: ['gst-calc~=1.0', "table-print>=0.1 ; extra == 'report'"]
Provides-Extra: ['report']
entry_points.txt: ['[console_scripts]', 'invoicing', '=', 'invoicing:main']


**Under the hood: verifying `RECORD`.** Each line is `path,sha256=<urlsafe base64
digest without padding>,size`. Recomputing every digest is what an installer does to
detect a corrupted or tampered wheel:

In [53]:
def record_digest(data: bytes) -> str:
    """The RECORD spelling of a SHA-256 digest."""
    return "sha256=" + base64.urlsafe_b64encode(hashlib.sha256(data).digest()).rstrip(b"=").decode()


with zipfile.ZipFile(invoicing_wheel) as wheel:
    record_lines = wheel.read(f"{dist_info_name}/RECORD").decode().splitlines()
    for line in record_lines:
        path, digest, size = line.rsplit(",", 2)
        if digest:  # RECORD cannot contain its own hash
            actual = record_digest(wheel.read(path))
            print(f"{path:<44} {'OK' if actual == digest else 'MISMATCH'}  {digest[:20]}…")
        else:
            print(f"{path:<44} (no hash: the RECORD file itself)")

invoicing/__init__.py                        OK  sha256=qm-iUS2GXRwSl…
invoicing-0.2.0.dist-info/WHEEL              OK  sha256=si2p8vW03Rnhy…
invoicing-0.2.0.dist-info/entry_points.txt   OK  sha256=kGDIjnJ5Sz4c5…
invoicing-0.2.0.dist-info/METADATA           OK  sha256=S-3jZut3SJGPG…
invoicing-0.2.0.dist-info/RECORD             (no hash: the RECORD file itself)


The **sdist** is a `.tar.gz` of the source tree plus a generated `PKG-INFO` (the same
metadata). It contains `pyproject.toml`, so anyone can rebuild the wheel; uv built
the wheel above from this very archive. `uv_build` stores a normalized copy of
`pyproject.toml` (comments dropped, lists on one line) and keeps your original as
`pyproject.toml.orig`.

In [54]:
with tarfile.open(invoicing_sdist) as sdist:
    for member in sdist.getmembers():
        if member.isfile():
            print(f"  {member.name:<48} {member.size:>6} bytes")
    pkg_info = (
        sdist.extractfile(f"{invoicing_sdist.name.removesuffix('.tar.gz')}/PKG-INFO")
        .read()
        .decode()
    )
print(pkg_info.split("\n\n")[0])

  invoicing-0.2.0/PKG-INFO                            216 bytes
  invoicing-0.2.0/pyproject.toml                      480 bytes
  invoicing-0.2.0/pyproject.toml.orig                 581 bytes
  invoicing-0.2.0/src/invoicing/__init__.py           289 bytes
Metadata-Version: 2.3
Name: invoicing
Version: 0.2.0
Summary: Add your description here
Requires-Dist: gst-calc~=1.0
Requires-Dist: table-print>=0.1 ; extra == 'report'
Requires-Python: >=3.15
Provides-Extra: report



**TestPyPI** (test.pypi.org) is a separate, throwaway copy of PyPI for rehearsing a
release. Declare it as an *explicit* index (never used for resolution unless a
package is pinned to it) with a publish URL, then `uv publish --index testpypi`, and
install from it to check the release works. The course does not run these commands.

```toml
[[tool.uv.index]]
name = "testpypi"
url = "https://test.pypi.org/simple/"
publish-url = "https://test.pypi.org/legacy/"
explicit = true
```

```bash
uv build
uv publish --index testpypi            # token or trusted publishing
uv run --with invoicing --index https://test.pypi.org/simple/ --no-project -- invoicing
```

**Trusted publishing (OIDC).** Instead of storing a long-lived PyPI API token as a CI
secret, you tell PyPI once: "releases of `invoicing` may come from GitHub repository
*owner/repo*, workflow `release.yml`, environment `pypi`". At release time the CI job
asks GitHub for a short-lived **OIDC identity token**, PyPI checks its claims against
that configuration and returns a 15-minute upload token. Nothing secret is stored,
and a leaked token expires almost at once. npm now offers the same thing (trusted
publishing with provenance).

```mermaid
sequenceDiagram
    participant CI as GitHub Actions job
    participant GH as GitHub OIDC provider
    participant PyPI
    CI->>GH: request ID token (audience: pypi)
    GH-->>CI: signed JWT (repo, workflow, environment)
    CI->>PyPI: exchange JWT for an upload token
    PyPI-->>CI: short-lived API token (if claims match the trusted publisher)
    CI->>PyPI: uv publish dist/*
```

<details><summary>ASCII fallback</summary>

```text
CI job --(1) ask for ID token--> GitHub OIDC
CI job <-(2) signed JWT: repo, workflow, env-- GitHub OIDC
CI job --(3) JWT--> PyPI checks the trusted-publisher config
CI job <-(4) 15-minute upload token-- PyPI
CI job --(5) uv publish dist/*--> PyPI
```

</details>

```yaml
# .github/workflows/release.yml (shown, not used by the course)
on:
  release:
    types: [published]
jobs:
  publish:
    runs-on: ubuntu-latest
    environment: pypi
    permissions:
      id-token: write        # allows the job to request the OIDC token
    steps:
      - uses: actions/checkout@v5
      - uses: astral-sh/setup-uv@v7
      - run: uv build
      - run: uv publish      # detects the OIDC environment: --trusted-publishing automatic
```

**SemVer vs CalVer.**

| Scheme | Format | Says | Used by |
| --- | --- | --- | --- |
| SemVer | `MAJOR.MINOR.PATCH` | Compatibility: major = breaking, minor = features, patch = fixes | most libraries, mypy, pydantic, uv (0.x: anything may break) |
| CalVer | `YY.MINOR` or `YYYY.MM.PATCH` | *When* it was released; no compatibility promise | pip, Black, Ubuntu, `certifi` |

Both are valid PEP 440 versions, so specifiers and ordering work the same. A SemVer
bump helper and the versions of tools in this environment:

In [55]:
def bump(version: str, part: str) -> str:
    """Next SemVer version: part is "major", "minor" or "patch"; pre-release parts are dropped."""
    major, minor, patch = Version(version).release[:3]
    match part:
        case "major":
            return f"{major + 1}.0.0"
        case "minor":
            return f"{major}.{minor + 1}.0"
        case "patch":
            return f"{major}.{minor}.{patch + 1}"
    raise ValueError(f"unknown part {part!r}")


print(f"{bump('1.4.2', 'patch')=}  {bump('1.4.2', 'minor')=}  {bump('1.4.2', 'major')=}")
print(f"{Version('2026.11.0') > Version('2026.2.3')=}  (CalVer compares numerically too)")
for distribution in ("pip", "mypy", "pydantic", "certifi"):
    try:
        print(f"  {distribution:<9} {importlib.metadata.version(distribution)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {distribution:<9} not installed")

bump('1.4.2', 'patch')='1.4.3'  bump('1.4.2', 'minor')='1.5.0'  bump('1.4.2', 'major')='2.0.0'
Version('2026.11.0') > Version('2026.2.3')=True  (CalVer compares numerically too)
  pip       26.3
  mypy      2.4.1
  pydantic  2.14.3
  certifi   2026.10.23


**Changelogs.** Keep a human-written `CHANGELOG.md` in the
[Keep a Changelog](https://keepachangelog.com) format: newest release first, an
`Unreleased` section at the top, entries grouped under *Added*, *Changed*,
*Deprecated*, *Removed*, *Fixed*, *Security*. With Conventional Commit messages
(`feat:`, `fix:`, `feat!:`), tools such as `git-cliff`, `commitizen` or
`python-semantic-release` can draft the entries and even pick the next SemVer
version. The idea in a few lines:

In [56]:
def draft_changelog(current_version: str, commit_subjects: list[str]) -> str:
    """Group Conventional Commit subjects into a Keep-a-Changelog section and pick the bump."""
    sections: dict[str, list[str]] = {"Added": [], "Fixed": [], "Changed": []}
    part = "patch"
    for subject in commit_subjects:
        kind, _, description = subject.partition(": ")
        if kind.endswith("!"):
            sections["Changed"].append(f"**Breaking:** {description}")
            part = "major"
        elif kind.startswith("feat"):
            sections["Added"].append(description)
            part = "minor" if part == "patch" else part
        elif kind.startswith("fix"):
            sections["Fixed"].append(description)
    lines = [f"## [{bump(current_version, part)}]"]
    for heading, entries in sections.items():
        lines += [f"### {heading}", *(f"- {entry}" for entry in entries)] if entries else []
    return "\n".join(lines)


print(
    draft_changelog(
        "1.4.2",
        [
            "feat(cli): add --currency option",
            "fix: round GST half up",
            "docs: fix typo",
            "feat!: drop Python 3.11",
        ],
    )
)

## [2.0.0]
### Added
- add --currency option
### Fixed
- round GST half up
### Changed
- **Breaking:** drop Python 3.11


### 4.1.9 Supply-chain hygiene

Every dependency is code you run with your permissions. The main risks and the tool
for each:

| Risk | Defence |
| --- | --- |
| A dependency has a published vulnerability | `pip-audit` (or `uv` + Dependabot/Renovate) in CI |
| You install a look-alike name (`requets`) | Review new dependencies; check name, author, download history |
| A file is swapped in transit or on a mirror | Hash checking (lockfiles, `--require-hashes`) |
| `pyproject.toml` drifts from the code (missing or unused dependencies) | `deptry` |
| A license you cannot ship (copyleft in a closed product) | Read `License-Expression` metadata; license checkers |
| Building an sdist runs arbitrary code | Prefer wheels (`--only-binary :all:`, `uv … --no-build`) |

**`pip-audit`** checks installed packages or a requirements file against the Python
Packaging Advisory Database (via PyPI or OSV). It needs the network, so the cell
checks first. The requirements file pins two deliberately old releases. The
findings change over time as advisories are published.

In [57]:
audit_file = UNIT41_DIR / "audit-requirements.txt"
audit_file.write_text("requests==2.19.0\njinja2==2.10\n")
if not network_available():
    print("SKIP: no network connection to pypi.org; pip-audit needs the advisory database")
else:
    exit_code, audit_output = run_tool(
        "pip-audit",
        "-r",
        str(audit_file),
        "--no-deps",
        "--disable-pip",
        "--progress-spinner",
        "off",
        "--desc",
        "off",
    )
    lines = audit_output.splitlines()
    print(next((line for line in lines if line.startswith("Found")), "no summary line"))
    table = [line for line in lines if re.match(r"(Name |---|requests |jinja2 )", line)]
    print(*list(dict.fromkeys(table))[:8], "...", sep="\n")
    print(f"exit code {exit_code} (1 = vulnerabilities found)")

Found 22 known vulnerabilities in 2 packages
Name     Version ID              Fix Versions
-------- ------- --------------- ------------
requests 2.19.0  PYSEC-2018-28   2.20.0
requests 2.19.0  PYSEC-2023-74   2.31.0
requests 2.19.0  PYSEC-2026-1873 2.32.0
requests 2.19.0  PYSEC-2026-1872 2.32.4
requests 2.19.0  PYSEC-2026-2275 2.33.0
jinja2   2.10    PYSEC-2021-66   2.11.3
...
exit code 1 (1 = vulnerabilities found)


**Typosquatting.** Attackers publish packages whose names are one keystroke away
from popular ones (`requets`, `python-dateutils`, `beautifulsoup`, `colourama` all
happened on PyPI). A quick defence when reviewing a new dependency is to compare it
with well-known names; `difflib` finds near misses:

In [58]:
import difflib

POPULAR = [
    "requests",
    "python-dateutil",
    "beautifulsoup4",
    "colorama",
    "numpy",
    "pandas",
    "urllib3",
    "setuptools",
    "cryptography",
    "pyyaml",
    "django",
    "flask",
]


def looks_like_typosquat(name: str) -> str | None:
    """The popular package this name is suspiciously close to, or None."""
    if name in POPULAR:
        return None
    close = difflib.get_close_matches(name, POPULAR, n=1, cutoff=0.85)
    return close[0] if close else None


for candidate in ["requets", "python-dateutils", "beautifulsoup", "colourama", "pydantic", "numpy"]:
    print(f"{candidate:<18} -> {looks_like_typosquat(candidate)}")

requets            -> requests
python-dateutils   -> python-dateutil
beautifulsoup      -> beautifulsoup4
colourama          -> colorama
pydantic           -> None
numpy              -> None


**Hash checking.** With `--require-hashes`, pip refuses any file whose SHA-256 is not
listed, so a swapped file on a mirror or a compromised index cannot slip in. uv
does the same with the hashes in `uv.lock`. `pip hash` prints the line to paste.
Below, the first install uses a wrong hash and is refused; the second uses the real
one.

In [59]:
wheel_path = WHEELHOUSE / "table_print-0.1.0-py3-none-any.whl"
pip("hash", str(wheel_path))
real_hash = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
for label, digest in [("tampered", "0" * 64), ("genuine", real_hash)]:
    (UNIT41_DIR / "hashed.txt").write_text(f"table-print==0.1.0 --hash=sha256:{digest}\n")
    print(f"--- {label} hash ---")
    pip(
        "install",
        "--require-hashes",
        "--force-reinstall",
        "-r",
        "hashed.txt",
        keep=r"Successfully|ERROR|Expected|Got",
    )

[<scratch>]$ python -m pip hash <scratch>/wheelhouse/table_print-0.1.0-py3-none-any.whl
  <scratch>/wheelhouse/table_print-0.1.0-py3-none-any.whl:
  --hash=sha256:5b2953bee5e74b466a94ef4c59fc1487a69ef8c0e8e4b3e73836c6e4477e6c21
--- tampered hash ---


[<scratch>]$ python -m pip install --require-hashes --force-reinstall -r hashed.txt
  ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
          Expected sha256 0000000000000000000000000000000000000000000000000000000000000000
               Got        5b2953bee5e74b466a94ef4c59fc1487a69ef8c0e8e4b3e73836c6e4477e6c21
  (exit code 1)


--- genuine hash ---


[<scratch>]$ python -m pip install --require-hashes --force-reinstall -r hashed.txt


**`deptry`** compares the imports in your code with the dependencies you declare:
`DEP001` an import that is not declared (works today only because something else
pulled it in), `DEP002` a declared dependency nobody imports, `DEP003` a transitive
dependency used directly, `DEP004` a dev dependency used in production code. First
the idea in a few lines of `ast` (Unit 3.2), offline:

In [60]:
import ast


def check_dependencies(source: str, declared: list[str]) -> dict[str, list[str]]:
    """Mini deptry: top-level imports vs declared distributions (module = name with _)."""
    imported = {
        alias.name.split(".")[0] if isinstance(node, ast.Import) else node.module.split(".")[0]
        for node in ast.walk(ast.parse(source))
        if isinstance(node, (ast.Import, ast.ImportFrom)) and getattr(node, "level", 0) == 0
        for alias in (node.names if isinstance(node, ast.Import) else [node.names[0]])
    }
    third_party = imported - set(sys.stdlib_module_names)
    declared_modules = {name.replace("-", "_").lower(): name for name in declared}
    return {
        "DEP001 missing": sorted(third_party - set(declared_modules)),
        "DEP002 unused": sorted(declared_modules[m] for m in set(declared_modules) - third_party),
    }


APP_SOURCE = "import json\nimport requests\nfrom rupee_fmt import fmt\n"
print(check_dependencies(APP_SOURCE, ["rupee-fmt", "table-print"]))

{'DEP001 missing': ['requests'], 'DEP002 unused': ['table-print']}


The real `deptry` (a Rust-accelerated tool, not a course dependency) runs through
`uvx`, which needs the network the first time to download it:

In [61]:
deptry_root = write_files(
    fresh_dir("deptry-demo"),
    {
        "pyproject.toml": """
        [project]
        name = "deptry-demo"
        version = "0.1.0"
        dependencies = ["rupee-fmt", "table-print"]
    """,
        "app.py": APP_SOURCE,
    },
)
if not network_available():
    print("SKIP: no network connection; `uvx deptry .` needs to download deptry once")
else:
    shell("uvx", "deptry", ".", "--no-ansi", cwd=deptry_root, keep=r"DEP\d|Found")

[<scratch>/deptry-demo]$ uvx deptry . --no-ansi
  app.py:2:8: DEP001 'requests' imported but missing from the dependency definitions
  pyproject.toml: DEP002 'table-print' defined as a dependency but not used in the codebase
  Found 2 dependency issues.
  (exit code 1)


**Licenses.** Installed packages declare their license in metadata: the modern
`License-Expression` field (an SPDX expression, PEP 639), the older free-text
`License`, or `License ::` classifiers. A first look at what this course installs:

In [62]:
def license_of(distribution_name: str) -> str:
    """Best available license information from installed metadata."""
    metadata = importlib.metadata.metadata(distribution_name)
    if expression := metadata.get("License-Expression"):
        return f"{expression} (SPDX)"
    classifiers = [
        c.split(" :: ")[-1]
        for c in metadata.get_all("Classifier") or []
        if c.startswith("License ::")
    ]
    if classifiers:
        return f"{', '.join(classifiers)} (classifier)"
    return (metadata.get("License") or "unknown").splitlines()[0][:40]


for name in ("mypy", "pyright", "pip-audit", "pydantic", "jupyterlab", "nbformat", "rich"):
    print(f"{name:<11} {license_of(name)}")

mypy        MIT (SPDX)
pyright     MIT
pip-audit   Apache Software License (classifier)
pydantic    MIT (SPDX)
jupyterlab  BSD License (classifier)
nbformat    BSD License (classifier)
rich        MIT License (classifier)


### 4.1.10 Tool comparison

| Python tool | What it does | Lockfile | Closest JS tool |
| --- | --- | --- | --- |
| **pip + venv** | Install into an environment you create; `requirements.txt` | none (freeze is a snapshot) | `npm install` without a lockfile |
| **pip-tools** | `pip-compile` turns `requirements.in` into a fully pinned, hashed `requirements.txt`; `pip-sync` installs exactly it | the compiled `requirements.txt` | `npm shrinkwrap` |
| **Poetry** | Projects, resolver, lockfile, build backend (`poetry-core`), publishing | `poetry.lock` | yarn (classic) |
| **PDM** | Standards-first project manager (PEP 621, PEP 735), plugin system | `pdm.lock` | pnpm |
| **Hatch** | Environments and test matrices, version bumping, the `hatchling` backend | none built in (uses others) | npm scripts + a build tool |
| **Conda** (+ mamba, pixi) | Cross-language binary packages (CUDA, GDAL, R) from conda-forge, its own environments | `environment.yml` / `pixi.lock` | nvm + apt together |
| **uv** | All of pip, pip-tools, virtualenv, pipx, pyenv (`uv python`), twine, workspaces; 10–100× faster | `uv.lock` (+ `pylock.toml` export) | pnpm + npx + nvm in one binary |

Rules of thumb: new projects use **uv**; data science with native non-Python
libraries often needs **conda/pixi**; existing Poetry or PDM projects can stay where
they are (all of them read PEP 621 `[project]` today, so moving is cheap). Which of
these are on this machine:

In [63]:
TOOLS = {
    "pip": "pip",
    "pip-compile": "pip-tools",
    "poetry": "Poetry",
    "pdm": "PDM",
    "hatch": "Hatch",
    "conda": "Conda",
    "pixi": "pixi",
    "uv": "uv",
}
for command, product in TOOLS.items():
    found = executable(command)
    version = run_tool(command, "--version")[1].splitlines()[0] if found else "not installed"
    print(f"{product:<10} {command:<12} {tidy(version)}")

pip        pip          pip 26.3 from <repo>/.venv/lib/python3.15/site-packages/pip (python 3.15)
pip-tools  pip-compile  not installed
Poetry     poetry       not installed
PDM        pdm          not installed
Hatch      hatch        not installed
Conda      conda        not installed
pixi       pixi         not installed
uv         uv           uv 0.13.5 (x86_64-unknown-linux-gnu)


**Check: the course project was never touched.** Every demo above ran in
`notebooks/_scratch/unit41/`. The course's `pyproject.toml` and `uv.lock` still
have the digests recorded by the unit's setup cell:

In [64]:
for name in COURSE_FILES:
    unchanged = file_digest(REPO_ROOT / name) == COURSE_DIGESTS_AT_START[name]
    print(f"{name:<15} unchanged: {unchanged}")

pyproject.toml  unchanged: True
uv.lock         unchanged: True


### Lab 4.1

Four exercises. Complete each stub, then run its check cell. Until then the checks
print FAIL; that is expected. The checks build and run your work with uv in
`notebooks/_scratch/unit41/lab/`, offline, against the local wheelhouse.

#### Exercise 1 — annotate every line of a `pyproject.toml`

Return `PLAIN_PYPROJECT` (below) with a comment on **every** non-blank line,
explaining what the line does. A comment is `# ` followed by at least three words,
either at the end of the line or on its own line directly above it. The TOML must
still parse to exactly the same data.

In [65]:
PLAIN_PYPROJECT = """
[build-system]
requires = ["uv_build>=0.13,<0.14"]
build-backend = "uv_build"

[project]
name = "shipping-rates"
version = "0.3.0"
requires-python = ">=3.12"
dependencies = ["rupee-fmt~=1.1", "win-colors; sys_platform == 'win32'"]

[project.optional-dependencies]
report = ["table-print>=0.1"]

[project.scripts]
ship-quote = "shipping_rates.cli:main"

[dependency-groups]
dev = ["gst-calc>=1,<2"]

[tool.ruff]
line-length = 100
"""


def annotated_pyproject() -> str:
    """PLAIN_PYPROJECT with a comment for every non-blank line."""
    raise NotImplementedError("Lab 4.1 exercise 1: annotate the file")

Run the checks.

In [66]:
def annotation_problems(plain: str, annotated: str) -> list[str]:
    """Lines of plain that have no 3-word comment (inline or on the line above) in annotated."""
    problems: list[str] = []
    comment_above = False
    content_lines = [line.strip() for line in textwrap.dedent(plain).splitlines() if line.strip()]
    seen = 0
    for line in textwrap.dedent(annotated).splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            comment_above = len(stripped.lstrip("# ").split()) >= 3
            continue
        if not stripped:
            comment_above = False
            continue
        content, _, inline = stripped.partition(" # ")
        if len(inline.split()) < 3 and not comment_above:
            problems.append(content.strip())
        seen += 1
        comment_above = False
    if seen != len(content_lines):
        problems.append(f"expected {len(content_lines)} TOML lines, found {seen}")
    return problems


run_checks(
    [
        (
            "parses to the same data",
            lambda: (
                tomllib.loads(textwrap.dedent(annotated_pyproject()))
                == tomllib.loads(textwrap.dedent(PLAIN_PYPROJECT))
            ),
            True,
        ),
        (
            "every line annotated",
            lambda: annotation_problems(PLAIN_PYPROJECT, annotated_pyproject()),
            [],
        ),
        (
            "still a build-system table",
            lambda: tomllib.loads(textwrap.dedent(annotated_pyproject()))["build-system"][
                "build-backend"
            ],
            "uv_build",
        ),
    ]
)

FAIL  parses to the same data: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
FAIL  every line annotated: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
FAIL  still a build-system table: raised NotImplementedError: Lab 4.1 exercise 1: annotate the file
0/3 checks passed


#### Exercise 2 — a `src/`-layout library with a console script, extras and groups

Return the files (relative path → text) of a project that builds with uv:

| Requirement | Detail |
| --- | --- |
| Name, version, Python | `invoice-tools`, `0.1.0`, `requires-python = ">=3.12"`, no runtime dependencies |
| Build backend | `uv_build` (`requires = ["uv_build>=0.13,<0.14"]`) |
| Layout | `src/invoice_tools/` with `__init__.py`, `cli.py` and `py.typed`; nothing importable at the root |
| `__version__` | read with `importlib.metadata.version("invoice-tools")` in `__init__.py` |
| Console script | `invoice-total` → `invoice_tools.cli:main`, which prints `total: <sum>` of its arguments with two decimals (`invoice-total 10 20.5` prints `total: 30.50`) |
| Extra | `report` → `table-print>=0.1` |
| Dependency group | `dev` → `rupee-fmt>=1` (must **not** appear in the wheel metadata) |

In [67]:
def library_files() -> dict[str, str]:
    """Relative path -> file content for the invoice-tools project."""
    raise NotImplementedError("Lab 4.1 exercise 2: write the project files")

Run the checks. They build your project with `uv build`, then inspect and run the wheel.

In [68]:
LAB_DIR = UNIT41_DIR / "lab"


_learner_builds: dict[str, tuple[Path, dict[str, str]]] = {}


def build_learner_library() -> tuple[Path, dict[str, str]]:
    """Build library_files() with uv (once per distinct content); return (wheel, file texts)."""
    key = json.dumps(library_files(), sort_keys=True)
    if key not in _learner_builds:
        _learner_builds[key] = _build_library()
    return _learner_builds[key]


def _build_library() -> tuple[Path, dict[str, str]]:
    """Write library_files() under lab/, build with uv, read METADATA and entry points."""
    root = write_files(fresh_dir("lab/invoice-tools"), library_files())
    exit_code, output = run_tool("uv", "build", "--offline", cwd=root)
    if exit_code:
        raise RuntimeError(output.splitlines()[-1])
    wheel_path = next((root / "dist").glob("*.whl"))
    with zipfile.ZipFile(wheel_path) as wheel:
        names = wheel.namelist()
        dist_info = next(name.split("/")[0] for name in names if ".dist-info/" in name)
        report = {
            "names": "\n".join(names),
            "metadata": wheel.read(f"{dist_info}/METADATA").decode(),
            "entry_points": wheel.read(f"{dist_info}/entry_points.txt").decode(),
        }
    return wheel_path, report


def run_learner_cli(*arguments: str) -> str:
    """Run invoice-total from the built wheel with uvx."""
    wheel_path, _ = build_learner_library()
    return run_tool(
        "uvx", "--quiet", "--offline", "--from", str(wheel_path), "invoice-total", *arguments
    )[1]


def learner_version() -> str:
    wheel_path, _ = build_learner_library()
    return run_tool(
        "uv",
        "run",
        "--quiet",
        "--offline",
        "--no-project",
        "--with",
        str(wheel_path),
        "python",
        "-c",
        "import invoice_tools; print(invoice_tools.__version__)",
        cwd=LAB_DIR,
    )[1]


run_checks(
    [
        (
            "src layout",
            lambda: sorted(p for p in library_files() if p.startswith("src/invoice_tools/")),
            [
                "src/invoice_tools/__init__.py",
                "src/invoice_tools/cli.py",
                "src/invoice_tools/py.typed",
            ],
        ),
        (
            "nothing importable at the root",
            lambda: [p for p in library_files() if p.endswith(".py") and "/" not in p],
            [],
        ),
        (
            "console script in entry_points.txt",
            lambda: (
                "invoice-total = invoice_tools.cli:main"
                in build_learner_library()[1]["entry_points"]
            ),
            True,
        ),
        (
            "extra published",
            lambda: "Provides-Extra: report" in build_learner_library()[1]["metadata"],
            True,
        ),
        (
            "extra requirement",
            lambda: (
                re.search(
                    r"Requires-Dist: table-print>=0\.1 ?; extra == .report.",
                    build_learner_library()[1]["metadata"],
                )
                is not None
            ),
            True,
        ),
        (
            "dev group not published",
            lambda: "rupee-fmt" in build_learner_library()[1]["metadata"],
            False,
        ),
        (
            "py.typed in the wheel",
            lambda: "invoice_tools/py.typed" in build_learner_library()[1]["names"],
            True,
        ),
        ("invoice-total 10 20.5", lambda: run_learner_cli("10", "20.5"), "total: 30.50"),
        ("__version__ from metadata", learner_version, "0.1.0"),
    ]
)

FAIL  src layout: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  nothing importable at the root: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  console script in entry_points.txt: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  extra published: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  extra requirement: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  dev group not published: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  py.typed in the wheel: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  invoice-total 10 20.5: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
FAIL  __version__ from metadata: raised NotImplementedError: Lab 4.1 exercise 2: write the project files
0/9 checks passed


#### Exercise 3 — a PEP 723 single-file script

Return the source of a script with an inline metadata block that declares
`requires-python = ">=3.12"` and the single dependency `rupee-fmt>=2`. When run, it
prints two lines: `fmt(1234.5)` from `rupee_fmt`, then `rupee-fmt <version>` using
`importlib.metadata`. The check runs it with `uv run` against the wheelhouse.
Which version should the resolver choose, given that `2.1.0rc1` exists?

In [69]:
def pep723_script() -> str:
    """Source of a self-describing script for `uv run`."""
    raise NotImplementedError("Lab 4.1 exercise 3: write the script")

Run the checks. `read_script_metadata` uses the reference regular expression from PEP 723.

In [70]:
PEP723_BLOCK = re.compile(
    r"(?m)^# /// (?P<type>[a-zA-Z0-9-]+)$\s(?P<content>(^#(| .*)$\s)+)^# ///$"
)


def read_script_metadata(script: str) -> dict[str, object]:
    """Parse the `# /// script` block of a PEP 723 script."""
    script = textwrap.dedent(script)
    match = next((m for m in PEP723_BLOCK.finditer(script) if m["type"] == "script"), None)
    if match is None:
        raise ValueError("no `# /// script` block")
    toml_text = "".join(
        line[2:] if line.startswith("# ") else line[1:]
        for line in match["content"].splitlines(keepends=True)
    )
    return tomllib.loads(toml_text)


def run_learner_script() -> list[str]:
    root = fresh_dir("lab/script")
    (root / "quote.py").write_text(textwrap.dedent(pep723_script()))
    output = run_tool(
        "uv",
        "run",
        "--quiet",
        "--offline",
        "--no-index",
        "--find-links",
        str(WHEELHOUSE),
        "quote.py",
        cwd=root,
    )[1]
    return output.splitlines()[-2:]


run_checks(
    [
        (
            "requires-python",
            lambda: read_script_metadata(pep723_script())["requires-python"],
            ">=3.12",
        ),
        (
            "dependencies",
            lambda: [
                d.replace(" ", "") for d in read_script_metadata(pep723_script())["dependencies"]
            ],
            ["rupee-fmt>=2"],
        ),
        ("output", run_learner_script, ["\u20b91,234.50", "rupee-fmt 2.0.0"]),
    ]
)

FAIL  requires-python: raised NotImplementedError: Lab 4.1 exercise 3: write the script
FAIL  dependencies: raised NotImplementedError: Lab 4.1 exercise 3: write the script
FAIL  output: raised NotImplementedError: Lab 4.1 exercise 3: write the script
0/3 checks passed


#### Exercise 4 — build a wheel and inspect its contents

Write `wheel_summary(wheel_path)` returning a dict with these keys, read only from
the wheel file (use `zipfile` and `email.parser`; do not install anything):

| Key | Value |
| --- | --- |
| `"name"`, `"version"` | from `METADATA` |
| `"tags"` | list of the `Tag:` values in `WHEEL` |
| `"packages"` | sorted top-level folders or modules outside `.dist-info/` |
| `"console_scripts"` | dict command → `module:function` from `entry_points.txt` (empty if none) |
| `"requires_dist"` | list of `Requires-Dist` values |
| `"record_ok"` | `True` if every file listed in `RECORD` with a hash matches its SHA-256 |

The check uses `gst-calc 1.0.0` from the wheelhouse and a tampered copy of it.

In [71]:
def wheel_summary(wheel_path: Path) -> dict[str, object]:
    """Facts about a wheel, read from its files."""
    raise NotImplementedError("Lab 4.1 exercise 4: inspect the wheel")

Run the checks.

In [72]:
GST_WHEEL = build_wheelhouse() / "gst_calc-1.0.0-py3-none-any.whl"


def tampered_wheel() -> Path:
    """A copy of GST_WHEEL whose gst_calc/__init__.py was edited after building."""
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    target = LAB_DIR / GST_WHEEL.name
    with zipfile.ZipFile(GST_WHEEL) as source, zipfile.ZipFile(target, "w") as copy:
        for item in source.infolist():
            data = source.read(item)
            if item.filename == "gst_calc/__init__.py":
                data += b"\nprint('surprise')\n"
            copy.writestr(item, data)
    return target


run_checks(
    [
        ("name", lambda: wheel_summary(GST_WHEEL)["name"], "gst-calc"),
        ("version", lambda: wheel_summary(GST_WHEEL)["version"], "1.0.0"),
        ("tags", lambda: wheel_summary(GST_WHEEL)["tags"], ["py3-none-any"]),
        ("packages", lambda: wheel_summary(GST_WHEEL)["packages"], ["gst_calc"]),
        (
            "console scripts",
            lambda: wheel_summary(GST_WHEEL)["console_scripts"],
            {"gst": "gst_calc:main"},
        ),
        ("three requirements", lambda: len(wheel_summary(GST_WHEEL)["requires_dist"]), 3),
        (
            "one extra requirement",
            lambda: sum("extra ==" in r for r in wheel_summary(GST_WHEEL)["requires_dist"]),
            1,
        ),
        ("RECORD verifies", lambda: wheel_summary(GST_WHEEL)["record_ok"], True),
        ("tampering detected", lambda: wheel_summary(tampered_wheel())["record_ok"], False),
    ]
)

FAIL  name: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  version: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  tags: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  packages: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  console scripts: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  three requirements: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  one extra requirement: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  RECORD verifies: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
FAIL  tampering detected: raised NotImplementedError: Lab 4.1 exercise 4: inspect the wheel
0/9 checks passed


### Recap — Unit 4.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Wheels and tags | `py3-none-any` fits everywhere; `cp315-cp315t-manylinux_2_28_x86_64` fits one interpreter, ABI and platform | prebuilt binaries per platform (`node-pre-gyp`) |
| sdist | Source + `pyproject.toml`; installing it runs the build backend | publishing raw source with an `install` script |
| Frontend / backend | uv and pip call PEP 517 hooks of `uv_build`, `hatchling`, `setuptools`, … | none |
| pip | Backtracking resolver; `-e` editable; `-c` constraints; `freeze` is not a lock | npm |
| uv | `init`, `add`, `lock --upgrade-package`, `sync --locked/--frozen`, `run`, `tool`, `uvx`, scripts, workspaces, `export`, `build`, `publish` | pnpm + npx + nvm |
| `pyproject.toml` | `[build-system]`, `[project]`, scripts, extras, `[dependency-groups]`, `[tool.*]` | `package.json` |
| PEP 440 | `~=1.4` is `>=1.4,==1.*`; pre-releases opt-in; markers and extras | semver ranges |
| Lockfiles | `uv.lock` (universal), `pylock.toml` (standard); apps pin, libraries stay wide | `package-lock.json` |
| Layout | `src/` forces tests to use the installed package; editable = `.pth` | `npm link` |
| Publishing | Inspect the wheel, rehearse on TestPyPI, use trusted publishing; SemVer or CalVer; keep a changelog | `npm publish --provenance` |
| Supply chain | `pip-audit`, hashes, `deptry`, name checks, licenses | `npm audit`, lockfile integrity |

**Further reading**

- [Python Packaging User Guide](https://packaging.python.org/),
  [`pyproject.toml` specification](https://packaging.python.org/en/latest/specifications/pyproject-toml/),
  [Version specifiers](https://packaging.python.org/en/latest/specifications/version-specifiers/),
  [Platform compatibility tags](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/),
  [Binary distribution format](https://packaging.python.org/en/latest/specifications/binary-distribution-format/)
- [uv documentation](https://docs.astral.sh/uv/) (projects, workspaces, scripts,
  tools, publishing), [pip documentation](https://pip.pypa.io/)
- [`importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html),
  [`tomllib`](https://docs.python.org/3/library/tomllib.html),
  [`zipfile`](https://docs.python.org/3/library/zipfile.html),
  [`venv`](https://docs.python.org/3/library/venv.html)
- [PEP 440](https://peps.python.org/pep-0440/), [PEP 508](https://peps.python.org/pep-0508/),
  [PEP 517](https://peps.python.org/pep-0517/), [PEP 518](https://peps.python.org/pep-0518/),
  [PEP 621](https://peps.python.org/pep-0621/), [PEP 639](https://peps.python.org/pep-0639/),
  [PEP 660](https://peps.python.org/pep-0660/), [PEP 723](https://peps.python.org/pep-0723/),
  [PEP 735](https://peps.python.org/pep-0735/), [PEP 751](https://peps.python.org/pep-0751/)
- [PyPI trusted publishers](https://docs.pypi.org/trusted-publishers/),
  [Keep a Changelog](https://keepachangelog.com/), [pip-audit](https://github.com/pypa/pip-audit),
  [deptry](https://deptry.com/)

## Unit 4.2 — The Type System (Python's TypeScript)

**Objectives**

- Explain gradual typing: what a type checker checks, what the interpreter ignores,
  and why Python classes are nominal while Protocols are structural.
- Annotate variables, parameters, returns, class and constant attributes, and know
  when annotations are evaluated (*Changed in 3.14*).
- Use collection, union, literal, `TypedDict`, `NewType`, `Self` and generic types;
  narrow unions so the checker can prove your code safe.
- Type callables and decorators precisely (`ParamSpec`, `Concatenate`, `@overload`)
  and write your own narrowing functions (`TypeIs`).
- Read type hints at run time (`Annotated`, `get_type_hints`) the way Pydantic and
  FastAPI do.
- Run mypy and pyright in strict mode, configure them, and adopt them gradually.

**TS bridge: Python's TypeScript**

| TypeScript | Python | The difference that bites |
| --- | --- | --- |
| `tsc` (part of the toolchain) | mypy, pyright (separate tools) | **Python never refuses to run** badly typed code; the interpreter ignores hints |
| Types erased at compile time | Annotations kept at run time as objects | Libraries (Pydantic, FastAPI, dataclasses) read them |
| `let total: number = 0` | `total: int = 0` | `int` and `float` are different types (an `int` is accepted where `float` is expected) |
| `string \| null` (`strictNullChecks`) | `str \| None` | Same idea; mypy always treats `None` strictly |
| `any` | `Any` | Same: disables checking |
| `unknown` | `object` | Must narrow before use |
| `value as Product` | `cast(Product, value)` | Neither checks anything at run time |
| `// @ts-expect-error` | `# type: ignore[code]` | Give the error code so other errors still show |
| Structural typing everywhere | **Nominal** classes, structural `Protocol`s | A class with the right shape is *not* accepted unless you use a Protocol |
| `interface Address { city: string; pin?: string }` | `class Address(TypedDict)` with `NotRequired` | `TypedDict` is only for dicts |
| `"GET" \| "POST"` | `Literal["GET", "POST"]` | Same |
| `function first<T>(xs: T[]): T` | `def first[T](items: Sequence[T]) -> T` | PEP 695 syntax (3.12) |
| `value is Product` (type predicate) | `-> TypeIs[Product]` | 3.13 |
| `never` | `Never` / `NoReturn` | Same |
| `readonly` | `Final`, `ReadOnly`, frozen dataclasses | Checked statically only |
| `.d.ts`, DefinitelyTyped | `.pyi` stubs, typeshed, `types-*` packages | Same idea |
| `"strict": true` in `tsconfig.json` | `mypy --strict`, pyright `strict` | Configure in `pyproject.toml` |

**How the checker demos run.** The interpreter does not check types, so demos that
show checker errors write a small file to `notebooks/_scratch/unit42/` and run the
real mypy and/or pyright on it in a child process: `check(source)` prints both
outputs. Line numbers count from the first line of the source string. Error codes
in brackets (`[arg-type]`, `[reportArgumentType]`) are what you put in an ignore
comment or search for in the docs.

In [73]:
import annotationlib
import functools
import importlib.util
import inspect
import json
import linecache
import os
import re
import subprocess
import sys
import textwrap
import time
import typing
from collections.abc import Callable, Iterable, Mapping, Sequence  # noqa: F401  (lab hints use them)
from dataclasses import dataclass, field, fields
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Annotated, Literal

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
UNIT42_DIR = scratch_dir / "unit42"
UNIT42_DIR.mkdir(parents=True, exist_ok=True)
VENV_BIN = Path(sys.executable).parent
CHECKER_ENV = {
    **os.environ,
    "NO_COLOR": "1",
    "FORCE_COLOR": "0",
    "MYPY_FORCE_COLOR": "0",
    "PYTHON_COLORS": "0",
}
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
PAISE = Decimal("0.01")


def write_demo(source: str, name: str, header: str = "") -> Path:
    """Write dedented source to <scratch>/<name>.py and return the path."""
    path = UNIT42_DIR / f"{name}.py"
    path.write_text(header + textwrap.dedent(source).lstrip("\n"))
    return path


def run_mypy(
    source: str, *, name: str = "demo", strict: bool = False, options: tuple[str, ...] = ()
) -> list[str]:
    """Type-check source with mypy in a child process; return its diagnostics, one per line."""
    mypy = VENV_BIN / "mypy"
    if not mypy.exists():
        return ["SKIP: mypy is not installed (uv add --dev mypy)"]
    path = write_demo(source, name)
    command = [
        str(mypy),
        "--no-error-summary",
        "--no-color-output",
        "--no-pretty",
        "--cache-dir",
        str(UNIT42_DIR / ".mypy_cache"),
        *(["--strict"] if strict else []),
        *options,
        path.name,
    ]
    completed = subprocess.run(
        command, cwd=UNIT42_DIR, capture_output=True, text=True, env=CHECKER_ENV, timeout=300
    )
    return (completed.stdout + completed.stderr).strip().splitlines()


def run_pyright(source: str, *, name: str = "demo", strict: bool = False) -> list[str]:
    """Type-check source with pyright (JSON output) in a child process; one line per finding."""
    pyright = VENV_BIN / "pyright"
    if not pyright.exists():
        return ["SKIP: pyright is not installed (uv add --dev pyright)"]
    path = write_demo(source, name, "# pyright: strict\n" if strict else "")
    completed = subprocess.run(
        [str(pyright), "--outputjson", "--pythonpath", sys.executable, path.name],
        cwd=UNIT42_DIR,
        capture_output=True,
        text=True,
        env=CHECKER_ENV,
        timeout=300,
    )
    try:
        report = json.loads(completed.stdout)
    except ValueError:
        return [
            f"SKIP: pyright did not run: {(completed.stderr or completed.stdout).strip()[:120]}"
        ]
    findings = []
    for diagnostic in report["generalDiagnostics"]:
        line_number = diagnostic["range"]["start"]["line"] + 1 - (1 if strict else 0)
        message = " ".join(part.strip() for part in diagnostic["message"].splitlines()[:2])
        rule = f"  [{diagnostic['rule']}]" if diagnostic.get("rule") else ""
        findings.append(f"{path.name}:{line_number}: {diagnostic['severity']}: {message}{rule}")
    return findings


def check(
    source: str,
    *,
    name: str = "demo",
    strict: bool = False,
    checkers: tuple[str, ...] = ("mypy", "pyright"),
) -> None:
    """Print what each checker reports for source (line numbers count from its first line)."""
    for checker in checkers:
        runner = run_mypy if checker == "mypy" else run_pyright
        findings = runner(source, name=name, strict=strict)
        print(f"{checker}{' (strict)' if strict else ''}:")
        print(textwrap.indent("\n".join(findings) or "no issues", "  "))


def source_of(obj: object) -> str:
    """Source code of a function or class, also for classes defined in a notebook cell."""
    try:
        return textwrap.dedent(inspect.getsource(obj))
    except OSError, TypeError:
        if not isinstance(obj, type):
            raise
    # inspect cannot find the file of a notebook class; use the file of one of its functions
    for member in vars(obj).values():
        member = getattr(member, "__func__", None) or getattr(member, "fget", None) or member
        code_object = getattr(inspect.unwrap(member), "__code__", None)
        lines = linecache.getlines(code_object.co_filename) if code_object else []
        if lines:
            start = obj.__firstlineno__ - 1
            while start > 0 and lines[start - 1].lstrip().startswith("@"):
                start -= 1
            return textwrap.dedent("".join(inspect.getblock(lines[start:])))
    # no functions (e.g. an exception class with only a docstring): search the cached cell sources
    header = re.compile(rf"\s*class\s+{obj.__name__}\b")
    for _size, _mtime, lines, _name in reversed(list(linecache.cache.values())):
        if len(lines) >= obj.__firstlineno__ and header.match(lines[obj.__firstlineno__ - 1]):
            return textwrap.dedent("".join(inspect.getblock(lines[obj.__firstlineno__ - 1 :])))
    raise OSError(f"source of {obj.__name__} not found")


LAB_PRELUDE = """
import functools
import typing
import re
import time
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass, field
from decimal import ROUND_HALF_UP, Decimal
from typing import Annotated, Any, Literal, Self, cast, get_type_hints, overload, reveal_type

PAISE = Decimal("0.01")
"""


@functools.cache
def mypy_report(source: str, strict: bool = True) -> tuple[str, ...]:
    """mypy's lines for LAB_PRELUDE + source (cached: identical source is checked once)."""
    return tuple(run_mypy(LAB_PRELUDE + "\n\n" + source, name="lab_check", strict=strict))


def mypy_errors(*objects: object, strict: bool = True, extra: str = "") -> list[str]:
    """Error lines (no notes) mypy reports for the source of objects, plus extra code."""
    source = "\n\n".join([*(source_of(obj) for obj in objects), textwrap.dedent(extra)])
    return [
        line.split(": error: ", 1)[1] for line in mypy_report(source, strict) if ": error: " in line
    ]


def revealed_types(*objects: object, extra: str = "") -> list[str]:
    """The types mypy reveals for reveal_type(...) calls in extra (after the objects' source)."""
    source = "\n\n".join([*(source_of(obj) for obj in objects), textwrap.dedent(extra)])
    return re.findall(r'Revealed type is "([^"]+)"', "\n".join(mypy_report(source, True)))


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


for tool in ("mypy", "pyright"):
    found = VENV_BIN / tool
    if found.exists():
        version = subprocess.run([str(found), "--version"], capture_output=True, text=True).stdout
        print(f"{tool}: {version.strip()}")
    else:
        print(f"{tool}: not installed; checker cells will print a skip message")

mypy: mypy 2.4.1 (compiled: yes)


pyright: pyright 1.1.414


### 4.2.1 Gradual typing

**PEP 484** (2014) defined the type-hint syntax and its meaning, but left checking to
external tools. Three facts follow:

1. **The interpreter ignores hints.** It stores them and moves on; a badly typed
   call runs (and maybe fails later, or silently does the wrong thing).
2. **Static checkers** (mypy, pyright, and newer ones such as `ty` and Pyrefly)
   read the source without running it and report what cannot be proven safe.
3. **Typing is gradual.** Unannotated code is treated as `Any`, so typed and untyped
   code mix freely and you can add hints one module at a time, like `allowJs` and
   `// @ts-check` in a JavaScript project moving to TypeScript.

The interpreter runs this badly typed call without complaint:

In [74]:
def label(sku: str, quantity: int) -> str:
    """Return a shelf label such as "3 x TEA-500"."""
    return f"{quantity} x {sku}"


print(f"{label('TEA-500', 3)=}")
print(f"{label(3, 'TEA-500')=}  (wrong order: runs anyway, wrong answer)")
print(f"{label.__annotations__=}")

label('TEA-500', 3)='3 x TEA-500'
label(3, 'TEA-500')='TEA-500 x 3'  (wrong order: runs anyway, wrong answer)
label.__annotations__={'sku': <class 'str'>, 'quantity': <class 'int'>, 'return': <class 'str'>}


mypy, given the same code as a file, reports both swapped arguments before anything runs:

In [75]:
check(
    """
    def label(sku: str, quantity: int) -> str:
        return f"{quantity} x {sku}"

    label(3, "TEA-500")
    """,
    checkers=("mypy",),
)

mypy:
  demo.py:4: error: Argument 1 to "label" has incompatible type "int"; expected "str"  [arg-type]
  demo.py:4: error: Argument 2 to "label" has incompatible type "str"; expected "int"  [arg-type]


**Gradual means unannotated code is not checked (by default).** mypy skips the
bodies of functions without annotations and treats their results as `Any`.
`--strict` (or `--check-untyped-defs`) turns that off. pyright *infers* types inside
unannotated functions, so it finds more even in its default mode.

In [76]:
UNTYPED = """
def legacy_total(items):
    return sum(item.price for item in items) + "INR"   # str + number: a bug

def report(prices: list[float]) -> str:
    return legacy_total(prices)                        # Any: accepted silently
"""
check(UNTYPED, name="untyped", checkers=("mypy",))
check(UNTYPED, name="untyped", strict=True, checkers=("mypy",))
check(UNTYPED, name="untyped", checkers=("pyright",))

mypy:
  no issues


mypy (strict):
  untyped.py:1: error: Function is missing a type annotation  [no-untyped-def]
  untyped.py:5: error: Returning Any from function declared to return "str"  [no-any-return]
  untyped.py:5: error: Call to untyped function "legacy_total" in typed context  [no-untyped-call]


pyright:
  untyped.py:2: error: Operator "+" not supported for types "int" and "Literal['INR']"  [reportOperatorIssue]


**Nominal vs structural typing.** TypeScript is *structural*: any value with the
right shape fits an interface or class type. Python classes are *nominal*: a
parameter typed `Duck` accepts `Duck` and its subclasses only. For structural typing
you opt in with a `typing.Protocol` (Unit 2.3.3).

```typescript
class Duck  { quack(): string { return "Quack"; } }
class Robot { quack(): string { return "Beep";  } }
function makeNoise(duck: Duck) { return duck.quack(); }
makeNoise(new Robot());   // OK in TypeScript: same shape
```

In [77]:
check(
    """
    from typing import Protocol

    class Duck:
        def quack(self) -> str:
            return "Quack"

    class Robot:                      # same shape, unrelated class
        def quack(self) -> str:
            return "Beep"

    class Quacker(Protocol):          # structural: anything with quack() -> str
        def quack(self) -> str: ...

    def make_noise(duck: Duck) -> str:
        return duck.quack()

    def make_any_noise(quacker: Quacker) -> str:
        return quacker.quack()

    make_noise(Robot())               # nominal: rejected
    make_any_noise(Robot())           # structural: accepted
    """,
    name="nominal",
    checkers=("mypy",),
)

mypy:
  nominal.py:20: error: Argument 1 to "make_noise" has incompatible type "Robot"; expected "Duck"  [arg-type]


**Predict the output.**

```python
def double(value: int) -> int:
    return value * 2

print(double("ab"), double([0]))
```

<details><summary>Answer</summary>

`abab [0, 0]`. The interpreter never checks hints, and `*` works for strings and
lists. mypy would report both calls (`[arg-type]`); at run time you only find out
when the wrong result causes trouble somewhere else.

</details>

Run the next cell to check your prediction.

In [78]:
def double(value: int) -> int:
    return value * 2


print(double("ab"), double([0]))

abab [0, 0]


### 4.2.2 Annotation syntax

| Where | Syntax | TypeScript |
| --- | --- | --- |
| Variable | `retries: int = 3` | `let retries: number = 3` |
| Declaration only | `customer_name: str` (no value: **no variable is created**) | `let customerName: string;` |
| Parameter, default | `def ship(weight_kg: float, express: bool = False)` | `(weightKg: number, express = false)` |
| `*args`, `**kwargs` | `*skus: str`, `**options: int` (type of **each** item) | `...skus: string[]` |
| Return | `-> Decimal`, `-> None` | `: Decimal`, `: void` |
| Class attribute shared by all instances | `TAX_RATE: ClassVar[Decimal]` | `static TAX_RATE: Decimal` |
| Constant | `MAX_RETRIES: Final = 3` | `const MAX_RETRIES = 3` |

In [79]:
retries: int = 3


class CheckoutSettings:
    currency: str = "INR"
    customer_name: str  # annotation only: recorded, but no attribute is created


def running_total() -> int:
    total: int  # annotation only: no local variable yet
    return total  # noqa: F821  (deliberate: no value was assigned)


def ship(*skus: str, express: bool = False, **weights_kg: float) -> None:
    """Each sku is a str; each keyword argument value is a float."""


print(f"{retries=}  {CheckoutSettings.__annotations__=}")
print(f"{hasattr(CheckoutSettings, 'customer_name')=}")
print(f"{raised_message(running_total)=}")
print(f"{inspect.signature(ship)=}")
print(f"{ship.__annotations__=}")

retries=3  CheckoutSettings.__annotations__={'currency': <class 'str'>, 'customer_name': <class 'str'>}
hasattr(CheckoutSettings, 'customer_name')=False
raised_message(running_total)="UnboundLocalError: cannot access local variable 'total' where it is not associated with a value"
inspect.signature(ship)=<Signature (*skus: str, express: bool = False, **weights_kg: float) -> None>
ship.__annotations__={'skus': <class 'str'>, 'express': <class 'bool'>, 'weights_kg': <class 'float'>, 'return': None}


**`ClassVar` and `Final`.** `ClassVar` marks an attribute that belongs to the class,
not to instances; dataclasses leave `ClassVar` fields out of `__init__`. `Final`
marks a name that must not be re-bound (with no type given, the checker infers it
from the value). Neither is enforced at run time:

In [80]:
from typing import ClassVar, Final

MAX_RETRIES: Final = 3


@dataclass
class InvoiceLine:
    TAX_RATE: ClassVar[Decimal] = Decimal("0.18")  # class-level, not a field
    description: str
    amount: Decimal


line = InvoiceLine("Tea", Decimal("100"))
print(f"{[f.name for f in fields(InvoiceLine)]=}  (TAX_RATE is not a field)")
MAX_RETRIES = 5  # the interpreter allows it; the checker below does not
print(f"{MAX_RETRIES=}")

[f.name for f in fields(InvoiceLine)]=['description', 'amount']  (TAX_RATE is not a field)
MAX_RETRIES=5


In [81]:
check(
    """
    from dataclasses import dataclass
    from decimal import Decimal
    from typing import ClassVar, Final

    MAX_RETRIES: Final = 3
    MAX_RETRIES = 5

    @dataclass
    class InvoiceLine:
        TAX_RATE: ClassVar[Decimal] = Decimal("0.18")
        description: str
        amount: Decimal

    line = InvoiceLine("Tea", Decimal("100"))
    line.TAX_RATE = Decimal("0.05")
    """,
    name="classvar_final",
    checkers=("mypy",),
)

mypy:
  classvar_final.py:6: error: Cannot assign to final name "MAX_RETRIES"  [misc]
  classvar_final.py:15: error: Cannot assign to class variable "TAX_RATE" via instance  [misc]


**Deferred evaluation of annotations (PEP 649 / PEP 749, *Changed in 3.14*).** Until
3.13, annotations were evaluated when the `def` or `class` statement ran, so a class
could not mention a class defined later without quoting it (`"Customer"`) or using
`from __future__ import annotations`. Since 3.14 the compiler stores annotations in
a generated `__annotate__` function and evaluates them **only when someone asks**.
Forward references just work, and modules that never read their annotations do not
pay for them.

In [82]:
class Order:
    customer: Customer  # defined further down: fine in 3.14+

    def ship_to(self) -> Address: ...


print(f"{Order.__annotate__=}")
print(f"{raised_message(lambda: Order.__annotations__)=}  (Customer does not exist yet)")

Order.__annotate__=<function Order.__annotate__ at 0x7fe5540ede80>
raised_message(lambda: Order.__annotations__)="NameError: name 'Customer' is not defined"  (Customer does not exist yet)


The new standard-library module **`annotationlib`** (*New in 3.14*) reads
annotations in three formats, so tools can cope with names that do not exist yet:

| `Format` | Result for `customer: Customer` before `Customer` exists |
| --- | --- |
| `VALUE` (what `__annotations__` uses) | raises `NameError` |
| `FORWARDREF` | real objects where possible, `ForwardRef('Customer')` otherwise |
| `STRING` | the source text: `'Customer'` |

In [83]:
from annotationlib import Format, get_annotations

print(f"{get_annotations(Order, format=Format.FORWARDREF)=}")
print(f"{get_annotations(Order, format=Format.STRING)=}")
print(f"{get_annotations(Order.ship_to, format=Format.STRING)=}")


class Customer:
    name: str


print(f"{get_annotations(Order, format=Format.VALUE)=}  (now it resolves)")

get_annotations(Order, format=Format.FORWARDREF)={'customer': ForwardRef('Customer', is_class=True, owner=<class '__main__.Order'>)}
get_annotations(Order, format=Format.STRING)={'customer': 'Customer'}
get_annotations(Order.ship_to, format=Format.STRING)={'return': 'Address'}
get_annotations(Order, format=Format.VALUE)={'customer': <class '__main__.Customer'>}  (now it resolves)


> **Note:** `from __future__ import annotations` (PEP 563, stringified annotations)
> is no longer needed on 3.14+ and is expected to be deprecated. Libraries that read
> annotations should call `annotationlib.get_annotations()` or
> `typing.get_type_hints()` rather than touching `__annotations__` directly.

### 4.2.3 Collection types

Built-in collections are generic since 3.9: subscript them with the element type.

| Python | Means | TypeScript |
| --- | --- | --- |
| `list[int]` | list of ints | `number[]` |
| `dict[str, list[User]]` | str keys, values are lists of `User` | `Record<string, User[]>` |
| `set[str]`, `frozenset[str]` | sets of str | `Set<string>` |
| `tuple[int, str]` | **exactly two** items: an int then a str | `[number, string]` |
| `tuple[int, ...]` | **any number** of ints | `readonly number[]` |
| `tuple[()]` | the empty tuple | `[]` |

`typing.List`, `typing.Dict` and friends are deprecated aliases; ruff's `UP006`
rewrites them. At run time, `list[int]` is a `types.GenericAlias` that remembers its
arguments, but `isinstance` cannot check element types:

In [84]:
@dataclass
class User:
    name: str


team_members: dict[str, list[User]] = {"backend": [User("Asha")], "frontend": []}
alias = dict[str, list[User]]
print(f"{type(alias).__name__=}  {typing.get_origin(alias)=}  {typing.get_args(alias)=}")
print(f"{raised_message(lambda: isinstance(team_members, dict[str, list[User]]))=}")
print(f"{typing.get_args(tuple[int, str])=}  {typing.get_args(tuple[int, ...])=}")

type(alias).__name__='GenericAlias'  typing.get_origin(alias)=<class 'dict'>  typing.get_args(alias)=(<class 'str'>, list[__main__.User])
raised_message(lambda: isinstance(team_members, dict[str, list[User]]))='TypeError: isinstance() argument 2 cannot be a parameterized generic'
typing.get_args(tuple[int, str])=(<class 'int'>, <class 'str'>)  typing.get_args(tuple[int, ...])=(<class 'int'>, Ellipsis)


The checker does track element types and tuple lengths:

In [85]:
check(
    """
    from dataclasses import dataclass

    @dataclass
    class User:
        name: str

    point: tuple[int, int] = (1, 2, 3)
    scores: tuple[int, ...] = (90, 85, 77)
    team_members: dict[str, list[User]] = {"backend": [User("Asha")]}
    team_members["frontend"] = User("Ravi")
    team_members["qa"].append("Meena")
    """,
    name="collections_demo",
    checkers=("mypy",),
)

mypy:
  collections_demo.py:7: error: Incompatible types in assignment (expression has type "tuple[int, int, int]", variable has type "tuple[int, int]")  [assignment]
  collections_demo.py:10: error: Incompatible types in assignment (expression has type "User", target has type "list[User]")  [assignment]
  collections_demo.py:11: error: Argument 1 to "append" of "list" has incompatible type "str"; expected "User"  [arg-type]


**Accept abstract types, return concrete ones.** A parameter typed `list[float]`
rejects tuples, sets and generators even when the function only loops over it.
Type parameters with the *least* you need from `collections.abc` and return a
concrete type so callers get every method:

| You only… | Accept | Accepts |
| --- | --- | --- |
| loop once | `Iterable[T]` | list, tuple, set, dict keys, generators, files |
| need `len`, `in` and repeated loops | `Collection[T]` | list, tuple, set, dict views |
| index or slice, read-only | `Sequence[T]` | list, tuple, str, range |
| read a mapping | `Mapping[K, V]` | dict, `MappingProxyType`, `ChainMap` |
| change it in place | `MutableSequence[T]`, `MutableMapping[K, V]` | list / dict |

In [86]:
check(
    """
    from collections.abc import Iterable

    def total_strict(prices: list[float]) -> float:
        return sum(prices)

    def total(prices: Iterable[float]) -> float:
        return sum(prices)

    def prices_with_gst(prices: Iterable[float]) -> list[float]:
        return [round(price * 1.18, 2) for price in prices]   # concrete result

    total_strict((10.0, 20.0))
    total((10.0, 20.0))
    total({10.0, 20.0})
    total(price for price in [10.0, 20.0])
    prices_with_gst((100.0,)).append(5.0)                     # list methods available
    """,
    name="abstract_params",
    checkers=("mypy",),
)

mypy:
  abstract_params.py:12: error: Argument 1 to "total_strict" has incompatible type "tuple[float, float]"; expected "list[float]"  [arg-type]


And at run time the `Iterable` version really does accept all of them:

In [87]:
def total(prices: Iterable[float]) -> float:
    """Sum any iterable of prices."""
    return sum(prices)


print(total([10.0, 20.0]), total((10.0, 20.0)), total({10.0, 20.0}), total(p for p in [10.0, 20.0]))
print(total({"tea": 120.5, "rice": 80.0}.values()))

30.0 30.0 30.0 30.0
200.5


### 4.2.4 Unions and narrowing

`X | Y` (PEP 604, 3.10) means "an `X` or a `Y`"; `X | None` replaces
`Optional[X]`. A union is only safe to use after **narrowing**: a check the checker
understands, after which it knows the more precise type in each branch. These are
the same ideas as TypeScript's control-flow analysis:

| Python check | TypeScript equivalent | Narrows `str \| None` / `int \| str` to |
| --- | --- | --- |
| `if name is None: return` / `if name is not None:` | `if (name === null)` | `str` after the check |
| `if isinstance(value, int):` | `typeof value === "number"` | `int` inside, `str` in `else` |
| `match value:` + `case int():` | `switch` on `typeof` | per `case` |
| `if name:` (truthiness) | `if (name)` | `str` (but `""` is excluded too) |
| early `return` / `raise` / `continue` | same | the rest of the block |
| `assert name is not None` | `name!` (non-null assertion) | `str` (checked at run time unless `-O`) |

First the bug: `find_customer` may return `None`, and the caller forgets. Both
checkers catch it.

In [88]:
UNNARROWED = """
def find_customer(customer_id: int) -> str | None:
    return {1: "Asha", 2: "Ravi"}.get(customer_id)

def greeting(customer_id: int) -> str:
    name = find_customer(customer_id)
    return "Hello, " + name.upper()
"""
check(UNNARROWED, name="unnarrowed")

mypy:
  unnarrowed.py:6: error: Item "None" of "str | None" has no attribute "upper"  [union-attr]


pyright:
  unnarrowed.py:6: error: "upper" is not a known attribute of "None"  [reportOptionalMemberAccess]


At run time the bug only appears for a missing id, which is why the checker is worth having:

In [89]:
def find_customer(customer_id: int) -> str | None:
    """Customer name, or None if unknown."""
    return {1: "Asha", 2: "Ravi"}.get(customer_id)


def greeting(customer_id: int) -> str:
    """Unsafe: forgets that find_customer can return None."""
    name = find_customer(customer_id)
    return "Hello, " + name.upper()


print(f"{greeting(1)=}")
print(f"{raised_message(lambda: greeting(99))=}")

greeting(1)='Hello, ASHA'
raised_message(lambda: greeting(99))="AttributeError: 'NoneType' object has no attribute 'upper'"


Three ways to narrow, each proven safe by both checkers. `reveal_type(x)` is a
checker instruction: it prints the type the checker inferred at that point
(mypy: `note: Revealed type is ...`, pyright: `information: Type of ...`).

In [90]:
check(
    """
    def find_customer(customer_id: int) -> str | None:
        return {1: "Asha", 2: "Ravi"}.get(customer_id)

    def greeting_is_none(customer_id: int) -> str:
        name = find_customer(customer_id)
        if name is None:
            return "Hello, guest"
        reveal_type(name)
        return "Hello, " + name.upper()

    def describe(value: int | str) -> str:
        if isinstance(value, int):
            reveal_type(value)
            return f"{value:,}"
        reveal_type(value)
        return value.strip()

    def describe_match(value: int | str | None) -> str:
        match value:
            case int():
                return f"{value:,}"
            case str():
                return value.strip()
            case None:
                return "-"
    """,
    name="narrowed",
)

mypy:
  narrowed.py:8: note: Revealed type is "str"
  narrowed.py:13: note: Revealed type is "int"
  narrowed.py:15: note: Revealed type is "str"


pyright:
  narrowed.py:8: information: Type of "name" is "str"
  narrowed.py:13: information: Type of "value" is "int"
  narrowed.py:15: information: Type of "value" is "str"


**`Never`, `NoReturn` and `assert_never`.** `Never` (alias `NoReturn`, the
conventional spelling for return types) is the empty type, TypeScript's `never`:

- A function that always raises returns `NoReturn`; code after a call to it is
  unreachable, so it narrows like a `return`.
- After every possibility of a union has been handled, the remaining type is
  `Never`. Passing the value to `assert_never()` (whose parameter is `Never`) makes
  the checker prove the `match` is **exhaustive**: add a new option and forget to
  handle it, and you get an error at that line.

In [91]:
EXHAUSTIVE = """
from typing import Literal, NoReturn, assert_never

type Speed = Literal["standard", "express", "same_day"]   # "same_day" was just added

def fail(message: str) -> NoReturn:
    raise ValueError(message)

def delivery_days(speed: Speed) -> int:
    match speed:
        case "standard":
            return 5
        case "express":
            return 2
        case _:
            assert_never(speed)          # same_day is not handled

def parse_days(text: str | None) -> int:
    if text is None:
        fail("missing value")
    return int(text)                    # text is str here: fail() never returns
"""
check(EXHAUSTIVE, name="exhaustive")

mypy:
  exhaustive.py:15: error: Argument 1 to "assert_never" has incompatible type "Literal['same_day']"; expected "Never"  [arg-type]


pyright:
  exhaustive.py:15: error: Argument of type "Literal['same_day']" cannot be assigned to parameter "arg" of type "Never" in function "assert_never" Type "Literal['same_day']" is not assignable to type "Never"  [reportArgumentType]


At run time `assert_never` raises `AssertionError`, a safety net for values the
checker could not see (data from JSON, for example):

In [92]:
from typing import Literal, Never, NoReturn, assert_never

type Speed = Literal["standard", "express"]


def delivery_days(speed: Speed) -> int:
    """Days to deliver; exhaustive over Speed."""
    match speed:
        case "standard":
            return 5
        case "express":
            return 2
        case _:
            assert_never(speed)


print(f"{delivery_days('express')=}")
print(f"{raised_message(lambda: delivery_days('drone'))=}")
print(f"{NoReturn!r} vs {Never!r}: different objects at run time, the same type to checkers")

delivery_days('express')=2
raised_message(lambda: delivery_days('drone'))="AssertionError: Expected code to be unreachable, but got: 'drone'"
typing.NoReturn vs typing.Never: different objects at run time, the same type to checkers


### 4.2.5 Escape hatches

| Hatch | Effect | TypeScript | Use it when |
| --- | --- | --- | --- |
| `Any` | Everything is allowed, in both directions; checking is off for that value | `any` | Truly dynamic data you will validate elsewhere; gradual migration |
| `object` | Anything can be passed in, but **nothing** can be done with it until you narrow | `unknown` | "I accept anything" parameters (logging, serializers) |
| `cast(T, value)` | Tells the checker "trust me, this is a `T`"; does nothing at run time | `value as T` | After a check the checker cannot follow |
| `# type: ignore[code]` | Silences one error code on one line | `// @ts-expect-error` | Checker bugs, untyped third-party code |

In [93]:
check(
    """
    import json
    from typing import Any, cast

    def log_any(value: Any) -> None:
        print(value.upper())        # Any: no error, may crash at run time

    def log_object(value: object) -> None:
        print(value.upper())        # object: error until narrowed

    def log_narrowed(value: object) -> None:
        if isinstance(value, str):
            print(value.upper())

    config = cast(dict[str, int], json.loads('{"retries": 3}'))
    reveal_type(json.loads("{}"))
    reveal_type(config)
    """,
    name="any_object",
    checkers=("mypy",),
)

mypy:
  any_object.py:8: error: "object" has no attribute "upper"  [attr-defined]
  any_object.py:15: note: Revealed type is "Any"
  any_object.py:16: note: Revealed type is "dict[str, int]"


`cast` is a lie the interpreter does not check: it returns its argument unchanged.
If the data does not match, the error appears later, far from the cause. Prefer
narrowing with `isinstance` or validating with a library such as Pydantic
(4.2.11).

In [94]:
from typing import cast

port = cast(int, "8080")  # the checker now believes port is an int
print(f"{port=}  {type(port).__name__=}")
print(f"{raised_message(lambda: port + 1)=}")

port='8080'  type(port).__name__='str'
raised_message(lambda: port + 1)='TypeError: can only concatenate str (not "int") to str'


**`# type: ignore[code]`.** Always give the error code: a bare `# type: ignore`
hides every error on the line, including ones you introduce later. With
`--warn-unused-ignores` mypy reports ignores that no longer suppress anything, and
`--enable-error-code ignore-without-code` rejects bare ones (both are in `--strict`
or recommended with it). pyright's own form is `# pyright: ignore[ruleName]`.

In [95]:
check(
    """
    def parse_port(text: str) -> int:
        return int(text)

    a: int = parse_port(8080)  # type: ignore[arg-type]
    b: int = parse_port(8080)  # type: ignore[assignment]
    c: int = parse_port("80")  # type: ignore[arg-type]
    d: int = parse_port(8080)  # type: ignore
    """,
    name="ignores",
    checkers=("mypy",),
)
STRICT_IGNORES = ("--warn-unused-ignores", "--enable-error-code", "ignore-without-code")
print("mypy --warn-unused-ignores --enable-error-code ignore-without-code:")
for finding in run_mypy(
    (UNIT42_DIR / "ignores.py").read_text(), name="ignores", options=STRICT_IGNORES
):
    print(f"  {finding}")

mypy:
  ignores.py:5: error: Argument 1 to "parse_port" has incompatible type "int"; expected "str"  [arg-type]
  ignores.py:5: note: Error code "arg-type" not covered by "type: ignore[assignment]" comment
mypy --warn-unused-ignores --enable-error-code ignore-without-code:


  ignores.py:5: error: Unused "type: ignore" comment  [unused-ignore]
  ignores.py:5: error: Argument 1 to "parse_port" has incompatible type "int"; expected "str"  [arg-type]
  ignores.py:5: note: Error code "arg-type" not covered by "type: ignore[assignment]" comment
  ignores.py:6: error: Unused "type: ignore" comment  [unused-ignore]
  ignores.py:7: error: "type: ignore" comment without error code (consider "type: ignore[arg-type]" instead)  [ignore-without-code]


### 4.2.6 Precise types

| Type | Means | TypeScript |
| --- | --- | --- |
| `Literal["GET", "POST"]` | Only these exact values | `"GET" \| "POST"` |
| `LiteralString` | A `str` built only from literals in the source (no user input) | none (a "template literal" brand) |
| `Final` | Never re-bound; with no type given, infers a literal type | `const` |
| `NewType("UserId", int)` | A distinct type for the checker, a plain `int` at run time | branded types (`number & {__brand: "UserId"}`) |
| `Self` | "The class of the current instance" (subclasses included) | `this` type |

**`Literal` and `LiteralString`.** `LiteralString` (PEP 675) protects APIs such as
SQL execution: only strings written in the source, or combined from such strings,
are accepted, so an f-string with user input is rejected. pyright implements it;
mypy currently treats `LiteralString` as plain `str`, a real difference between the
two checkers:

In [96]:
check(
    """
    from typing import Literal, LiteralString

    type Method = Literal["GET", "POST", "DELETE"]

    def request(method: Method, path: str) -> str:
        return f"{method} {path}"

    def run_query(sql: LiteralString) -> None:
        print(sql)

    request("GET", "/orders")
    request("PATCH", "/orders")

    def find_orders(customer: str) -> None:
        run_query("SELECT * FROM orders WHERE customer = %s")      # literal: OK
        run_query(f"SELECT * FROM orders WHERE customer = '{customer}'")
    """,
    name="literals",
)

mypy:
  literals.py:12: error: Argument 1 to "request" has incompatible type "Literal['PATCH']"; expected "Literal['GET', 'POST', 'DELETE']"  [arg-type]


pyright:
  literals.py:12: error: Argument of type "Literal['PATCH']" cannot be assigned to parameter "method" of type "Method" in function "request" Type "Literal['PATCH']" is not assignable to type "Method"  [reportArgumentType]
  literals.py:16: error: Argument of type "str" cannot be assigned to parameter "sql" of type "LiteralString" in function "run_query" "str" is not assignable to "LiteralString"  [reportArgumentType]


**`Final` infers literal types.** `TIMEOUT: Final = 30` has the type `Literal[30]`
where it is used, so it can feed a `Literal`-typed parameter:

In [97]:
check(
    """
    from typing import Final, Literal

    DEFAULT_PAGE_SIZE: Final = 20
    page_size = 20

    def fetch(size: Literal[10, 20, 50]) -> None: ...

    fetch(DEFAULT_PAGE_SIZE)     # Literal[20]: OK
    fetch(page_size)             # int: could be anything
    reveal_type(DEFAULT_PAGE_SIZE)
    """,
    name="final_literal",
    checkers=("mypy",),
)

mypy:
  final_literal.py:9: error: Argument 1 to "fetch" has incompatible type "int"; expected "Literal[10, 20, 50]"  [arg-type]
  final_literal.py:10: note: Revealed type is "Literal[20]?"


**`NewType` and `Self`.** `NewType` creates a distinct type for the checker (so a
`UserId` cannot be passed where an `OrderId` is expected) at zero run-time cost: the
"class" is an identity function. `Self` (3.11) types fluent APIs and alternative
constructors so subclasses get their own type back.

In [98]:
from typing import NewType, Self

UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)
user_id = UserId(42)
print(f"{user_id=}  {type(user_id).__name__=}  {user_id == 42=}  {UserId=}")


class QueryBuilder:
    """Fluent builder: every method returns the builder itself."""

    def __init__(self) -> None:
        self.conditions: list[str] = []

    def where(self, condition: str) -> Self:
        self.conditions.append(condition)
        return self


class OrderQuery(QueryBuilder):
    def unpaid(self) -> Self:
        return self.where("paid = false")


query = OrderQuery().where("city = 'Pune'").unpaid()  # where() returned OrderQuery
print(f"{type(query).__name__=}  {query.conditions=}")

user_id=42  type(user_id).__name__='int'  user_id == 42=True  UserId=__main__.UserId
type(query).__name__='OrderQuery'  query.conditions=["city = 'Pune'", 'paid = false']


In [99]:
check(
    """
    from typing import NewType, Self

    UserId = NewType("UserId", int)
    OrderId = NewType("OrderId", int)

    def cancel_order(order_id: OrderId) -> None: ...

    cancel_order(UserId(42))
    cancel_order(42)
    cancel_order(OrderId(42))

    class QueryBuilder:
        def where(self, condition: str) -> Self:
            return self

    class OrderQuery(QueryBuilder):
        def unpaid(self) -> Self:
            return self.where("paid = false")

    reveal_type(OrderQuery().where("city = 'Pune'"))
    """,
    name="newtype_self",
    checkers=("mypy",),
)

mypy:
  newtype_self.py:8: error: Argument 1 to "cancel_order" has incompatible type "UserId"; expected "OrderId"  [arg-type]
  newtype_self.py:9: error: Argument 1 to "cancel_order" has incompatible type "int"; expected "OrderId"  [arg-type]
  newtype_self.py:20: note: Revealed type is "newtype_self.OrderQuery"


**`TypedDict`** types dictionaries with a fixed set of string keys, such as JSON
payloads. It is TypeScript's `interface` for objects that stay plain dicts:

| Python | TypeScript |
| --- | --- |
| `class Address(TypedDict):` + `city: str` | `interface Address { city: string }` |
| `pin: NotRequired[str]` | `pin?: string` |
| `class Patch(TypedDict, total=False):` + `name: Required[str]` | all optional except `name` |
| `sku: ReadOnly[str]` (PEP 705, *New in 3.13*) | `readonly sku: string` |
| `def create(**fields: Unpack[ProductFields])` (PEP 692) | `function create(fields: ProductFields)` |

At run time a `TypedDict` *is* a `dict`: no validation, no class instance. The
class only records which keys are required, optional and read-only.

In [100]:
from typing import NotRequired, ReadOnly, Required, TypedDict, Unpack


class Address(TypedDict):
    city: str
    pin: NotRequired[str]


class ProductPatch(TypedDict, total=False):
    sku: Required[ReadOnly[str]]
    name: str
    price: Decimal


def update_product(**changes: Unpack[ProductPatch]) -> dict[str, object]:
    """Apply a partial update; keyword arguments are checked against ProductPatch."""
    return dict(changes)


home: Address = {"city": "Pune"}
print(f"{home=}  {type(home).__name__=}")
print(f"{Address.__required_keys__=}  {Address.__optional_keys__=}")
print(f"{ProductPatch.__required_keys__=}  {ProductPatch.__readonly_keys__=}")
print(f"{update_product(sku='TEA-500', price=Decimal('120.50'))=}")

home={'city': 'Pune'}  type(home).__name__='dict'
Address.__required_keys__=frozenset({'city'})  Address.__optional_keys__=frozenset({'pin'})
ProductPatch.__required_keys__=frozenset({'sku'})  ProductPatch.__readonly_keys__=frozenset({'sku'})
update_product(sku='TEA-500', price=Decimal('120.50'))={'sku': 'TEA-500', 'price': Decimal('120.50')}


The checker enforces the keys, their types, read-only keys and the `**kwargs` names:

In [101]:
check(
    """
    from decimal import Decimal
    from typing import NotRequired, ReadOnly, Required, TypedDict, Unpack

    class Address(TypedDict):
        city: str
        pin: NotRequired[str]

    class ProductPatch(TypedDict, total=False):
        sku: Required[ReadOnly[str]]
        name: str
        price: Decimal

    def update_product(**changes: Unpack[ProductPatch]) -> None: ...

    office: Address = {"pin": "411001"}                   # city missing
    shop: Address = {"city": "Pune", "zip": "411001"}     # unknown key
    patch: ProductPatch = {"sku": "TEA-500", "name": "Tea"}
    patch["sku"] = "TEA-1KG"                             # read-only
    update_product(sku="TEA-500", colour="green")        # unknown keyword
    update_product(price=99)                             # sku missing, int is not Decimal
    """,
    name="typed_dicts",
    checkers=("mypy",),
)

mypy:
  typed_dicts.py:15: error: Missing key "city" for TypedDict "Address"  [typeddict-item]
  typed_dicts.py:16: error: Extra key "zip" for TypedDict "Address"  [typeddict-unknown-key]
  typed_dicts.py:18: error: ReadOnly TypedDict key "sku" TypedDict is mutated  [typeddict-readonly-mutated]
  typed_dicts.py:19: error: Unexpected keyword argument "colour" for "update_product"  [call-arg]
  typed_dicts.py:20: error: Missing named argument "sku" for "update_product"  [call-arg]
  typed_dicts.py:20: error: Argument "price" to "update_product" has incompatible type "int"; expected "Decimal"  [arg-type]


### 4.2.7 Generics

**PEP 695 syntax** (3.12) declares type parameters in square brackets right after
the name, like TypeScript's angle brackets:

| Python 3.12+ | TypeScript |
| --- | --- |
| `def first[T](items: Sequence[T]) -> T:` | `function first<T>(items: T[]): T` |
| `class Stack[T]:` | `class Stack<T>` |
| `type Pair[T] = tuple[T, T]` | `type Pair<T> = [T, T]` |
| `[T: Hashable]` (bound) | `<T extends Hashable>` |
| `[T: (int, float)]` (constraints) | `<T extends number \| bigint>`, roughly |
| `[T = str]` (default, PEP 696, 3.13) | `<T = string>` |

Type parameters are real objects at run time, available as `__type_params__`:

In [102]:
def first[T](items: Sequence[T]) -> T:
    """The first item; the result has the element type of the sequence."""
    return items[0]


class Stack[T]:
    """A last-in, first-out stack of T."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)


type Pair[T] = tuple[T, T]

print(f"{first.__type_params__=}  {Stack.__type_params__=}")
print(f"{Stack[int]=}  {typing.get_args(Stack[int])=}")
print(f"{type(Pair).__name__=}  {Pair.__value__=}  {Pair[float]=}")

first.__type_params__=(T,)  Stack.__type_params__=(T,)
Stack[int]=__main__.Stack[int]  typing.get_args(Stack[int])=(<class 'int'>,)
type(Pair).__name__='TypeAliasType'  Pair.__value__=tuple[T, T]  Pair[float]=__main__.Pair[float]


`type` aliases are evaluated lazily (`__value__` is computed on first access), so
they can be **recursive**: the classic JSON type needs no quotes.

In [103]:
type Json = dict[str, Json] | list[Json] | str | int | float | bool | None


def count_leaves(value: Json) -> int:
    """Number of non-container values in a JSON document."""
    if isinstance(value, dict):
        return sum(count_leaves(item) for item in value.values())
    if isinstance(value, list):
        return sum(count_leaves(item) for item in value)
    return 1


print(f"{count_leaves({'order': {'id': 7, 'lines': [1, 2, {'sku': 'TEA'}]}, 'paid': True})=}")
print(f"{Json.__value__=}")

count_leaves({'order': {'id': 7, 'lines': [1, 2, {'sku': 'TEA'}]}, 'paid': True})=5
Json.__value__=dict[str, __main__.Json] | list[__main__.Json] | str | int | float | bool | None


**Bounds vs constraints, and defaults.** A *bound* `[T: Base]` accepts `Base` or any
subtype and keeps the exact type (`bool` stays `bool`). *Constraints*
`[T: (int, str)]` mean `T` is exactly one of the listed types (a `bool` is solved
as `int`). A *default* (PEP 696, *New in 3.13*) is used when the parameter is not
given, which is how `Generator[int]` can omit its other two parameters.

In [104]:
check(
    """
    from collections.abc import Hashable, Sequence

    def bounded[T: int](value: T) -> T:
        return value

    def constrained[T: (int, str)](value: T) -> T:
        return value

    def dedupe[T: Hashable](items: Sequence[T]) -> list[T]:
        return list(dict.fromkeys(items))

    class Page[T = dict[str, object]]:
        def __init__(self, items: list[T]) -> None:
            self.items = items

    def plain_page() -> Page:
        return Page([])

    reveal_type(bounded(True))
    reveal_type(constrained(True))
    reveal_type(dedupe(["a", "b", "a"]))
    dedupe([[1], [2]])
    reveal_type(plain_page().items)
    """,
    name="bounds",
    checkers=("mypy",),
)

mypy:
  bounds.py:19: note: Revealed type is "bool"
  bounds.py:20: note: Revealed type is "int"
  bounds.py:21: note: Revealed type is "list[str]"
  bounds.py:22: error: Value of type variable "T" of "dedupe" cannot be "list[int]"  [type-var]
  bounds.py:23: note: Revealed type is "list[dict[str, object]]"


In [105]:
class Page[T = dict[str, object]]:
    """A page of results; items default to JSON-like dicts."""


(page_param,) = Page.__type_params__
print(f"{page_param.has_default()=}  {page_param.__default__=}")


def dedupe[T: typing.Hashable](items: Sequence[T]) -> list[T]:
    """Drop repeated items, keeping the first of each."""
    return list(dict.fromkeys(items))


(dedupe_param,) = dedupe.__type_params__
print(f"{dedupe_param.__bound__=}  {dedupe([3, 1, 3, 2, 1])=}")

page_param.has_default()=True  page_param.__default__=dict[str, object]
dedupe_param.__bound__=typing.Hashable  dedupe([3, 1, 3, 2, 1])=[3, 1, 2]


**Variance** answers: if `bool` is a subtype of `int`, is `Box[bool]` a subtype of
`Box[int]`?

| Variance | Rule | Examples | Why |
| --- | --- | --- | --- |
| Covariant | `Box[bool]` **is** a `Box[int]` | `Sequence`, `tuple`, `frozenset`, `Mapping` values, read-only classes | You can only *read* `int`s out of it |
| Invariant | No relation | `list`, `dict`, `set`, mutable classes | Someone could *write* an `int` into your list of `bool`s |
| Contravariant | `Handler[int]` **is** a `Handler[bool]` | `Callable` parameters, write-only sinks | Something that handles any `int` can handle a `bool` |

TypeScript checks method parameters *bivariantly* (unsound on purpose); Python
checkers are strict here. With PEP 695 syntax the checker **infers** variance from
how the class uses `T`.

In [106]:
check(
    """
    from collections.abc import Sequence
    from dataclasses import dataclass

    def average(values: list[float]) -> float:
        return sum(values) / len(values)

    def average_ro(values: Sequence[float]) -> float:
        return sum(values) / len(values)

    flags: list[bool] = [True, False]
    ints: list[int] = [1, 2, 3]
    average(ints)          # list is invariant
    average_ro(ints)       # Sequence is covariant

    @dataclass(frozen=True)
    class ReadOnlyBox[T]:  # only reads T: inferred covariant
        item: T

    class Box[T]:          # reads and writes T: inferred invariant
        def __init__(self, item: T) -> None:
            self.item = item

    def open_int_box(box: ReadOnlyBox[int]) -> int:
        return box.item

    def open_mutable(box: Box[int]) -> int:
        return box.item

    open_int_box(ReadOnlyBox(True))
    bool_box: Box[bool] = Box(True)
    open_mutable(bool_box)
    """,
    name="variance",
    checkers=("mypy",),
)

mypy:
  variance.py:12: error: Argument 1 to "average" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
  variance.py:12: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
  variance.py:12: note: Consider using "Sequence" instead, which is covariant
  variance.py:31: error: Argument 1 to "open_mutable" has incompatible type "Box[bool]"; expected "Box[int]"  [arg-type]


**The old `TypeVar` syntax** is what you will read in most existing code (and still
write when supporting Python ≤ 3.11). Same meaning, more ceremony: the type variable
is a module-level object, classes inherit from `Generic[T]`, and variance must be
declared by hand. ruff's `UP046`/`UP047` rewrite it to PEP 695 syntax for
`target-version = "py312"` and later, hence the `noqa` comments.

In [107]:
from typing import Generic, TypeVar

T = TypeVar("T")
T_co = TypeVar("T_co", covariant=True)
Number = TypeVar("Number", int, float)  # constraints
AnyStr_ = TypeVar("AnyStr_", bound=str | bytes)  # bound


class OldBox(Generic[T]):  # noqa: UP046
    def __init__(self, item: T) -> None:
        self.item = item


def old_first(items: Sequence[T]) -> T:  # noqa: UP047
    """Same as first[T], old spelling."""
    return items[0]


print(f"{OldBox.__parameters__=}  {OldBox[int]=}")
print(f"{T_co.__covariant__=}  {Number.__constraints__=}  {AnyStr_.__bound__=}")
new_param = Stack.__type_params__[0]
print(f"PEP 695 parameter: {new_param!r}, infer variance: {new_param.__infer_variance__}")

OldBox.__parameters__=(~T,)  OldBox[int]=__main__.OldBox[int]
T_co.__covariant__=True  Number.__constraints__=(<class 'int'>, <class 'float'>)  AnyStr_.__bound__=str | bytes
PEP 695 parameter: T, infer variance: True


### 4.2.8 Callable types

| Python | TypeScript |
| --- | --- |
| `Callable[[int, str], bool]` | `(a: number, b: string) => boolean` |
| `Callable[..., Decimal]` | `(...args: any[]) => Decimal` |
| `Callable[[], None]` | `() => void` |
| callback `Protocol` with `__call__` | call signature in an interface: `{ (amount: Decimal): Decimal }` |

`Callable` cannot describe keyword names, defaults or overloads. For those, write a
**callback Protocol**: a Protocol whose `__call__` has the exact signature.

In [108]:
check(
    """
    from collections.abc import Callable
    from decimal import Decimal
    from typing import Protocol

    class PriceRule(Protocol):
        def __call__(self, amount: Decimal, *, tier: str = "basic") -> Decimal: ...

    def ten_percent_off(amount: Decimal, *, tier: str = "basic") -> Decimal:
        return amount * Decimal("0.9")

    def wrong_keyword(amount: Decimal, *, level: str = "basic") -> Decimal:
        return amount

    def simple(amount: Decimal) -> Decimal:
        return amount

    def apply(rule: PriceRule, amount: Decimal) -> Decimal:
        return rule(amount, tier="gold")

    def apply_simple(rule: Callable[[Decimal], Decimal], amount: Decimal) -> Decimal:
        return rule(amount)

    apply(ten_percent_off, Decimal("100"))
    apply(wrong_keyword, Decimal("100"))
    apply(simple, Decimal("100"))
    apply_simple(simple, Decimal("100"))
    apply_simple(len, Decimal("100"))
    """,
    name="callables",
    checkers=("mypy",),
)

mypy:
  callables.py:24: error: Argument 1 to "apply" has incompatible type "def wrong_keyword(amount: Decimal, *, level: str = ...) -> Decimal"; expected "PriceRule"  [arg-type]
  callables.py:24: note: "PriceRule.__call__" has type "def __call__(self, amount: Decimal, *, tier: str = ...) -> Decimal"
  callables.py:25: error: Argument 1 to "apply" has incompatible type "Callable[[Decimal], Decimal]"; expected "PriceRule"  [arg-type]
  callables.py:25: note: "PriceRule.__call__" has type "def __call__(self, amount: Decimal, *, tier: str = ...) -> Decimal"
  callables.py:27: error: Argument 1 to "apply_simple" has incompatible type "Callable[[Sized], int]"; expected "Callable[[Decimal], Decimal]"  [arg-type]


**`ParamSpec`: decorators that keep the signature.** A decorator typed with
`Callable[..., Any]` erases what it wraps: callers lose argument checking and the
return type becomes `Any`. `[**P, R]` captures the wrapped function's parameters as
`P` and its return type as `R`, so the wrapper keeps both (Unit 1.12 used this
spelling already).

In [109]:
check(
    """
    import functools
    from collections.abc import Callable
    from typing import Any

    def logged_loose(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            return func(*args, **kwargs)
        return wrapper

    def logged[**P, R](func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            print(f"calling {func.__name__}")
            return func(*args, **kwargs)
        return wrapper

    @logged_loose
    def price_loose(sku: str, quantity: int = 1) -> float:
        return 10.0 * quantity

    @logged
    def price(sku: str, quantity: int = 1) -> float:
        return 10.0 * quantity

    price_loose(42)            # not caught
    price(42)                  # caught
    reveal_type(price)
    reveal_type(price_loose("TEA"))
    """,
    name="paramspec",
    checkers=("mypy",),
)

mypy:
  paramspec.py:26: error: Argument 1 to "price" has incompatible type "int"; expected "str"  [arg-type]
  paramspec.py:27: note: Revealed type is "def (sku: str, quantity: int =) -> float"
  paramspec.py:28: note: Revealed type is "Any"


**`Concatenate`** describes a decorator that **adds or removes** leading
parameters. A classic case is dependency injection: the decorated function takes a
connection as its first parameter, and the wrapper supplies it, so callers do not.
Decorator *factories* (decorators with arguments) just add one more layer:
`def retry(times: int) -> Callable[[Callable[P, R]], Callable[P, R]]`, which is
Lab 4.2 exercise 3.

In [110]:
from typing import Concatenate


class Connection:
    def execute(self, sql: str) -> list[tuple[object, ...]]:
        return [(sql.split()[-1], 1)]


def with_connection[**P, R](func: Callable[Concatenate[Connection, P], R]) -> Callable[P, R]:
    """Supply a fresh Connection as the first argument."""

    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(Connection(), *args, **kwargs)

    return wrapper


@with_connection
def count_orders(connection: Connection, city: str) -> int:
    return len(connection.execute(f"SELECT 1 FROM orders WHERE city = {city}"))


print(f"{count_orders('Pune')=}  {inspect.signature(count_orders)=}")

count_orders('Pune')=1  inspect.signature(count_orders)=<Signature (connection: __main__.Connection, city: str) -> int>


Note that `inspect.signature` (which follows `__wrapped__`) still shows the original
parameters; only the checker knows the connection parameter was removed. mypy
agrees with the run-time behaviour:

In [111]:
check(
    """
    import functools
    from collections.abc import Callable
    from typing import Concatenate

    class Connection: ...

    def with_connection[**P, R](func: Callable[Concatenate[Connection, P], R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            return func(Connection(), *args, **kwargs)
        return wrapper

    @with_connection
    def count_orders(connection: Connection, city: str) -> int:
        return 1

    reveal_type(count_orders)
    count_orders("Pune")
    count_orders(Connection(), "Pune")
    """,
    name="concatenate",
    checkers=("mypy",),
)

mypy:
  concatenate.py:17: note: Revealed type is "def (city: str) -> int"
  concatenate.py:19: error: Too many arguments for "count_orders"  [call-arg]
  concatenate.py:19: error: Argument 1 to "count_orders" has incompatible type "Connection"; expected "str"  [arg-type]


**`@overload`** declares several signatures for one function when the return type
depends on the argument types (TypeScript function overloads). The `@overload`
stubs are for the checker only; the last, undecorated definition is the real
implementation. `typing.get_overloads()` (3.11) lists them at run time.

In [112]:
from typing import get_overloads, overload


@overload
def parse_amount(raw: str) -> Decimal: ...
@overload
def parse_amount(raw: None) -> None: ...
def parse_amount(raw: str | None) -> Decimal | None:
    """Parse "1,234.50" into Decimal; None stays None."""
    return None if raw is None else Decimal(raw.replace(",", ""))


print(f"{parse_amount('1,234.50')=}  {parse_amount(None)=}")
print(f"{len(get_overloads(parse_amount))=}")
check(
    """
    from decimal import Decimal
    from typing import overload

    @overload
    def parse_amount(raw: str) -> Decimal: ...
    @overload
    def parse_amount(raw: None) -> None: ...
    def parse_amount(raw: str | None) -> Decimal | None:
        return None if raw is None else Decimal(raw.replace(",", ""))

    reveal_type(parse_amount("1,234.50"))
    reveal_type(parse_amount(None))
    parse_amount(12)
    """,
    name="overloads",
    checkers=("mypy",),
)

parse_amount('1,234.50')=Decimal('1234.50')  parse_amount(None)=None
len(get_overloads(parse_amount))=2


mypy:
  overloads.py:11: note: Revealed type is "decimal.Decimal"
  overloads.py:12: note: Revealed type is "None"
  overloads.py:13: error: No overload variant of "parse_amount" matches argument type "int"  [call-overload]
  overloads.py:13: note: Possible overload variants:
  overloads.py:13: note:     def parse_amount(raw: str) -> Decimal
  overloads.py:13: note:     def parse_amount(raw: None) -> None


### 4.2.9 Class typing

| Tool | Means | TypeScript |
| --- | --- | --- |
| `type[Product]` | The class `Product` itself (or a subclass), not an instance | `typeof Product` / `new () => Product` |
| `@override` (PEP 698, 3.12) | "This method overrides a base method"; error if it does not | `override` keyword |
| `@final` on a class | Must not be subclassed | none (`sealed` in C#) |
| `@final` on a method | Must not be overridden | none |
| `class Repository[T](Protocol)` | A generic structural interface | `interface Repository<T>` |

At run time `@override` and `@final` only set a marker attribute (`__override__`,
`__final__`); the checker enforces them.

In [113]:
from typing import final, override


class Model:
    def describe(self) -> str:
        return f"{type(self).__name__} stored in {type(self).__name__.lower()}s"


class Product(Model):
    @override
    def describe(self) -> str:
        return "Product stored in the catalogue"


def create[M: Model](model_class: type[M]) -> M:
    """Instantiate any Model subclass; the result has that subclass's type."""
    return model_class()


@final
class Money: ...


print(f"{create(Product).describe()=}  {create(Model).describe()=}")
print(f"{Product.describe.__override__=}  {Money.__final__=}")

create(Product).describe()='Product stored in the catalogue'  create(Model).describe()='Model stored in models'
Product.describe.__override__=True  Money.__final__=True


A generic Protocol plus the class-typing errors the checker reports:

In [114]:
check(
    """
    from typing import Protocol, final, override

    class Repository[T](Protocol):
        def get(self, key: str) -> T | None: ...
        def add(self, key: str, item: T) -> None: ...

    class InMemoryRepository[T]:
        def __init__(self) -> None:
            self._items: dict[str, T] = {}
        def get(self, key: str) -> T | None:
            return self._items.get(key)
        def add(self, key: str, item: T) -> None:
            self._items[key] = item

    def restock(repository: Repository[int], sku: str) -> None:
        repository.add(sku, (repository.get(sku) or 0) + 10)

    restock(InMemoryRepository[int](), "TEA-500")
    restock(InMemoryRepository[str](), "TEA-500")

    class Shipment:
        def cost(self) -> int:
            return 50
        @final
        def tracking_id(self) -> str:
            return "TRK-1"

    class Express(Shipment):
        @override
        def costs(self) -> int:       # typo: overrides nothing
            return 120
        def tracking_id(self) -> str:  # overrides a final method
            return "EXP-1"

    @final
    class Money: ...

    class Rupees(Money): ...

    def build(kind: type[Shipment]) -> Shipment:
        return kind()

    build(Express)
    build(Money)
    """,
    name="class_typing",
    checkers=("mypy",),
)

mypy:
  class_typing.py:19: error: Argument 1 to "restock" has incompatible type "InMemoryRepository[str]"; expected "Repository[int]"  [arg-type]
  class_typing.py:30: error: Method "costs" is marked as an override, but no base method was found with this name  [misc]
  class_typing.py:32: error: Cannot override final attribute "tracking_id" (previously declared in base class "Shipment")  [misc]
  class_typing.py:38: error: Cannot inherit from final class "Money"  [misc]
  class_typing.py:44: error: Argument 1 to "build" has incompatible type "type[Money]"; expected "type[Shipment]"  [arg-type]


### 4.2.10 Custom narrowing

When a check is too complex for the checker to follow, write a function that
returns `bool` and tell the checker what `True` means:

| | `TypeGuard[T]` (PEP 647, 3.10) | `TypeIs[T]` (PEP 742, *New in 3.13*) |
| --- | --- | --- |
| Narrows when `True` | to `T` | to `T` (intersected with the input type) |
| Narrows when `False` | no | **yes**: to "input type minus `T`" |
| `T` must be a subtype of the input type | no (can narrow `list[object]` to `list[str]`) | yes |
| TypeScript | | `function isProduct(x: unknown): x is Product` |

Prefer `TypeIs`; use `TypeGuard` only for narrowing to a non-subtype, typically a
container whose element type you checked.

In [115]:
from typing import TypeGuard, TypeIs


def is_str_list(values: list[object]) -> TypeGuard[list[str]]:
    """True if every item is a str."""
    return all(isinstance(value, str) for value in values)


def is_decimal(value: object) -> TypeIs[Decimal]:
    """True if value is a Decimal."""
    return isinstance(value, Decimal)


print(f"{is_str_list(['a', 'b'])=}  {is_str_list(['a', 1])=}  {is_decimal(Decimal('1'))=}")
check(
    """
    from decimal import Decimal
    from typing import TypeGuard, TypeIs

    def is_str_list(values: list[object]) -> TypeGuard[list[str]]:
        return all(isinstance(value, str) for value in values)

    def is_decimal(value: object) -> TypeIs[Decimal]:
        return isinstance(value, Decimal)

    def handle(values: list[object], amount: Decimal | float) -> None:
        if is_str_list(values):
            reveal_type(values)
        if is_decimal(amount):
            reveal_type(amount)
        else:
            reveal_type(amount)     # TypeIs also narrows the else branch
    """,
    name="type_is",
    checkers=("mypy",),
)

is_str_list(['a', 'b'])=True  is_str_list(['a', 1])=False  is_decimal(Decimal('1'))=True


mypy:
  type_is.py:12: note: Revealed type is "list[str]"
  type_is.py:14: note: Revealed type is "decimal.Decimal"
  type_is.py:16: note: Revealed type is "float"


**Discriminated (tagged) unions.** Give every member of a union a field with a
different `Literal` value; checking that field narrows to one member. This is
exactly TypeScript's tagged union (`type Event = {kind: "paid", …} | {kind: …}`),
and it combines with `match` and `assert_never` for exhaustiveness. It works with
dataclasses, `TypedDict`s and Pydantic models.

In [116]:
@dataclass(frozen=True)
class Paid:
    order_id: int
    amount: Decimal
    kind: Literal["paid"] = "paid"


@dataclass(frozen=True)
class Refunded:
    order_id: int
    reason: str
    kind: Literal["refunded"] = "refunded"


type PaymentEvent = Paid | Refunded


def describe_event(event: PaymentEvent) -> str:
    """One line per event; exhaustive over the tag."""
    match event.kind:
        case "paid":
            return f"order {event.order_id}: paid INR {event.amount}"
        case "refunded":
            return f"order {event.order_id}: refunded ({event.reason})"


for event in (Paid(7, Decimal("499")), Refunded(8, "damaged")):
    print(describe_event(event))

order 7: paid INR 499
order 8: refunded (damaged)


In [117]:
check(
    """
    from dataclasses import dataclass
    from typing import Literal, TypedDict, assert_never

    class CardPayment(TypedDict):
        method: Literal["card"]
        last4: str

    class UpiPayment(TypedDict):
        method: Literal["upi"]
        vpa: str

    class CashPayment(TypedDict):
        method: Literal["cash"]

    type Payment = CardPayment | UpiPayment | CashPayment

    def receipt_line(payment: Payment) -> str:
        if payment["method"] == "card":
            reveal_type(payment)
            return f"card ending {payment['last4']}"
        if payment["method"] == "upi":
            return f"UPI {payment['vpa']}"
        reveal_type(payment)
        return payment["vpa"]            # a cash payment has no vpa
    """,
    name="tagged_unions",
    checkers=("mypy",),
)

mypy:
  tagged_unions.py:19: note: Revealed type is "TypedDict(tagged_unions.CardPayment, {'method': Literal['card'], 'last4': str})"
  tagged_unions.py:23: note: Revealed type is "TypedDict(tagged_unions.CashPayment, {'method': Literal['cash']})"
  tagged_unions.py:24: error: TypedDict "CashPayment" has no key "vpa"  [typeddict-item]


### 4.2.11 `Annotated` and runtime introspection

`Annotated[T, metadata, ...]` (PEP 593) attaches arbitrary objects to a type. The
**checker sees only `T`**; libraries read the metadata at run time. This is how
Pydantic expresses constraints (`Annotated[int, Field(gt=0)]`) and how FastAPI
declares where a parameter comes from (`Annotated[str, Query(max_length=50)]`,
`Annotated[Session, Depends(get_session)]`, Phase 6). TypeScript has no run-time
types at all, which is why JS libraries such as Zod define schemas as values
instead.

| Function | Returns |
| --- | --- |
| `typing.get_type_hints(obj)` | Evaluated hints, `Annotated` **stripped** |
| `typing.get_type_hints(obj, include_extras=True)` | Evaluated hints with `Annotated` kept |
| `typing.get_origin(tp)` | The unsubscripted form: `list` for `list[int]`, `Annotated` for `Annotated[...]`, `types.UnionType` for `X \| Y` |
| `typing.get_args(tp)` | The arguments: `(int,)`, `(int, Range(0, 150))`, … |

In [118]:
from typing import Annotated, get_args, get_origin, get_type_hints


@dataclass(frozen=True)
class Range:
    minimum: int
    maximum: int


@dataclass
class Signup:
    name: Annotated[str, "display name"]
    age: Annotated[int, Range(18, 120)]
    referral: str | None = None


print(f"{get_type_hints(Signup)=}")
hints = get_type_hints(Signup, include_extras=True)
print(f"{hints['age']=}")
print(f"{get_origin(hints['age']) is Annotated=}  {get_args(hints['age'])=}")
print(f"{hints['age'].__metadata__=}")
print(f"{get_origin(hints['referral'])=}  {get_args(hints['referral'])=}")

get_type_hints(Signup)={'name': <class 'str'>, 'age': <class 'int'>, 'referral': str | None}
hints['age']=typing.Annotated[int, Range(minimum=18, maximum=120)]
get_origin(hints['age']) is Annotated=True  get_args(hints['age'])=(<class 'int'>, Range(minimum=18, maximum=120))
hints['age'].__metadata__=(Range(minimum=18, maximum=120),)
get_origin(hints['referral'])=<class 'typing.Union'>  get_args(hints['referral'])=(<class 'str'>, <class 'NoneType'>)


**Under the hood: a mini validator.** Reading the metadata is all a framework does
before deciding what to check. Lab 4.2 exercise 7 builds a complete one; this cell
only walks a function's parameters:

In [119]:
def constraints_of(function: Callable[..., object]) -> dict[str, list[object]]:
    """Parameter name -> metadata objects found in Annotated hints."""
    found: dict[str, list[object]] = {}
    for name, hint in get_type_hints(function, include_extras=True).items():
        if get_origin(hint) is Annotated:
            found[name] = list(hint.__metadata__)
    return found


def register(name: Annotated[str, "1-40 chars"], age: Annotated[int, Range(18, 120)]) -> None:
    """Register a customer (the metadata is for a framework to enforce)."""


print(constraints_of(register))
print(f"{inspect.signature(register).parameters['age'].annotation=}")

{'name': ['1-40 chars'], 'age': [Range(minimum=18, maximum=120)]}
inspect.signature(register).parameters['age'].annotation=typing.Annotated[int, Range(minimum=18, maximum=120)]


**Pydantic** turns hints plus `Annotated` metadata into a validator and parser
(Unit 4.6 and Phase 6 go deeper). Constraints from `Field(...)` and from the
`annotated-types` library (`Gt`, `Len`, …) both work:

In [120]:
if importlib.util.find_spec("pydantic") is None:
    print("SKIP: pydantic is not installed (uv add pydantic)")
else:
    from annotated_types import Len
    from pydantic import BaseModel, Field, ValidationError

    class SignupModel(BaseModel):
        name: Annotated[str, Len(1, 40)]
        age: Annotated[int, Field(ge=18, le=120)]
        referral: str | None = None

    print(SignupModel(name="Asha", age="31"))  # "31" is parsed to 31
    try:
        SignupModel(name="", age=12)
    except ValidationError as error:
        for problem in error.errors():
            print(f"  {problem['loc'][0]}: {problem['msg']}")

name='Asha' age=31 referral=None
  name: String should have at least 1 character
  age: Input should be greater than or equal to 18


FastAPI (Phase 6) reads the same structure to wire HTTP requests to functions:

```python
from typing import Annotated
from fastapi import Depends, FastAPI, Query

app = FastAPI()

@app.get("/orders")
def list_orders(
    city: Annotated[str, Query(max_length=40)],          # from ?city=...
    session: Annotated[Session, Depends(get_session)],   # injected
) -> list[Order]: ...
```

**Predict the output.**

```python
from typing import Annotated, get_type_hints

def ship(weight: Annotated[float, "kg"]) -> None: ...

print(get_type_hints(ship))
print(ship.__annotations__["weight"] == Annotated[float, "kg"])
```

<details><summary>Answer</summary>

`{'weight': <class 'float'>, 'return': <class 'NoneType'>}` then `True`.
`get_type_hints` strips `Annotated` unless you pass `include_extras=True`, and it
turns the `None` return annotation into `NoneType`. `__annotations__` holds the raw
objects, and `Annotated` aliases compare equal when their type and metadata match.

</details>

Run the next cell to check your prediction.

In [121]:
from typing import Annotated, get_type_hints


def ship(weight: Annotated[float, "kg"]) -> None: ...


print(get_type_hints(ship))
print(ship.__annotations__["weight"] == Annotated[float, "kg"])

{'weight': <class 'float'>, 'return': <class 'NoneType'>}
True


### 4.2.12 Running checkers

| | mypy | pyright |
| --- | --- | --- |
| Written in | Python (compiled with mypyc) | TypeScript (runs on Node.js; powers VS Code's Pylance) |
| Install | `uv add --dev mypy` | `uv add --dev pyright` (downloads Node if needed) |
| Default mode | Skips unannotated functions | Checks everything, infers inside untyped code |
| Strict mode | `--strict` or `strict = true` | `typeCheckingMode = "strict"` or `# pyright: strict` |
| Config | `[tool.mypy]` (+ `[[tool.mypy.overrides]]`) | `[tool.pyright]` or `pyrightconfig.json` |
| Plugins | Yes (Django, Pydantic, SQLAlchemy) | No (relies on standard typing features) |
| TypeScript equivalent | `tsc --noEmit` with `"strict": true` in `tsconfig.json` | the same, and pyright is what VS Code runs as you type (like the TS language server) |

`--strict` is a bundle of flags; mypy lists them in its help:

In [122]:
help_text = subprocess.run(
    [str(VENV_BIN / "mypy"), "--help"], capture_output=True, text=True, env=CHECKER_ENV
).stdout
help_text = ANSI_ESCAPE.sub("", help_text)  # Python 3.14+ argparse can colour its help
strict_help = re.search(r"--strict\s+(.*?)(?=\n  --)", help_text, flags=re.S)
print(textwrap.fill(" ".join(strict_help[1].split()), width=96) if strict_help else "not found")

Strict mode; enables the following flags: --disallow-any-generics, --disallow-subclassing- any,
--disallow-untyped-calls, --disallow-untyped- defs, --disallow-incomplete-defs, --check-untyped-
defs, --disallow-untyped-decorators, --warn- redundant-casts, --warn-unused-ignores, --warn-
return-any, --no-implicit-reexport, --strict- equality, --extra-checks


**Configuration in `pyproject.toml`.** A realistic setup for a project that is
partly typed: strict everywhere, except a legacy package that is excluded for now.
Both checkers run from the project folder and read the same file.

In [123]:
typed_shop = UNIT42_DIR / "typed-shop"
files = {
    "pyproject.toml": """
        [tool.mypy]
        strict = true
        files = ["src"]
        mypy_path = "src"
        warn_unreachable = true

        [[tool.mypy.overrides]]
        module = ["legacy", "legacy.*"]
        ignore_errors = true                  # gradual adoption: not checked yet

        [tool.pyright]
        include = ["src"]
        exclude = ["src/legacy"]
        typeCheckingMode = "strict"
        pythonVersion = "3.15"
    """,
    "src/shop/__init__.py": "",
    "src/shop/pricing.py": """
        from decimal import Decimal

        def with_gst(amount: Decimal, rate: Decimal = Decimal("0.18")) -> Decimal:
            return amount * (1 + rate)

        def total(amounts: list[Decimal]) -> Decimal:
            return sum(amounts, Decimal(0))

        def bad_total(amounts):                  # untyped: strict mode rejects it
            return sum(amounts)
    """,
    "src/legacy/__init__.py": "",
    "src/legacy/report.py": "def make_report(rows):\n    return '\\n'.join(rows) + 1\n",
}
for relative_path, text in files.items():
    (typed_shop / relative_path).parent.mkdir(parents=True, exist_ok=True)
    (typed_shop / relative_path).write_text(textwrap.dedent(text).lstrip("\n"))
for command in (["mypy", "--cache-dir", "../.mypy_cache"], ["pyright"]):
    result = subprocess.run(
        [str(VENV_BIN / command[0]), *command[1:]],
        cwd=typed_shop,
        capture_output=True,
        text=True,
        env=CHECKER_ENV,
    )
    print(
        f"$ {command[0]}\n"
        + textwrap.indent(
            ANSI_ESCAPE.sub("", result.stdout).replace(str(typed_shop), "<project>").strip(), "  "
        )
    )

$ mypy
  src/shop/pricing.py:9: error: Function is missing a type annotation  [no-untyped-def]
  Found 1 error in 1 file (checked 4 source files)


$ pyright
  <project>/src/shop/pricing.py
    <project>/src/shop/pricing.py:9:15 - error: Type of parameter "amounts" is unknown (reportUnknownParameterType)
    <project>/src/shop/pricing.py:9:15 - error: Type annotation is missing for parameter "amounts" (reportMissingParameterType)
    <project>/src/shop/pricing.py:10:16 - error: Argument type is unknown
      Argument corresponds to parameter "iterable" in function "sum" (reportUnknownArgumentType)
  3 errors, 0 warnings, 0 informations


**`reveal_type` at run time.** Besides being a checker instruction,
`typing.reveal_type` (3.11) is a real function: it prints the *run-time* type to
stderr and returns its argument unchanged, so you can leave it in while debugging
and run the code. (Remove it afterwards; ruff and the checkers flag it.)

In [124]:
import contextlib
import io

captured = io.StringIO()
with contextlib.redirect_stderr(captured):
    returned = typing.reveal_type([1, 2, 3])
print(f"{returned=}  stderr: {captured.getvalue().strip()!r}")

returned=[1, 2, 3]  stderr: "Runtime type is 'list'"


**Stubs and `py.typed`.** A checker needs types for every library you import:

| Source of types | Example |
| --- | --- |
| Inline hints in a package that ships a **`py.typed`** marker file (PEP 561) | pydantic, httpx, rich |
| Stub files (`.pyi`: signatures only, bodies are `...`) bundled in **typeshed** | the standard library; mypy and pyright ship it |
| A separate stub package `types-<name>` or `<name>-stubs` | `types-requests`, `pandas-stubs` |
| Your own stubs in a folder on `mypy_path` / pyright's `stubPath` | private untyped code |

Without `py.typed`, an installed package's inline hints are **ignored**, as the next
demo shows. It installs a tiny untyped package into a scratch virtual environment
and points mypy at that environment's interpreter.

In [125]:
stub_demo = UNIT42_DIR / "stub-demo"
env_dir = stub_demo / "venv"
if not env_dir.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(env_dir)], check=True)
site_packages = next(env_dir.glob("lib/python3*/site-packages"))
package_dir = site_packages / "rupee_tools"
package_dir.mkdir(exist_ok=True)
(package_dir / "__init__.py").write_text(
    "def to_paise(amount: float) -> int:\n    return round(amount * 100)\n"
)
(package_dir / "py.typed").unlink(missing_ok=True)
APP = """
from rupee_tools import to_paise

reveal_type(to_paise(12.5))
to_paise("12.50")
"""
ENV_OPTION = ("--python-executable", str(env_dir / "bin" / "python"))
print("without py.typed:", *run_mypy(APP, name="stub_app", options=ENV_OPTION), sep="\n  ")
(package_dir / "py.typed").write_text("")
print("with py.typed:", *run_mypy(APP, name="stub_app", options=ENV_OPTION), sep="\n  ")

without py.typed:
  stub_app.py:1: error: Skipping analyzing "rupee_tools": module is installed, but missing library stubs or py.typed marker  [import-untyped]
  stub_app.py:1: note: See https://mypy.readthedocs.io/en/stable/running_mypy.html#missing-imports
  stub_app.py:3: note: Revealed type is "Any"
with py.typed:
  stub_app.py:3: note: Revealed type is "int"
  stub_app.py:4: error: Argument 1 to "to_paise" has incompatible type "str"; expected "float"  [arg-type]


When you cannot change the package, write a **stub** instead: a `.pyi` file with the
same module path, found through `mypy_path`. mypy's `stubgen` drafts one from the
source (here with `py.typed` removed again, so only the stub provides types):

In [126]:
(package_dir / "py.typed").unlink()
stubs_dir = stub_demo / "stubs"
subprocess.run(
    [str(VENV_BIN / "stubgen"), str(package_dir / "__init__.py"), "-o", str(stubs_dir)],
    capture_output=True,
    text=True,
    check=True,
)
stub_file = next(stubs_dir.rglob("*.pyi"))
print(f"stub written to: stubs/{stub_file.relative_to(stubs_dir).as_posix()}")
print(f"--- {stub_file.name} (generated by stubgen) ---\n{stub_file.read_text().strip()}")
stub_env = {**CHECKER_ENV, "MYPYPATH": str(stubs_dir)}
completed = subprocess.run(
    [
        str(VENV_BIN / "mypy"),
        "--no-error-summary",
        "--cache-dir",
        ".mypy_cache",
        *ENV_OPTION,
        "stub_app.py",
    ],
    cwd=UNIT42_DIR,
    capture_output=True,
    text=True,
    env=stub_env,
)
print("with the stub on MYPYPATH:", *completed.stdout.strip().splitlines(), sep="\n  ")

stub written to: stubs/rupee_tools/__init__.pyi
--- __init__.pyi (generated by stubgen) ---
def to_paise(amount: float) -> int: ...
with the stub on MYPYPATH:
  stub_app.py:3: note: Revealed type is "int"
  stub_app.py:4: error: Argument 1 to "to_paise" has incompatible type "str"; expected "float"  [arg-type]


**Gradual adoption.** Nobody types a large codebase in one go. A path that works:

1. Add mypy (or pyright) to CI in its **default** mode, fix what it finds, keep it
   green.
2. Turn on `strict = true` globally and list every not-yet-clean module in a
   `[[tool.mypy.overrides]]` block with `ignore_errors = true` (as in the config
   demo above). New modules are strict from day one.
3. Remove modules from the override list one at a time (a *ratchet*: the list may
   only shrink). Count errors per module to pick the cheapest next target.
4. Type the boundaries first: public functions, data models, I/O. Inference does the
   rest inside function bodies.
5. Keep `warn_unused_ignores` on so stale `# type: ignore` comments disappear.

A ratchet in a few lines: count mypy errors per file of the `typed-shop` project
when everything (legacy included) is checked strictly.

In [127]:
from collections import Counter

all_strict = subprocess.run(
    [
        str(VENV_BIN / "mypy"),
        "--strict",
        "--no-error-summary",
        "--cache-dir",
        "../.mypy_cache",
        "src",
    ],
    cwd=typed_shop,
    capture_output=True,
    text=True,
    env=CHECKER_ENV,
)
per_file = Counter(
    line.split(":")[0] for line in all_strict.stdout.splitlines() if ": error:" in line
)
for path, error_count in per_file.most_common():
    print(f"{error_count:>3} errors  {path}")

  1 errors  src/shop/pricing.py


### Lab 4.2

Seven exercises. For the stubs, replace `raise NotImplementedError`. Exercises 1, 5
and 6 are different: the code already works but is untyped or badly typed, and you
**fix its types in place**. Every check cell runs mypy (`--strict` unless stated)
on the source of your code, which it gets with `source_of(...)`, and also tests the
behaviour. The checker prelude imports `Callable`, `Iterable`, `Iterator`,
`Mapping`, `Sequence` (from `collections.abc`), `dataclass`, `field`, `Decimal`,
`ROUND_HALF_UP`, `Annotated`, `Any`, `Literal`, `Self`, `cast`, `overload` and
`get_type_hints`, so use those names. Until you finish, the checks print FAIL; that
is expected.

#### Exercise 1 — add precise hints to 10 functions

Annotate the functions in the next cell (change nothing else) so that
`mypy --strict` reports no errors and the hints are exactly these (spaces do not
matter; `T` is a PEP 695 type parameter):

| Function | Hints |
| --- | --- |
| `count_words` | `(text: str) -> dict[str, int]` |
| `average` | `(values: Iterable[float]) -> float` |
| `find_user` | `(users: Mapping[int, str], user_id: int) -> str \| None` |
| `pair_up` | `(names: Sequence[str], scores: Sequence[int]) -> list[tuple[str, int]]` |
| `parse_point` | `(text: str) -> tuple[float, float]` |
| `chunk[T]` | `(items: Sequence[T], size: int) -> list[list[T]]` |
| `merge_settings` | `(defaults: Mapping[str, object], overrides: Mapping[str, object]) -> dict[str, object]` |
| `first_or_default[T]` | `(items: Iterable[T], default: T) -> T` |
| `apply_twice[T]` | `(func: Callable[[T], T], value: T) -> T` |
| `status_text` | `(code: int) -> Literal["ok", "client error", "server error", "unknown"]` |

In [128]:
def count_words(text):
    words = text.lower().split()
    counts = dict.fromkeys(words, 0)
    for word in words:
        counts[word] += 1
    return counts


def average(values):
    total = 0.0
    count = 0
    for value in values:
        total += value
        count += 1
    return total / count if count else 0.0


def find_user(users, user_id):
    return users.get(user_id)


def pair_up(names, scores):
    return list(zip(names, scores, strict=True))


def parse_point(text):
    x_text, y_text = text.split(",")
    return float(x_text), float(y_text)


def chunk(items, size):
    return [list(items[start : start + size]) for start in range(0, len(items), size)]


def merge_settings(defaults, overrides):
    return {**defaults, **overrides}


def first_or_default(items, default):
    for item in items:
        return item
    return default


def apply_twice(func, value):
    return func(func(value))


def status_text(code):
    if 200 <= code < 300:
        return "ok"
    if 400 <= code < 500:
        return "client error"
    if 500 <= code < 600:
        return "server error"
    return "unknown"

Run the checks.

In [129]:
HINT_FUNCTIONS = [
    count_words,
    average,
    find_user,
    pair_up,
    parse_point,
    chunk,
    merge_settings,
    first_or_default,
    apply_twice,
    status_text,
]
EXPECTED_HINTS = {
    "count_words": {"text": "str", "return": "dict[str, int]"},
    "average": {"values": "Iterable[float]", "return": "float"},
    "find_user": {"users": "Mapping[int, str]", "user_id": "int", "return": "str | None"},
    "pair_up": {
        "names": "Sequence[str]",
        "scores": "Sequence[int]",
        "return": "list[tuple[str, int]]",
    },
    "parse_point": {"text": "str", "return": "tuple[float, float]"},
    "chunk": {"items": "Sequence[T]", "size": "int", "return": "list[list[T]]"},
    "merge_settings": {
        "defaults": "Mapping[str, object]",
        "overrides": "Mapping[str, object]",
        "return": "dict[str, object]",
    },
    "first_or_default": {"items": "Iterable[T]", "default": "T", "return": "T"},
    "apply_twice": {"func": "Callable[[T], T]", "value": "T", "return": "T"},
    "status_text": {
        "code": "int",
        "return": 'Literal["ok", "client error", "server error", "unknown"]',
    },
}


def hint_strings(function: Callable[..., object]) -> dict[str, str]:
    """The annotations as written in the source, without spaces."""
    written = annotationlib.get_annotations(function, format=annotationlib.Format.STRING)
    return {name: text.replace(" ", "").replace("'", '"') for name, text in written.items()}


run_checks(
    [
        *(
            (
                f"hints of {function.__name__}",
                lambda function=function: hint_strings(function),
                {
                    name: text.replace(" ", "")
                    for name, text in EXPECTED_HINTS[function.__name__].items()
                },
            )
            for function in HINT_FUNCTIONS
        ),
        ("mypy --strict is clean", lambda: mypy_errors(*HINT_FUNCTIONS), []),
        (
            "behaviour unchanged",
            lambda: (
                count_words("a b a"),
                chunk([1, 2, 3], 2),
                status_text(404),
                first_or_default([], 7),
                apply_twice(lambda n: n * 3, 2),
            ),
            ({"a": 2, "b": 1}, [[1, 2], [3]], "client error", 7, 18),
        ),
    ]
)

FAIL  hints of count_words: expected {'text': 'str', 'return': 'dict[str,int]'}, got {}
FAIL  hints of average: expected {'values': 'Iterable[float]', 'return': 'float'}, got {}
FAIL  hints of find_user: expected {'users': 'Mapping[int,str]', 'user_id': 'int', 'return': 'str|None'}, got {}
FAIL  hints of pair_up: expected {'names': 'Sequence[str]', 'scores': 'Sequence[int]', 'return': 'list[tuple[str,int]]'}, got {}
FAIL  hints of parse_point: expected {'text': 'str', 'return': 'tuple[float,float]'}, got {}
FAIL  hints of chunk: expected {'items': 'Sequence[T]', 'size': 'int', 'return': 'list[list[T]]'}, got {}
FAIL  hints of merge_settings: expected {'defaults': 'Mapping[str,object]', 'overrides': 'Mapping[str,object]', 'return': 'dict[str,object]'}, got {}
FAIL  hints of first_or_default: expected {'items': 'Iterable[T]', 'default': 'T', 'return': 'T'}, got {}
FAIL  hints of apply_twice: expected {'func': 'Callable[[T],T]', 'value': 'T', 'return': 'T'}, got {}
FAIL  hints of status_t

FAIL  mypy --strict is clean: expected [], got ['Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]']
PASS  behaviour unchanged
1/12 checks passed


#### Exercise 2 — generic `Stack[T]` and `Result[T, E]`

Implement the classes below using PEP 695 syntax (keep the signatures):

- `Stack[T]`: `push`, `pop` (raises `IndexError("pop from empty stack")` when
  empty), `peek` (`None` when empty), `__len__`, `is_empty`.
- `Ok[T]` and `Err[E]`: frozen dataclasses holding `value` / `error`, with
  `is_ok()` and `unwrap()`. `Err.unwrap()` raises
  `ValueError(f"unwrap on Err: {self.error!r}")`. `Result[T, E]` is their union
  (given).
- `safe_divide(a, b)` returns `Ok(a / b)` or `Err("division by zero")`.

The checks also ask mypy what `Stack[int]().pop()` and `safe_divide(1, 2)` are, and
expect an error for `Stack[int]().push("tea")`.

In [130]:
class Stack[T]:
    """A last-in, first-out stack."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.push")

    def pop(self) -> T:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.pop")

    def peek(self) -> T | None:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.peek")

    def __len__(self) -> int:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.__len__")

    def is_empty(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Stack.is_empty")


@dataclass(frozen=True)
class Ok[T]:
    value: T

    def is_ok(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Ok.is_ok")

    def unwrap(self) -> T:
        raise NotImplementedError("Lab 4.2 exercise 2: Ok.unwrap")


@dataclass(frozen=True)
class Err[E]:
    error: E

    def is_ok(self) -> bool:
        raise NotImplementedError("Lab 4.2 exercise 2: Err.is_ok")

    def unwrap(self) -> typing.Never:
        raise NotImplementedError("Lab 4.2 exercise 2: Err.unwrap")


type Result[T, E] = Ok[T] | Err[E]


def safe_divide(a: float, b: float) -> Result[float, str]:
    """Ok(a / b), or Err("division by zero")."""
    raise NotImplementedError("Lab 4.2 exercise 2: safe_divide")

Run the checks.

In [131]:
GENERIC_USAGE = """
type Result[T, E] = Ok[T] | Err[E]

reveal_type(Stack[int]().pop())
reveal_type(safe_divide(1, 2))
Stack[int]().push("tea")
"""


def stack_trace() -> list[object]:
    stack = Stack[str]()
    stack.push("tea")
    stack.push("rice")
    return [len(stack), stack.peek(), stack.pop(), stack.pop(), stack.is_empty(), stack.peek()]


def generic_errors() -> list[str]:
    errors = mypy_errors(Stack, Ok, Err, safe_divide, extra=GENERIC_USAGE)
    return [error.split("  [")[-1] for error in errors]


run_checks(
    [
        ("Stack behaviour", stack_trace, [2, "rice", "rice", "tea", True, None]),
        (
            "pop on empty",
            lambda: raised_message(lambda: Stack[int]().pop()),
            "IndexError: pop from empty stack",
        ),
        ("Ok", lambda: (Ok(5).is_ok(), Ok(5).unwrap()), (True, 5)),
        (
            "Err",
            lambda: (Err("boom").is_ok(), raised_message(lambda: Err("boom").unwrap())),
            (False, "ValueError: unwrap on Err: 'boom'"),
        ),
        (
            "safe_divide",
            lambda: (safe_divide(1, 4), safe_divide(1, 0)),
            (Ok(0.25), Err("division by zero")),
        ),
        (
            "PEP 695 parameters",
            lambda: (
                [p.__name__ for p in Stack.__type_params__],
                [p.__name__ for p in Result.__type_params__],
            ),
            (["T"], ["T", "E"]),
        ),
        (
            "mypy: types revealed",
            lambda: revealed_types(Stack, Ok, Err, safe_divide, extra=GENERIC_USAGE)[:2],
            ["int", "lab_check.Ok[float] | lab_check.Err[str]"],
        ),
        ("mypy: only the push('tea') error", generic_errors, ["arg-type]"]),
    ]
)

FAIL  Stack behaviour: raised NotImplementedError: Lab 4.2 exercise 2: Stack.push
FAIL  pop on empty: expected 'IndexError: pop from empty stack', got 'NotImplementedError: Lab 4.2 exercise 2: Stack.pop'
FAIL  Ok: raised NotImplementedError: Lab 4.2 exercise 2: Ok.is_ok
FAIL  Err: raised NotImplementedError: Lab 4.2 exercise 2: Err.is_ok
FAIL  safe_divide: raised NotImplementedError: Lab 4.2 exercise 2: safe_divide
PASS  PEP 695 parameters
PASS  mypy: types revealed
PASS  mypy: only the push('tea') error
3/8 checks passed


#### Exercise 3 — a typed `@retry` with `ParamSpec`

Write the decorator factory `retry(times=3, *, exceptions=(Exception,), delay=0.0,
sleep=time.sleep)`, a simpler cousin of Unit 1.12's `retry`:

- Call the function up to `times` times; return the first successful result.
- After a failure with one of `exceptions`, call `sleep(delay)` before the next
  attempt (not after the last); re-raise the last error when attempts run out.
  Other exceptions propagate at once.
- Keep the wrapped function's metadata (`functools.wraps`) **and its signature for
  the checker**: use `[**P, R]` so `mypy --strict` accepts your decorator, keeps
  the parameters of the decorated function and rejects a badly typed call.

In [132]:
def retry[**P, R](
    times: int = 3,
    *,
    exceptions: tuple[type[Exception], ...] = (Exception,),
    delay: float = 0.0,
    sleep: Callable[[float], None] = time.sleep,
) -> Callable[[Callable[P, R]], Callable[P, R]]:
    """Retry the decorated function up to `times` times on the given exceptions."""
    raise NotImplementedError("Lab 4.2 exercise 3: write retry")

Run the checks. `flaky(n)` returns a function that fails `n` times, then succeeds.

In [133]:
SERVICE_DOWN = ConnectionError("down")


def flaky(failures: int, error: Exception = SERVICE_DOWN) -> Callable[[str], str]:
    """A function that raises `error` for its first `failures` calls."""
    calls = {"count": 0}

    def fetch(sku: str) -> str:
        calls["count"] += 1
        if calls["count"] <= failures:
            raise error
        return f"{sku} after {calls['count']} calls"

    return fetch


def retry_run(
    failures: int, times: int, error: Exception = SERVICE_DOWN
) -> tuple[str, list[float]]:
    """Decorate flaky(failures, error); return (result or "Error: message", recorded sleeps)."""
    sleeps: list[float] = []
    decorate = retry(times, exceptions=(ConnectionError,), delay=0.5, sleep=sleeps.append)
    decorated = decorate(flaky(failures, error))
    try:
        outcome = decorated("TEA")
    except Exception as caught:
        outcome = f"{type(caught).__name__}: {caught}"
    return outcome, sleeps


RETRY_USAGE = """
@retry(3, exceptions=(ConnectionError,))
def fetch_price(sku: str, quantity: int = 1) -> Decimal:
    return Decimal("10") * quantity

reveal_type(fetch_price)
fetch_price("TEA", 2)
fetch_price(42)
"""

run_checks(
    [
        ("succeeds on the 3rd call", lambda: retry_run(2, 3), ("TEA after 3 calls", [0.5, 0.5])),
        ("gives up after times", lambda: retry_run(5, 3), ("ConnectionError: down", [0.5, 0.5])),
        ("other errors propagate", lambda: retry_run(5, 3, KeyError("x")), ("KeyError: 'x'", [])),
        ("keeps metadata", lambda: retry()(flaky(0)).__name__, "fetch"),
        (
            "mypy: signature kept",
            lambda: revealed_types(retry, extra=RETRY_USAGE),
            ["def (sku: str, quantity: int =) -> decimal.Decimal"],
        ),
        (
            "mypy: bad call rejected, nothing else",
            lambda: [e.split("  [")[-1] for e in mypy_errors(retry, extra=RETRY_USAGE)],
            ["arg-type]"],
        ),
    ]
)

FAIL  succeeds on the 3rd call: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  gives up after times: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  other errors propagate: raised NotImplementedError: Lab 4.2 exercise 3: write retry
FAIL  keeps metadata: raised NotImplementedError: Lab 4.2 exercise 3: write retry


PASS  mypy: signature kept
PASS  mypy: bad call rejected, nothing else
2/6 checks passed


#### Exercise 4 — an `@overload`ed `parse_value`

`parse_value(text, kind="str")` converts text according to `kind`:

| Call | Returns | Checker type |
| --- | --- | --- |
| `parse_value("42", "int")` | `42` | `int` |
| `parse_value("2.5", "float")` | `2.5` | `float` |
| `parse_value("yes", "bool")` | `True` (`"true"`, `"yes"`, `"1"` → `True`; `"false"`, `"no"`, `"0"` → `False`, case-insensitive; anything else `ValueError`) | `bool` |
| `parse_value(" tea ")` | `"tea"` (stripped) | `str` |

Write four `@overload` signatures using `Literal` kinds (`"str"` is the default),
then the implementation. Any other `kind` must be a checker error. Keep everything
inside the function: the check sends only the overloads and the implementation to
mypy, not module-level constants.

In [134]:
def parse_value(text: str, kind: str = "str") -> object:
    """Convert text to str, int, float or bool according to kind."""
    raise NotImplementedError("Lab 4.2 exercise 4: overloads + implementation")

Run the checks.

In [135]:
OVERLOAD_USAGE = """
reveal_type(parse_value("42", "int"))
reveal_type(parse_value("2.5", "float"))
reveal_type(parse_value("yes", "bool"))
reveal_type(parse_value(" tea "))
parse_value("2026-11-07", "date")
"""


def overload_source() -> str:
    """The overloads and the implementation: source of the whole cell block."""
    return (
        "\n".join(source_of(f) for f in typing.get_overloads(parse_value))
        + "\n"
        + source_of(parse_value)
    )


run_checks(
    [
        ("int", lambda: parse_value("42", "int"), 42),
        ("float", lambda: parse_value("2.5", "float"), 2.5),
        (
            "bool",
            lambda: [parse_value(t, "bool") for t in ("Yes", "TRUE", "1", "no", "False", "0")],
            [True, True, True, False, False, False],
        ),
        ("bad bool", lambda: raised_error_name(lambda: parse_value("maybe", "bool")), "ValueError"),
        ("str (default)", lambda: parse_value(" tea "), "tea"),
        ("four overloads", lambda: len(typing.get_overloads(parse_value)), 4),
        (
            "mypy: types",
            lambda: re.findall(
                r'Revealed type is "([^"]+)"',
                "\n".join(mypy_report(overload_source() + textwrap.dedent(OVERLOAD_USAGE))),
            ),
            ["int", "float", "bool", "str"],
        ),
        (
            "mypy: unknown kind rejected",
            lambda: [
                line.split("  [")[-1]
                for line in mypy_report(overload_source() + textwrap.dedent(OVERLOAD_USAGE))
                if ": error:" in line
            ],
            ["call-overload]"],
        ),
    ]
)

FAIL  int: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  float: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  bool: raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  bad bool: expected 'ValueError', got 'NotImplementedError'
FAIL  str (default): raised NotImplementedError: Lab 4.2 exercise 4: overloads + implementation
FAIL  four overloads: expected 4, got 0
FAIL  mypy: types: expected ['int', 'float', 'bool', 'str'], got ['object', 'object', 'object', 'object']
FAIL  mypy: unknown kind rejected: expected ['call-overload]'], got []
0/8 checks passed


#### Exercise 5 — fix 8 mypy errors in a provided module

The next cell is a small order-report module. It is annotated, but mypy (default
mode) reports **8 errors**, and some of them are real bugs. Fix the code in place
until mypy reports nothing **and** the behaviour checks pass:

- `parse_line("TEA,2,120.50")` → `LineItem("TEA", 2, Decimal("120.50"))`
- `find_item` returns `None` for an unknown SKU (fix the annotation, not the
  behaviour); `quantity_of` returns `0` for an unknown SKU
- `order_total` returns a `Decimal`; `describe` returns `"TEA x 2"`
- `totals_by_sku` maps each SKU to its line total (a `Decimal`)
- `sample_item()` returns a `LineItem` for `"RICE"`, quantity 1, price `80`
- `cheapest` returns the **SKU** of the item with the lowest unit price

Fixing one error can reveal another (a function that now returns `X | None`
makes its callers narrow); keep going until mypy is silent.

The cell after it shows the 8 errors for the code as given.

In [136]:
@dataclass
class LineItem:
    sku: str
    quantity: int
    unit_price: Decimal

    def total(self) -> Decimal:
        return self.quantity * self.unit_price


def parse_line(raw: str) -> LineItem:
    sku, quantity, price = raw.split(",")
    return LineItem(sku, quantity, Decimal(price))


def find_item(items: list[LineItem], sku: str) -> LineItem:
    for item in items:
        if item.sku == sku:
            return item
    return None


def quantity_of(items: list[LineItem], sku: str) -> int:
    return find_item(items, sku).quantity


def order_total(items: list[LineItem]) -> Decimal:
    total = 0
    for item in items:
        total += item.total()
    return total


def describe(item: LineItem) -> str:
    return item.sku + " x " + item.quantity


def totals_by_sku(items: list[LineItem]) -> dict[str, int]:
    totals: dict[str, int] = {}
    for item in items:
        totals[item.sku] = item.total()
    return totals


def sample_item() -> LineItem:
    return LineItem(sku="RICE", quantity=1, price=Decimal("80"))


def cheapest(items: list[LineItem]) -> str:
    return min(items, key=lambda item: item.unit_price)

The errors as they stand (this cell is information, not a check):

In [137]:
ORDER_REPORT = [
    LineItem,
    parse_line,
    find_item,
    quantity_of,
    order_total,
    describe,
    totals_by_sku,
    sample_item,
    cheapest,
]
for error in mypy_errors(*ORDER_REPORT, strict=False):
    print(error)

Argument 2 to "LineItem" has incompatible type "str"; expected "int"  [arg-type]
Incompatible return value type (got "None", expected "LineItem")  [return-value]
Incompatible types in assignment (expression has type "Decimal", variable has type "int")  [assignment]
Incompatible return value type (got "int", expected "Decimal")  [return-value]
Unsupported operand types for + ("str" and "int")  [operator]
Incompatible types in assignment (expression has type "Decimal", target has type "int")  [assignment]
Unexpected keyword argument "price" for "LineItem"  [call-arg]
Incompatible return value type (got "LineItem", expected "str")  [return-value]


Run the checks.

In [138]:
SAMPLE_LINES = [parse_line("TEA,2,120.50"), parse_line("RICE,1,80")]

run_checks(
    [
        ("mypy is clean", lambda: mypy_errors(*ORDER_REPORT, strict=False), []),
        ("parse_line", lambda: parse_line("TEA,2,120.50"), LineItem("TEA", 2, Decimal("120.50"))),
        ("find_item missing", lambda: find_item(SAMPLE_LINES, "SALT"), None),
        (
            "quantity_of",
            lambda: (quantity_of(SAMPLE_LINES, "TEA"), quantity_of(SAMPLE_LINES, "SALT")),
            (2, 0),
        ),
        ("order_total", lambda: order_total(SAMPLE_LINES), Decimal("321.00")),
        ("order_total type", lambda: type(order_total([])).__name__, "Decimal"),
        ("describe", lambda: describe(SAMPLE_LINES[0]), "TEA x 2"),
        (
            "totals_by_sku",
            lambda: totals_by_sku(SAMPLE_LINES),
            {"TEA": Decimal("241.00"), "RICE": Decimal("80")},
        ),
        ("sample_item", sample_item, LineItem("RICE", 1, Decimal("80"))),
        ("cheapest", lambda: cheapest(SAMPLE_LINES), "RICE"),
    ]
)

FAIL  mypy is clean: expected [], got ['Argument 2 to "LineItem" has incompatible type "str"; expected "int"  [arg-type]', 'Incompatible return value type (got "None", expected "LineItem")  [return-value]', 'Incompatible types in assignment (expression has type "Decimal", variable has type "int")  [assignment]', 'Incompatible return value type (got "int", expected "Decimal")  [return-value]', 'Unsupported operand types for + ("str" and "int")  [operator]', 'Incompatible types in assignment (expression has type "Decimal", target has type "int")  [assignment]', 'Unexpected keyword argument "price" for "LineItem"  [call-arg]', 'Incompatible return value type (got "LineItem", expected "str")  [return-value]']
FAIL  parse_line: expected LineItem(sku='TEA', quantity=2, unit_price=Decimal('120.50')), got LineItem(sku='TEA', quantity='2', unit_price=Decimal('120.50'))
PASS  find_item missing
FAIL  quantity_of: raised AttributeError: 'NoneType' object has no attribute 'quantity'
FAIL  order_tot

#### Exercise 6 — type the Phase 2 model to pass `--strict`

Below is a condensed, **untyped** copy of the Phase 2 inventory model: `Money`
(Lab 2.4), `Product` and `Warehouse` (Phase 2 Project). Add annotations (you may
also add attribute annotations or turn `Product` into a dataclass) until
`mypy --strict` reports nothing. The behaviour must not change.

Hints: `__eq__` takes `other: object`; binary dunders may return `NotImplemented`
while being annotated `-> Money`; empty dict attributes need their type.

In [139]:
class CurrencyMismatchError(ValueError):
    """Two Money values in different currencies were combined."""


class Money:
    __slots__ = ("_amount", "_currency")

    def __init__(self, amount, currency="INR"):
        if isinstance(amount, float):
            raise TypeError("amount must be Decimal, int or str, not float")
        self._amount = Decimal(amount).quantize(PAISE, rounding=ROUND_HALF_UP)
        self._currency = currency

    @property
    def amount(self):
        return self._amount

    @property
    def currency(self):
        return self._currency

    def __repr__(self):
        return f"Money({str(self._amount)!r}, {self._currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self._amount, self._currency) == (other._amount, other._currency)

    def __hash__(self):
        return hash((self._amount, self._currency))

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if other._currency != self._currency:
            raise CurrencyMismatchError(f"{self._currency} vs {other._currency}")
        return Money(self._amount + other._amount, self._currency)

    def __mul__(self, factor):
        if isinstance(factor, bool) or not isinstance(factor, (int, Decimal)):
            return NotImplemented
        return Money(self._amount * factor, self._currency)


class InsufficientStockError(Exception):
    def __init__(self, sku, available, requested):
        super().__init__(f"not enough {sku}: {available} available, {requested} requested")
        self.sku, self.available, self.requested = sku, available, requested


class Product:
    def __init__(self, sku, name, unit_price):
        self.sku, self.name, self.unit_price = sku, name, unit_price

    def __repr__(self):
        return f"Product({self.sku!r}, {self.name!r}, {self.unit_price!r})"


class Warehouse:
    def __init__(self, code):
        self.code = code
        self._products = {}
        self._quantities = {}

    def add_product(self, product):
        self._products[product.sku] = product
        self._quantities[product.sku] = 0

    def receive(self, sku, quantity):
        self._quantities[sku] += quantity
        return self._quantities[sku]

    def ship(self, sku, quantity):
        available = self._quantities[sku]
        if quantity > available:
            raise InsufficientStockError(sku, available, quantity)
        self._quantities[sku] = available - quantity
        return self._quantities[sku]

    def stock_value(self):
        return sum(
            (product.unit_price * self._quantities[sku] for sku, product in self._products.items()),
            Money(0),
        )

Run the checks.

In [140]:
INVENTORY_MODEL = [CurrencyMismatchError, Money, InsufficientStockError, Product, Warehouse]


def warehouse_story() -> tuple[object, ...]:
    pune = Warehouse("PNQ")
    pune.add_product(Product("TEA-500", "Assam tea", Money("120.50")))
    pune.add_product(Product("RICE-1K", "Basmati", Money(80)))
    received = pune.receive("TEA-500", 10)
    pune.receive("RICE-1K", 5)
    left = pune.ship("TEA-500", 4)
    shortage = raised_message(lambda: pune.ship("RICE-1K", 9))
    return received, left, shortage, pune.stock_value()


run_checks(
    [
        ("mypy --strict is clean", lambda: mypy_errors(*INVENTORY_MODEL), []),
        (
            "Money arithmetic",
            lambda: (Money("10.005") + Money(5), Money(3) * 2),
            (Money("15.01"), Money("6.00")),
        ),
        (
            "currency mismatch",
            lambda: raised_error_name(lambda: Money(1) + Money(1, "USD")),
            "CurrencyMismatchError",
        ),
        ("float rejected", lambda: raised_error_name(lambda: Money(1.5)), "TypeError"),
        ("NotImplemented path", lambda: raised_error_name(lambda: Money(1) + 1), "TypeError"),
        (
            "warehouse story",
            warehouse_story,
            (
                10,
                6,
                "InsufficientStockError: not enough RICE-1K: 5 available, 9 requested",
                Money("1123.00"),
            ),
        ),
    ]
)

FAIL  mypy --strict is clean: expected [], got ['Function is missing a type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a return type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Call to untyped function "Money" in typed context  [no-untyped-call]', 'Function is missing a type annotation  [no-untyped-def]', 'Call to untyped function "Money" in typed context  [no-untyped-call]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing a type annotation  [no-untyped-def]', 'Function is missing

#### Exercise 7 — an `Annotated`-driven runtime validator

Write `validate(instance)` for dataclass instances. For every field, read the hint
with `get_type_hints(type(instance), include_extras=True)` and return a list of
error messages (empty when valid), in field order:

| Situation | Message |
| --- | --- |
| The value is not an instance of the field's base type (for `Annotated[int, …]` the base is `int`; `bool` is **not** accepted as `int`) | `"age: expected int, got str"` (then skip that field's constraints) |
| `Range(minimum, maximum)` violated | `"age: 200 is not between 0 and 150"` |
| `MaxLength(limit)` violated | `"name: longer than 20 characters"` |
| `Pattern(regex)` does not `fullmatch` | `"email: does not match [^@]+@[^@]+"` |

Fields whose hint is not `Annotated` only get the type check (and only for plain
classes such as `str`; skip hints like `list[str]` or `str | None`).

In [141]:
@dataclass(frozen=True)
class Range:
    minimum: int
    maximum: int


@dataclass(frozen=True)
class MaxLength:
    limit: int


@dataclass(frozen=True)
class Pattern:
    regex: str


def validate(instance: object) -> list[str]:
    """Error messages for a dataclass instance, driven by Annotated metadata."""
    raise NotImplementedError("Lab 4.2 exercise 7: write validate")

Run the checks.

In [142]:
@dataclass
class CustomerForm:
    name: Annotated[str, MaxLength(20)]
    age: Annotated[int, Range(0, 150)]
    email: Annotated[str, Pattern(r"[^@]+@[^@]+"), MaxLength(40)]
    city: str
    tags: list[str] = field(default_factory=list)


run_checks(
    [
        ("valid", lambda: validate(CustomerForm("Asha", 31, "asha@example.com", "Pune")), []),
        (
            "range",
            lambda: validate(CustomerForm("Asha", 200, "asha@example.com", "Pune")),
            ["age: 200 is not between 0 and 150"],
        ),
        (
            "length and pattern",
            lambda: validate(CustomerForm("A" * 21, 31, "not-an-email", "Pune")),
            ["name: longer than 20 characters", "email: does not match [^@]+@[^@]+"],
        ),
        (
            "wrong types",
            lambda: validate(CustomerForm("Asha", "31", "a@b", 42)),
            ["age: expected int, got str", "city: expected str, got int"],
        ),
        (
            "bool is not int",
            lambda: validate(CustomerForm("Asha", True, "a@b", "Pune")),
            ["age: expected int, got bool"],
        ),
        (
            "several constraints",
            lambda: validate(CustomerForm("Asha", 31, "x" * 41 + "@b", "Pune")),
            ["email: longer than 40 characters"],
        ),
        (
            "generic hints skipped",
            lambda: validate(CustomerForm("Asha", 31, "a@b", "Pune", tags="oops")),
            [],
        ),
    ]
)

FAIL  valid: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  range: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  length and pattern: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  wrong types: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  bool is not int: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  several constraints: raised NotImplementedError: Lab 4.2 exercise 7: write validate
FAIL  generic hints skipped: raised NotImplementedError: Lab 4.2 exercise 7: write validate
0/7 checks passed


### Recap — Unit 4.2

| Concept | One-line takeaway | TypeScript equivalent |
| --- | --- | --- |
| Gradual typing | Hints are checked by tools, ignored by the interpreter; untyped code is `Any` | `allowJs`, `// @ts-check` |
| Nominal vs structural | Classes are nominal; `Protocol` opts into structural typing | everything is structural |
| Annotation syntax | `x: int`, `-> T`, `ClassVar`, `Final`; lazy since 3.14 (`annotationlib`) | `: number`, `static`, `const` |
| Collections | `list[int]`, `dict[str, list[User]]`, `tuple[int, str]` vs `tuple[int, ...]`; accept abc, return concrete | `number[]`, `Record`, tuples |
| Unions | `X \| None`, narrow with `is None`, `isinstance`, `match`; `assert_never` for exhaustiveness | control-flow narrowing, `never` |
| Escape hatches | `Any` off, `object` safe, `cast` lies, `# type: ignore[code]` | `any`, `unknown`, `as`, `@ts-expect-error` |
| Precise types | `Literal`, `LiteralString`, `Final`, `NewType`, `Self`, `TypedDict` | literal types, brands, `this`, interfaces |
| Generics | `def f[T]`, `class C[T]`, `type A[T] = ...`; bounds, constraints, defaults; variance inferred | `<T extends …= …>` |
| Callables | `Callable`, callback Protocols, `ParamSpec`, `Concatenate`, `@overload` | function types, overloads |
| Class typing | `type[T]`, `@override`, `@final`, generic Protocols | `typeof C`, `override` |
| Custom narrowing | `TypeIs` (both branches) over `TypeGuard`; tagged unions | `x is T`, discriminated unions |
| `Annotated` | Metadata for run-time libraries; read with `get_type_hints(include_extras=True)` | (none: Zod schemas) |
| Checkers | mypy / pyright strict, `pyproject.toml` config, `py.typed`, stubs, ratchet adoption | `tsc --strict`, `.d.ts` |

**Further reading**

- [`typing`](https://docs.python.org/3/library/typing.html),
  [`annotationlib`](https://docs.python.org/3/library/annotationlib.html),
  [`collections.abc`](https://docs.python.org/3/library/collections.abc.html),
  [Annotations best practices](https://docs.python.org/3/howto/annotations.html)
- [Typing specification](https://typing.python.org/en/latest/spec/) and
  [typing guides](https://typing.python.org/en/latest/guides/)
- [mypy documentation](https://mypy.readthedocs.io/),
  [pyright documentation](https://microsoft.github.io/pyright/),
  [typeshed](https://github.com/python/typeshed)
- [PEP 484](https://peps.python.org/pep-0484/), [PEP 561](https://peps.python.org/pep-0561/),
  [PEP 593](https://peps.python.org/pep-0593/), [PEP 604](https://peps.python.org/pep-0604/),
  [PEP 612](https://peps.python.org/pep-0612/), [PEP 647](https://peps.python.org/pep-0647/),
  [PEP 649](https://peps.python.org/pep-0649/), [PEP 675](https://peps.python.org/pep-0675/),
  [PEP 692](https://peps.python.org/pep-0692/), [PEP 695](https://peps.python.org/pep-0695/),
  [PEP 696](https://peps.python.org/pep-0696/), [PEP 698](https://peps.python.org/pep-0698/),
  [PEP 705](https://peps.python.org/pep-0705/), [PEP 742](https://peps.python.org/pep-0742/),
  [PEP 749](https://peps.python.org/pep-0749/)